# Data import

In [ ]:
import os
import glob
import scanpy as sc
import anndata
import pandas as pd
from scipy import io

data_dir = "/home/NAS/kykim/Oval/E-MTAB-11491/"

# 🔹 filtered 데이터만 찾기
barcodes_files = sorted(glob.glob(os.path.join(data_dir, "*_run_filtered_feature_bc_matrix_barcodes.tsv.gz")))
sample_prefixes = sorted(list(set([
    os.path.basename(f).replace("_run_filtered_feature_bc_matrix_barcodes.tsv.gz", "")
    for f in barcodes_files
])))

# 🔹 uterus 또는 spleen 포함된 샘플만 선택
sample_prefixes = [
    p for p in sample_prefixes
    if ("uterus" in p.lower()) or ("spleen" in p.lower())
]

print(f"📂 선택된 샘플 ({len(sample_prefixes)}개):")
for s in sample_prefixes:
    print("  ┗", s)

adatas = []

for prefix in sample_prefixes:
    print(f"\n🔹 Loading: {prefix}")
    try:
        mtx_file = os.path.join(data_dir, f"{prefix}_run_filtered_feature_bc_matrix_matrix.mtx.gz")
        features_file = os.path.join(data_dir, f"{prefix}_run_filtered_feature_bc_matrix_features.tsv.gz")
        barcodes_file = os.path.join(data_dir, f"{prefix}_run_filtered_feature_bc_matrix_barcodes.tsv.gz")

        matrix = io.mmread(mtx_file).tocsr()
        var = pd.read_csv(features_file, header=None, sep="\t")
        obs = pd.read_csv(barcodes_file, header=None, sep="\t")

        var.columns = ["gene_id", "gene_name", "feature_type"][:var.shape[1]]
        obs.columns = ["barcode"]

        # 중복 gene_name 제거
        var = var.drop_duplicates(subset="gene_name")
        matrix = matrix[var.index, :]

        # obs 구성
        obs.index = [f"{prefix}_{bc}" for bc in obs["barcode"]]
        obs = pd.DataFrame(index=obs.index)
        obs["sample_id"] = prefix

        adata = anndata.AnnData(
            X=matrix.transpose(),
            obs=obs,
            var=var.set_index("gene_name")
        )

        adatas.append(adata)

    except Exception as e:
        print(f"❌ Failed to load {prefix}")
        print(f"   ┗ Error: {e}")
        continue

# 🔹 uterus + spleen 전체 병합
if len(adatas) > 0:
    adata = adatas[0].concatenate(adatas[1:], join="outer", batch_key=None)
    print("\n✅ Merge complete.")
    print("   ┗ Shape:", adata.shape)
    print("   ┗ obs.columns:", adata.obs.columns.tolist())
    print("   ┗ example index:", adata.obs.index[:3].tolist())
else:
    print("!")


In [ ]:
meta_path = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/meta_data.csv"
meta_df = pd.read_csv(meta_path)

meta_df

In [ ]:
# 필요한 컬럼 정의
meta_columns = [
    "Each mouse",
    "organism", 
    "age", 
    "developmental stage", 
    "sex", 
    "disease", 
    "organism part", 
    "clinical information"
]

# 🔹 Source Name을 인덱스로 설정
meta_df_indexed = meta_df.set_index("Source Name")

# 🔹 adata.obs에 필요한 정보 매핑
for col in meta_columns:
    adata.obs[col] = adata.obs["sample_id"].map(meta_df_indexed[col])

# 🔹 무결성 확인
# adata 기준: 매칭되지 않은 경우
unmatched_in_adata = adata.obs[adata.obs[meta_columns].isnull().any(axis=1)]["sample_id"].unique().tolist()

# meta_df 기준: 사용되지 않은 Source Name
used_sample_ids = adata.obs["sample_id"].unique().tolist()
unused_in_meta = meta_df[~meta_df["Source Name"].isin(used_sample_ids)]["Source Name"].tolist()

# 🔹 결과 출력
print("✅ Metadata successfully mapped.")
print(f"🧪 unmatched sample_id in adata.obs: {len(unmatched_in_adata)} → {unmatched_in_adata}")
print(f"📄 unused Source Name in meta_df: {len(unused_in_meta)} → {unused_in_meta}")


In [ ]:
output_path = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/merge_adata.h5ad"

adata.write(output_path)

# Data preprocess

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/merge_adata.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.n_obs

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("mt-")
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
adata.var["hb"] = adata.var_names.str.startswith(("Hba", "Hbb"))
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.0,
    multi_panel=True,
)
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", show=False, ax=axs[0])
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", show=False, ax=axs[1]);

In [ ]:
print("Before QC:", adata.shape)

sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[(adata.obs.n_genes_by_counts <= 6000), :]
adata = adata[adata.obs.pct_counts_mt < 20, :].copy()
adata = adata[(adata.obs.total_counts >= 500) & (adata.obs.total_counts <= 50000), :]

sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)
print("After QC:", adata.shape)

In [ ]:
%matplotlib inline
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.0,
    multi_panel=True,
    show=True  # 반드시 True
)
plt.show()


In [ ]:
import scrublet
import matplotlib.pyplot as plt
sc.pp.scrublet(adata, batch_key="sample_id")
plt.figure(figsize=(6, 4))
plt.hist(adata.obs['doublet_score'], bins=50, color='steelblue', edgecolor='black')
plt.xlabel('Doublet Score')
plt.ylabel('Number of Cells')
plt.title('Distribution of Doublet Scores')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
adata = adata[adata.obs['predicted_doublet'] == False].copy()
print("After removing doublets:", adata.n_obs)

In [ ]:
adata.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_QC_Before_HVG_raw.h5ad")

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_QC_Before_HVG_raw.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key="sample_id")
sc.pl.highly_variable_genes(adata)

In [ ]:
adata = adata[:, adata.var['highly_variable']].copy()

# Clustering

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(adata)
explained_variance_ratio = adata.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(adata, n_comps=11)

In [ ]:
sc.pl.pca(
    adata,
    color=["organism part"],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(
    adata,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(adata, color=["sample_id"])
sc.pl.umap(adata, color=["organism part"])
sc.pl.umap(adata, color=["clinical information"])

In [ ]:
import anndata
import scanpy as sc
import pandas as pd
sc.external.pp.bbknn(adata, batch_key="sample_id")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["sample_id"])
sc.pl.umap(adata, color=["organism part"])
sc.pl.umap(adata, color=["clinical information"])

In [ ]:
for res in [0.25 , 0.5, 0.75, 1]:
    sc.tl.leiden(
        adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph"
    )

In [ ]:
import scanpy as sc
sc.tl.rank_genes_groups(adata, groupby='leiden_res_0.50', method='t-test')  # 또는 'wilcoxon'

top_genes = {}

groups = adata.uns['rank_genes_groups']['names'].dtype.names  # 클러스터 이름들

for group in groups:
    genes = adata.uns['rank_genes_groups']['names'][group][:35]
    top_genes[group] = genes.tolist()

# 결과 확인
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
sc.pl.umap(
    adata,
    color='leiden_res_0.50',
    title='UMAP with confidence-based filtering',
    legend_loc='on data'  # 또는 'right margin', 'none'
)


In [ ]:
cluster_to_group = {
    0: "Immune - Lymphoid",           # AW112010, Nkg7, Il2rb 등 → T/NK 계열
    1: "Immune - Lymphoid",           # Hbb-bs, Hba-a1/2 등 → T/Lymphoid 계열
    2: "Immune - Lymphoid",           # Trbc2, Cd3d, Cd3g 등 → T 세포
    3: "Immune - B cells / APCs",     # Cd74, Cd79a, Igkc 등 → B cell / APC
    4: "Immune - B cells / APCs",     # Ebf1, Cd74, Igkc 등 → B cell / APC
    5: "Immune - B cells / APCs",     # Hbb-bs, Hba-a1/2 등 → B cell / APC (혼합)
    6: "Immune - Myeloid",            # Tyrobp, Cd83, Ctss, Cst3 등 → myeloid
    7: "Stromal / Fibroblasts",       # Col1a1/2, Dcn, Sparc 등 → fibroblast
    8: "Stromal / Fibroblasts",       # Dcn, Col1a1/2, Igfbp4/7 등 → fibroblast
    9: "Immune - Myeloid",            # Fcer1g, Tyrobp, Cd14 등 → myeloid / monocyte
    10: "Stromal / Fibroblasts",      # Dcn, Vim, Igfbp7, Sparc 등 → fibroblast / stromal
    11: "Stromal / Fibroblasts",      # Egr1, Jun, Fos 등 → activated fibroblast / stromal
    12: "Epithelial cells",           # Krt8, Krt18, Epcam 등 → epithelial
    13: "Erythroid lineage"           # Hba-a2, Hbb-bs 등 → 적혈구
}

# adata.obs에 매핑
adata.obs["celltype_grouped"] = adata.obs["leiden_res_0.50"].astype(int).map(cluster_to_group)

for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
sc.pl.umap(
    adata,
    color='celltype_grouped',
    title='UMAP with confidence-based filtering',
    legend_loc='on data'  # 또는 'right margin', 'none'
)


In [ ]:
group_markers = {
    "Epithelial cells": ['Krt8', 'Krt18', 'Epcam', 'Cldn3', 'Cldn4'],
    "Erythroid lineage": ['Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Gypa'],
    "Immune - B cells / APCs": ['Cd74', 'Cd79a', 'Igkc', 'Ebf1', 'Ighm'],
    "Immune - Lymphoid": ['Cd3d', 'Cd3g', 'Cd28', 'Il7r', 'Nkg7'],
    "Immune - Myeloid": ['Tyrobp', 'Fcer1g', 'Cd14', 'C1qa', 'Ctss'],
    "Stromal / Fibroblasts": ['Dcn', 'Col1a1', 'Col1a2', 'Sparc', 'Vim']
}
existing_genes = {cat: [gene for gene in genes if gene in adata.var_names] 
                  for cat, genes in group_markers.items()}
existing_genes = {cat: genes for cat, genes in existing_genes.items() if genes}

sc.pl.dotplot(adata, var_names=existing_genes, groupby="celltype_grouped", standard_scale="var")

In [ ]:
adata.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Clustering.h5ad")

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
group_markers = {
    "Epithelial cells": ['Krt8', 'Krt18', 'Epcam', 'Cldn3', 'Cldn4'],
    "Erythroid lineage": ['Hbb-bs', 'Hba-a1', 'Hba-a2', 'Alas2', 'Gypa'],
    "Immune - B cells / APCs": ['Cd74', 'Cd79a', 'Igkc', 'Ebf1', 'Ighm'],
    "Immune - Lymphoid": ['Cd3d', 'Cd3g', 'Cd28', 'Il7r', 'Nkg7'],
    "Immune - Myeloid": ['Tyrobp', 'Fcer1g', 'Cd14', 'C1qa', 'Ctss'],
    "Stromal / Fibroblasts": ['Dcn', 'Col1a1', 'Col1a2', 'Sparc', 'Vim']
}
existing_genes = {cat: [gene for gene in genes if gene in adata.var_names] 
                  for cat, genes in group_markers.items()}
existing_genes = {cat: genes for cat, genes in existing_genes.items() if genes}

sc.pl.dotplot(adata, var_names=existing_genes, groupby="leiden_res_0.50", standard_scale="var")

In [ ]:
# Clustering based pathway

In [ ]:
sc.tl.rank_genes_groups(adata, 'celltype_grouped', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)
sc.tl.dendrogram(adata, groupby='celltype_grouped')
sc.pl.rank_genes_groups_dotplot(
        adata, groupby="celltype_grouped", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata, groupby='celltype_grouped')
sc.pl.rank_genes_groups_dotplot(
        adata, groupby="celltype_grouped", standard_scale="var",  n_genes=5, use_raw=False
)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

In [ ]:
import sys
sys.path.append("/home/Data_Drive_8TB/kykim/QED")
from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect

In [ ]:
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()


In [ ]:
from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = sorted(df[group_by].unique())  # ✅ X축 정렬에 맞춤

    # 🔑 Y축을 X축 순서에 맞춰 계단식으로 GO term 배치
    for group in group_order_for_plot:
        current_group_terms = (
            df_ranked_per_group[df_ranked_per_group[group_by] == group]
            .sort_values(order_by)
        )
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    # row_order 적용
    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=5,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=13,
    ylabel_fontsize=13,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

# Cell type proportion

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 스타일 설정
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# 공통 함수: stacked barplot 그리기
def plot_stacked_bar(df, index_col, columns_col, values_col, title="", ylabel=""):
    total = df.groupby(index_col)[values_col].sum().reset_index(name='total')
    df = df.merge(total, on=index_col)
    df["proportion"] = df[values_col] / df["total"]
    
    pivot_df = df.pivot(index=index_col, columns=columns_col, values="proportion").fillna(0)
    colors = sns.color_palette("tab10", n_colors=pivot_df.shape[1])
    
    ax = pivot_df.plot(kind="bar", stacked=True, color=colors, figsize=(10, 6))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(title=columns_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# 1. organism part → celltype grouped
df1 = adata.obs.groupby(["organism part", "celltype_grouped"]).size().reset_index(name='count')
plot_stacked_bar(
    df1,
    index_col="organism part",
    columns_col="celltype_grouped",
    values_col="count",
    title="Celltype Proportion by Organism Part",
)

# 2. clinical information → celltype grouped
df2 = adata.obs.groupby(["clinical information", "celltype_grouped"]).size().reset_index(name='count')
plot_stacked_bar(
    df2,
    index_col="clinical information",
    columns_col="celltype_grouped",
    values_col="count",
    title="Celltype Proportion by Clinical Information",
    ylabel="Proportion"
)

# 3. organism part → clinical information
df3 = adata.obs.groupby(["organism part", "clinical information"]).size().reset_index(name='count')
plot_stacked_bar(
    df3,
    index_col="organism part",
    columns_col="clinical information",
    values_col="count",
    title="Clinical Info Proportion by Organism Part",
    ylabel="Proportion"
)

# 4. clinical information → organism part
df4 = adata.obs.groupby(["clinical information", "organism part"]).size().reset_index(name='count')
plot_stacked_bar(
    df4,
    index_col="clinical information",
    columns_col="organism part",
    values_col="count",
    title="Organism Part Proportion by Clinical Information",
    ylabel="Proportion"
)


In [ ]:
import seaborn as sns
import numpy as np
def plot_stacked_bar(df, index_col, columns_col, values_col, title="", ylabel=""):
    total = df.groupby(index_col)[values_col].sum().reset_index(name='total')
    df = df.merge(total, on=index_col)
    df["proportion"] = df[values_col] / df["total"]

    pivot_df = df.pivot(index=index_col, columns=columns_col, values="proportion").fillna(0)
    colors = sns.color_palette("tab10", n_colors=pivot_df.shape[1])

    ax = pivot_df.plot(kind="bar", stacked=True, color=colors, figsize=(10, 6))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(title=columns_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
# age_num 추출 후 그룹화
adata.obs["age_num"] = adata.obs["age"].str.extract(r"(\d+)").astype(int)
adata.obs["age_group"] = np.where(adata.obs["age_num"] > 70, "Aged", "Young")

# organism part × age_group × celltype_grouped count
df2 = (
    adata.obs
    .groupby(["organism part", "age_group", "celltype_grouped"])
    .size()
    .reset_index(name="count")
)

# stacked barplot 함수 호출
plot_stacked_bar(
    df2,
    index_col=["organism part", "age_group"],   # index에 age_group 포함
    columns_col="celltype_grouped",
    values_col="count",
    title="Celltype Proportion by Organism Part and Age Group",
    ylabel="Proportion",
)


In [ ]:
import pandas as pd

counts = adata.obs['celltype_grouped'].value_counts()
percent = adata.obs['celltype_grouped'].value_counts(normalize=True) * 100

df_summary = pd.DataFrame({
    'count': counts,
    'percent': percent.round(2)   # 소수점 둘째 자리 반올림
})

df_summary


In [ ]:
ㅁ

# Epithelial subtypeing

In [ ]:
import scanpy as sc
path_after = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Clustering.h5ad"
path_before = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_QC_Before_HVG_raw.h5ad"

adata_after = sc.read_h5ad(path_after)
adata_before = sc.read_h5ad(path_before)
adata_before.obs['celltype_grouped'] = adata_after.obs.loc[adata_before.obs.index, 'celltype_grouped']

In [ ]:
adata_before.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw_clustering(large).h5ad")

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw_clustering(large).h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'].value_counts()

In [ ]:
##############################################################################

In [ ]:
#########################Epithelial subtype###################################

In [ ]:
Epithelial = adata[adata.obs['celltype_grouped'] == 'Epithelial cells'].copy()
Epithelial.layers["counts"] = Epithelial.X.copy()
sc.pp.normalize_total(Epithelial)
sc.pp.log1p(Epithelial)
sc.pp.highly_variable_genes(Epithelial, n_top_genes=2000, batch_key="Each mouse")
sc.pl.highly_variable_genes(Epithelial)
Epithelial = Epithelial[:, Epithelial.var['highly_variable']].copy()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(Epithelial)
explained_variance_ratio = Epithelial.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(Epithelial, n_comps=11)

In [ ]:
sc.pl.pca(
    Epithelial,
    color=["sample_id"],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(Epithelial)
sc.tl.umap(Epithelial)
sc.pl.umap(
    Epithelial,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(Epithelial, color=["sample_id"])
sc.pl.umap(Epithelial, color=["organism part"])
sc.pl.umap(Epithelial, color=["clinical information"])
sc.pl.umap(Epithelial, color=["age"])

In [ ]:
for res in [0.05, 0.10,0.25,0.5]:
    sc.tl.leiden(
        Epithelial, key_added=f"leiden_res_{res:4.2f}", resolution=res, 
        flavor="igraph"
    )

In [ ]:
sc.pl.umap(Epithelial, color=["leiden_res_0.05"])
sc.pl.umap(Epithelial, color=["leiden_res_0.10"])
sc.pl.umap(Epithelial, color=["leiden_res_0.25"])

In [ ]:
genes_to_check = ['Lif', 'Hbegf', 'Muc4', 'Muc1']
for gene in genes_to_check:
    if gene in adata.var_names:
        print(f"{gene} is present in the dataset.")
    else:
        print(f"{gene} is NOT present in the dataset.")

In [ ]:
import scanpy as sc
sc.tl.rank_genes_groups(Epithelial, groupby='leiden_res_0.10', method='t-test')  # 또는 'wilcoxon'
top_genes = {}
groups = Epithelial.uns['rank_genes_groups']['names'].dtype.names  # 클러스터 이름들
for group in groups:
    genes = Epithelial.uns['rank_genes_groups']['names'][group][:20]
    top_genes[group] = genes.tolist()


for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
sc.pl.umap(
    Epithelial,
    color='leiden_res_0.10',
    title='UMAP with confidence-based filtering',
    legend_loc='on data'
)

In [ ]:
Epithelial.obs["leiden_res_0.10"] = Epithelial.obs["leiden_res_0.10"].astype(int)

cluster_to_epithelial_subtype = {
    0: "Proliferative_Luminal_epi",     
    1: "Secretory_Luminal_epi",      
    2: "Receptive_Luminal_epi",          
    3: "Proliferative_Luminal_epi"          
}

Epithelial.obs['epithelial_subtype'] = Epithelial.obs['leiden_res_0.10'].map(cluster_to_epithelial_subtype)

In [ ]:
epithelial_subtype_markers = {
    "Luminal_Epithelial_Markers": ['S100g','Krt8','Krt18','Cebpd','Wfdc2', 'Gpx3', 'Clu','Muc1'],                                                      # 공통 마커
    "Proliferative_Luminal_epi": ['Hes1', 'Id1', 'Id2', 'Msx1', 'Gstm1'],
    "Receptive_Luminal_epi": ['Lif', 'Il1a', 'Hbegf', 'Icam1', 'Nfkbia'],
    "Secretory_Luminal_epi": ['Ltf', 'Sprr2f', 'Fcgbp', 'Anxa1', 'Ceacam1'],               
   }                          
existing_subtype_markers = {
    subtype: [gene for gene in genes if gene in Epithelial.var_names]
    for subtype, genes in epithelial_subtype_markers.items()
}
sc.pl.dotplot(
    Epithelial,
    var_names=existing_subtype_markers,
    groupby="epithelial_subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(7, 3)
)

sc.pl.umap(
    Epithelial,
    color="epithelial_subtype",
    legend_loc="on data",
    frameon=False,
    title="Epithelial Subtypes"
)


In [ ]:
adata_after = Epithelial
adata_before = adata[adata.obs['celltype_grouped'] == 'Epithelial cells'].copy()
adata_before.obs['subtype'] = adata_after.obs.loc[adata_before.obs.index, 'epithelial_subtype']

In [ ]:
Epithelial.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Epithelial_subtype_raw.h5ad")

# Fibroblast subtyping

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw_clustering(large).h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'].value_counts()

In [ ]:
Fibroblast = adata[adata.obs['celltype_grouped'] == 'Stromal / Fibroblasts'].copy()

In [ ]:
Fibroblast = adata[adata.obs['celltype_grouped'] == 'Stromal / Fibroblasts'].copy()
Fibroblast.layers["counts"] = Fibroblast.X.copy()
sc.pp.normalize_total(Fibroblast)
sc.pp.log1p(Fibroblast)
sc.pp.highly_variable_genes(Fibroblast, n_top_genes=2000, batch_key="Each mouse")
sc.pl.highly_variable_genes(Fibroblast)
Fibroblast = Fibroblast[:, Fibroblast.var['highly_variable']].copy()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(Fibroblast)
explained_variance_ratio = Fibroblast.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(Fibroblast, n_comps=16)

In [ ]:
sc.pl.pca(
    Fibroblast,
    color=["sample_id"]
)

In [ ]:
sc.pp.neighbors(Fibroblast)
sc.tl.umap(Fibroblast)
sc.pl.umap(
    Fibroblast,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(Fibroblast, color=["sample_id"])
sc.pl.umap(Fibroblast, color=["organism part"])
sc.pl.umap(Fibroblast, color=["clinical information"])
sc.pl.umap(Fibroblast, color=["age"])

In [ ]:
for res in [0.10,0.25,0.5,0.75]:
    sc.tl.leiden(
        Fibroblast, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph"
    )

In [ ]:
sc.pl.umap(Fibroblast, color=["leiden_res_0.10"])
sc.pl.umap(Fibroblast, color=["leiden_res_0.25"])
sc.pl.umap(Fibroblast, color=["leiden_res_0.50"])
sc.pl.umap(Fibroblast, color=["leiden_res_0.75"])

In [ ]:
import pandas as pd
Fibroblast.obs['leiden_res_0.10'] = Fibroblast.obs['leiden_res_0.10'].astype(str)
sc.tl.rank_genes_groups(Fibroblast, groupby='leiden_res_0.10', method='t-test')
top_genes = {}
groups = Fibroblast.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = Fibroblast.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}
sc.pl.dotplot(
    Fibroblast,
    var_names=top5_genes_per_cluster,
    groupby='leiden_res_0.10',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    Fibroblast,
    color='leiden_res_0.10',
    title='UMAP',
    legend_loc='on data'
)


In [ ]:
Fibroblast.obs["leiden_res_0.10"] = Fibroblast.obs["leiden_res_0.10"].astype(int)

cluster_to_fibro_subtype = {
    0: "ECM_fibroblast",        
    1: "Metabolic_fibroblast",
    2: "Signaling_fibroblast"
}

Fibroblast.obs['fibroblast_subtype'] = Fibroblast.obs['leiden_res_0.10'].map(cluster_to_fibro_subtype)

In [ ]:
fibroblast_subtype_markers = {
    "ECM_fibroblast": ['Col1a1', 'Col1a2', 'Col3a1', 'Dcn', 'Fbln1'],
    "Metabolic_fibroblast": ['mt-Nd2', 'mt-Nd4', 'Dio2', 'P2ry14', 'Ramp3'],
    "Signaling_fibroblast": ['Fst', 'Sfrp4', 'Igfbp3', 'Hoxa10', 'Ctla2a']}

existing_subtype_markers = {
    subtype: [gene for gene in genes if gene in Fibroblast.var_names]
    for subtype, genes in fibroblast_subtype_markers.items()
}

sc.pl.dotplot(
    Fibroblast,
    var_names=existing_subtype_markers,
    groupby="fibroblast_subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(7, 2)
)

sc.pl.umap(
    Fibroblast,
    color="fibroblast_subtype",
    legend_loc="on data",
    frameon=False,
    title="Fibroblast Subtypes"
)


In [ ]:
adata_after = Fibroblast
adata_before = adata[adata.obs['celltype_grouped'] == 'Stromal / Fibroblasts'].copy()
adata_before.obs['subtype'] = adata_after.obs.loc[adata_before.obs.index, 'fibroblast_subtype']

In [ ]:
Fibroblast.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Fibroblast_subtype_raw.h5ad")

# Myeloid subtyping

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw_clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'].value_counts()

In [ ]:
Myeoloid = adata[adata.obs['celltype_grouped'] == 'Immune - Myeloid'].copy()
Myeoloid.layers["counts"] = Myeoloid.X.copy()
sc.pp.normalize_total(Myeoloid)
sc.pp.log1p(Myeoloid)
sc.pp.highly_variable_genes(Myeoloid, n_top_genes=5000, batch_key="Each mouse")
sc.pl.highly_variable_genes(Myeoloid)
Myeoloid = Myeoloid[:, Myeoloid.var['highly_variable']].copy()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(Myeoloid)
explained_variance_ratio = Myeoloid.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(Myeoloid, n_comps=21)

In [ ]:
sc.pl.pca(
    Myeoloid,
    color=["sample_id", "clinical information", 'organism part'],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(Myeoloid)
sc.tl.umap(Myeoloid)
sc.pl.umap(
    Myeoloid,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(Myeoloid, color=["sample_id"])
sc.pl.umap(Myeoloid, color=["organism part"])
sc.pl.umap(Myeoloid, color=["clinical information"])
sc.pl.umap(Myeoloid, color=["age"])

In [ ]:
for res in [0.05, 0.10,0.25]:
    sc.tl.leiden(
        Myeoloid, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph"
    )

In [ ]:
sc.pl.umap(Myeoloid, color=["leiden_res_0.05"])
sc.pl.umap(Myeoloid, color=["leiden_res_0.10"])
sc.pl.umap(Myeoloid, color=["leiden_res_0.25"])

In [ ]:
import pandas as pd

# 1. leiden_res_0.10 컬럼을 문자열 타입으로 변환 (cat 필요)
Myeoloid.obs['leiden_res_0.25'] = Myeoloid.obs['leiden_res_0.25'].astype(str)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(Myeoloid, groupby='leiden_res_0.25', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = Myeoloid.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = Myeoloid.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    Myeoloid,
    var_names=top5_genes_per_cluster,
    groupby='leiden_res_0.25',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    Myeoloid,
    color='leiden_res_0.25',
    title='UMAP',
    legend_loc='on data'
)


In [ ]:
Myeoloid.obs["leiden_res_0.25"] = Myeoloid.obs["leiden_res_0.25"].astype(int)

cluster_to_myeloid_subtype = {
    0: "Myeloid_1",
    1: "Myeloid_2",
    2: "Epithelial",               # Krt14 등
    3: "Epithelial",         # Epithelial + MHC II
    4: "Myeloid_3",
    5: "Myeloid_4",
    6: "Macrophage_1",             # C1qa, Apoe, Mrc1 등
    7: "Epithelial",      # mt-Co genes + Krt
    8: "Macrophage_2",
    9: "Macrophage_3",        # lipid-associated or osteoclast-like
    10: "Myeloid_5"
}

Myeoloid.obs['myeloid_macrophage'] = Myeoloid.obs['leiden_res_0.25'].map(cluster_to_myeloid_subtype)
sc.pl.umap(
    Myeoloid,
    color='myeloid_macrophage',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
Myeloid = Myeoloid[Myeoloid.obs['myeloid_macrophage'] != 'Epithelial'].copy()

In [ ]:
import pandas as pd

# 1. leiden_res_0.10 컬럼을 문자열 타입으로 변환 (cat 필요)
Myeloid.obs['leiden_res_0.25'] = Myeloid.obs['myeloid_macrophage'].astype(str)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(Myeloid, groupby='myeloid_macrophage', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = Myeloid.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = Myeloid.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    Myeloid,
    var_names=top5_genes_per_cluster,
    groupby='myeloid_macrophage',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    Myeloid,
    color='myeloid_macrophage',
    title='UMAP',
    legend_loc='on data'
)


In [ ]:
cluster_to_myeloid_subtype = {
    "Macrophage_1": "M2_macrophage",
    "Macrophage_2": "M2_macrophage",
    "Macrophage_3": "LAM_macrophage",
    "Myeloid_1": "Inflammatory_monocyte",
    "Myeloid_2": "Inflammatory_monocyte",
    "Myeloid_3": "APC_myeloid",
    "Myeloid_4": "Tissue_remodeling_myeloid",
    "Myeloid_5": "M1_macrophage"
}


In [ ]:
Myeloid.obs['myeloid_subtype'] = Myeloid.obs['myeloid_macrophage'].map(cluster_to_myeloid_subtype)
sc.pl.umap(
    Myeloid,
    color='myeloid_subtype',
    title='UMAP',
    legend_loc='on data'
)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(Myeloid, groupby='myeloid_subtype', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = Myeloid.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = Myeloid.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    Myeloid,
    var_names=top5_genes_per_cluster,
    groupby='myeloid_subtype',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    Myeloid,
    color='myeloid_subtype',
    title='UMAP',
    legend_loc='on data'
)


In [ ]:
# 각 subtype별 대표 마커 5개씩 선별 (예시)
myeloid_representative_markers = {
    "APC_myeloid": ['Cd74', 'H2-Aa', 'Lgals3', 'Cd83', 'Lyz2'],
    "Inflammatory_monocyte": ['S100a9', 'S100a8', 'Cxcl2', 'G0s2', 'Slc7a11'],
    "LAM_macrophage": ['Fabp4', 'Pou2f2', 'Tcf7l2', 'Fcgr4', 'Nrip1'],
    "M1_macrophage": ['Il6', 'Ccl4', 'Gzmb', 'Ccl3', 'Gata2'],
    "M2_macrophage": ['Apoe', 'C1qa', 'Mrc1', 'C1qb', 'Selenop','Cd163'],
    "Tissue_remodeling_myeloid": ['S100a4', 'Lyz2', 'Crip1', 'Plac8', 'Ifitm3']
}

# 실제 데이터에 존재하는 유전자 필터링
existing_markers = {
    subtype: [gene for gene in genes if gene in Myeloid.var_names]
    for subtype, genes in myeloid_representative_markers.items()
}

# dotplot 시각화
sc.pl.dotplot(
    Myeloid,
    var_names=existing_markers,
    groupby="myeloid_subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(10, 4)
)


In [ ]:
Myeloid.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Myeloid_subtype.h5ad")

In [ ]:
sc.tl.rank_genes_groups(Myeloid, 'myeloid_subtype', method="wilcoxon", use_raw=False)
sc.pl.rank_genes_groups(Myeloid, n_genes=25, sharey=False)
sc.tl.dendrogram(Myeloid, groupby='myeloid_subtype')
sc.pl.rank_genes_groups_dotplot(
    Myeloid,
    groupby="myeloid_subtype",
    standard_scale="var",
    n_genes=5,
    use_raw=False
)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Myeloid.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(Myeloid, groupby='myeloid_subtype')
sc.pl.rank_genes_groups_dotplot(
        Myeloid, groupby="myeloid_subtype", standard_scale="var",  n_genes=5, use_raw=False)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Myeloid.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

In [ ]:
from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

In [ ]:
from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=5,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=13,
    ylabel_fontsize=13,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

In [ ]:
#######Celltypist validation

In [ ]:
import celltypist
from celltypist import models
models.models_description()

In [ ]:
model = models.Model.load(model = 'Immune_All_High.pkl')

In [ ]:
predictions = celltypist.annotate(Myeloid, model = 'Immune_All_High.pkl', majority_voting = True)

In [ ]:
Myeloid = predictions.to_adata()

In [ ]:
sc.pl.umap(Myeloid, color = ['predicted_labels', 'majority_voting'], legend_loc = 'on data')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(Myeloid.obs['conf_score'], bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Confidence Score')
plt.ylabel('Number of Cells')
plt.title('Histogram of Confidence Scores in Myeloid Cells')
plt.show()


# Lymphoid subtype annotation

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw_clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'].value_counts()

In [ ]:
Lymphoid = adata[adata.obs['celltype_grouped'] == 'Immune - Lymphoid'].copy()
Lymphoid.layers["counts"] = Lymphoid.X.copy()
sc.pp.normalize_total(Lymphoid)
sc.pp.log1p(Lymphoid)
sc.pp.highly_variable_genes(Lymphoid, n_top_genes=5000, batch_key="Each mouse")
sc.pl.highly_variable_genes(Lymphoid)
Lymphoid = Lymphoid[:, Lymphoid.var['highly_variable']].copy()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(Lymphoid)
explained_variance_ratio = Lymphoid.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(Lymphoid, n_comps=21)

In [ ]:
sc.pl.pca(
    Lymphoid,
    color=["sample_id", "clinical information", 'organism part'],
    ncols=2,
    size=2,
)


In [ ]:
sc.pp.neighbors(Lymphoid)
sc.tl.umap(Lymphoid)
sc.pl.umap(
    Lymphoid,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(Lymphoid, color=["sample_id"])
sc.pl.umap(Lymphoid, color=["organism part"])
sc.pl.umap(Lymphoid, color=["clinical information"])
sc.pl.umap(Lymphoid, color=["age"])

In [ ]:
for res in [0.05, 0.10,0.25]:
    sc.tl.leiden(
        Lymphoid, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph"
    )

sc.pl.umap(Lymphoid, color=["leiden_res_0.05"])
sc.pl.umap(Lymphoid, color=["leiden_res_0.10"])
sc.pl.umap(Lymphoid, color=["leiden_res_0.25"])

In [ ]:
import pandas as pd

# 1. leiden_res_0.10 컬럼을 문자열 타입으로 변환 (cat 필요)
Lymphoid.obs['leiden_res_0.25'] = Lymphoid.obs['leiden_res_0.25'].astype(str)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(Lymphoid, groupby='leiden_res_0.25', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = Lymphoid.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = Lymphoid.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    Lymphoid,
    var_names=top5_genes_per_cluster,
    groupby='leiden_res_0.25',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    Lymphoid,
    color='leiden_res_0.25',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
Lymphoid.obs["leiden_res_0.25"] = Lymphoid.obs["leiden_res_0.25"].astype(int)

cluster_to_lymphoid_subtype = {
    0: "Epithelial (Contamination)",
    1: "Treg",
    2: "Effector_CD8_T",
    3: "Tph/Th17_like",
    4: "Th2_like",
    5: "Naive_T/Tcm",
    6: "Naive_T/Tcm",         # 병합
    7: "NK/NKT_cell",
    8: "NK/NKT_cell",         # 병합
    9: "Mast_cell"
}

Lymphoid.obs['lymphoid_subtype'] = Lymphoid.obs['leiden_res_0.25'].map(cluster_to_lymphoid_subtype)
sc.pl.umap(
    Lymphoid,
    color='lymphoid_subtype',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
Lymphoid = Lymphoid[Lymphoid.obs['lymphoid_subtype'] != 'Epithelial (Contamination)'].copy()

In [ ]:
cluster_to_lymphoid_subtype = {
    0: "Non-lymphoid_keratin+",
    1: "Treg",
    2: "Effector_CD8_T",
    3: "Tph/Th17_like",
    4: "Th2_like",
    5: "Naive_T/Tcm",
    6: "Naive_T/Tcm",         # 병합
    7: "NK/NKT_cell",
    8: "NK/NKT_cell",         # 병합
    9: "Mast_cell"
}

In [ ]:
Lymphoid.obs['lymphoid_subtype'] = Lymphoid.obs['leiden_res_0.25'].map(cluster_to_lymphoid_subtype)
sc.pl.umap(
    Lymphoid,
    color='lymphoid_subtype',
    title='UMAP',
    legend_loc='on data'
)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(Lymphoid, groupby='lymphoid_subtype', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = Lymphoid.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = Lymphoid.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    Lymphoid,
    var_names=top5_genes_per_cluster,
    groupby='lymphoid_subtype',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    Lymphoid,
    color='lymphoid_subtype',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
lymphoid_representative_markers = {
    "Effector_CD8_T": ['Gzmk', 'Ccl5', 'Hopx', 'Cd28', 'Eomes'],
    "Mast_cell": ['Cma1', 'Tpsb2', 'Cpa3', 'Hdc', 'Mcpt4'],
    "NK/NKT_cell": ['Nkg7', 'Il2rb', 'Fcer1g', 'Irf8', 'Prf1'],
    "Naive_T/Tcm": ['Lef1', 'Ccr7', 'Satb1', 'Klf2', 'S1pr1'],
    "Th2_like": ['Gata3', 'Il5', 'Rora', 'Areg', 'Nfkbiz'],
    "Tph/Th17_like": ['Cxcr6', 'Icos', 'Rora', 'Il7r', 'Ikzf3'],
    "Treg": ['Ctla4', 'Tnfrsf4', 'Tox', 'Maf', 'Ikzf2'],
}

# 실제 데이터에 존재하는 유전자 필터링
existing_markers = {
    subtype: [gene for gene in genes if gene in Lymphoid.var_names]
    for subtype, genes in lymphoid_representative_markers.items()
}

# dotplot 시각화
sc.pl.dotplot(
    Lymphoid,
    var_names=existing_markers,
    groupby="lymphoid_subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(10, 4)
)
    # 

In [ ]:
Lymphoid.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Lymphoid_subtype.h5ad")

In [ ]:
sc.tl.rank_genes_groups(Lymphoid, 'lymphoid_subtype', method="wilcoxon", use_raw=False)
sc.pl.rank_genes_groups(Lymphoid, n_genes=25, sharey=False)
sc.tl.dendrogram(Lymphoid, groupby='lymphoid_subtype')
sc.pl.rank_genes_groups_dotplot(
    Lymphoid,
    groupby="lymphoid_subtype",
    standard_scale="var",
    n_genes=5,
    use_raw=False
)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Lymphoid.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(Lymphoid, groupby='lymphoid_subtype')
sc.pl.rank_genes_groups_dotplot(
        Lymphoid, groupby="lymphoid_subtype", standard_scale="var",  n_genes=5, use_raw=False)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Lymphoid.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

In [ ]:
from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

In [ ]:
from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=5,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=13,
    ylabel_fontsize=13,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

In [ ]:
import celltypist
from celltypist import models
model = models.Model.load(model = 'Immune_All_Low.pkl')

In [ ]:
predictions = celltypist.annotate(Lymphoid, model = 'Immune_All_Low.pkl', majority_voting = True)

In [ ]:
Lymphoid = predictions.to_adata()

In [ ]:
sc.pl.umap(Lymphoid, color = ['predicted_labels', 'majority_voting'], legend_loc = 'on data')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(Lymphoid.obs['conf_score'], bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Confidence Score')
plt.ylabel('Number of Cells')
plt.title('Histogram of Confidence Scores in Myeloid Cells')
plt.show()

In [ ]:
lym = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Lymphoid_subtype.h5ad")

In [ ]:
cd4_markers = ['Cd4', 'Il7r', 'Icos', 'Cxcr6', 'Rora', 'Gata3', 'Ctla4', 'Tnfrsf4', 'Ikzf2', 'Il1rl1']
cd8_markers = ['Cd8a', 'Cd8b1', 'Gzmk', 'Prf1', 'Nkg7', 'Ccl5', 'Klrk1']

# 클러스터별 평균 발현
sc.tl.score_genes(lym, cd4_markers, score_name='CD4_score', use_raw=False)
sc.tl.score_genes(lym, cd8_markers, score_name='CD8_score', use_raw=False)

var_dict = {
    "CD4_markers": cd4_markers,
    "CD8_markers": cd8_markers
}
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(20, 5))

sc.pl.umap(lym, color='lymphoid_subtype', ax=axs[0], show=False, title='Subtype')
sc.pl.umap(lym, color='CD4_score', ax=axs[1], show=False, title='CD4 score')
sc.pl.umap(lym, color='CD8_score', ax=axs[2], show=False, title='CD8 score')

plt.tight_layout()
plt.show()


In [ ]:

sc.pl.dotplot(
    lym,
    var_names=var_dict,
    groupby="lymphoid_subtype",
    standard_scale="var",
    swap_axes=False
)

# B/APC subtype annotation

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw_clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'].value_counts()

In [ ]:
B = adata[adata.obs['celltype_grouped'] == 'Immune - B cells / APCs'].copy()
B.layers["counts"] = B.X.copy()
sc.pp.normalize_total(B)
sc.pp.log1p(B)
sc.pp.highly_variable_genes(B, n_top_genes=5000, batch_key="Each mouse")
sc.pl.highly_variable_genes(B)
B = B[:, B.var['highly_variable']].copy()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(B)
explained_variance_ratio = B.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(B, n_comps=16)

In [ ]:
sc.pl.pca(
    B,
    color=["sample_id", "clinical information", 'organism part'],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(B)
sc.tl.umap(B)
sc.pl.umap(
    B,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(B, color=["sample_id"])
sc.pl.umap(B, color=["organism part"])
sc.pl.umap(B, color=["clinical information"])
sc.pl.umap(B, color=["age"])

In [ ]:
for res in [0.05, 0.10,0.25,0.35]:
    sc.tl.leiden(
        B, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph"
    )
sc.pl.umap(B, color=["leiden_res_0.05"])
sc.pl.umap(B, color=["leiden_res_0.10"])
sc.pl.umap(B, color=["leiden_res_0.25"])
sc.pl.umap(B, color=["leiden_res_0.35"])

In [ ]:
import pandas as pd

# 1. leiden_res_0.10 컬럼을 문자열 타입으로 변환 (cat 필요)
B.obs['leiden_res_0.25'] = B.obs['leiden_res_0.25'].astype(str)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(B, groupby='leiden_res_0.25', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = B.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = B.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    B,
    var_names=top5_genes_per_cluster,
    groupby='leiden_res_0.25',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    B,
    color='leiden_res_0.25',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
B.obs["leiden_res_0.25"] = B.obs["leiden_res_0.25"].astype(int)

cluster_to_bcell_subtype = {
    0: "MZ_like_B_cell",
    1: "Naive_B_cell",
    2: "Stressed_B_cell",
    3: "Proliferating_B_cell",
    4: "Follicular_B_cell",
    5: "Myeloid_contamination",
    6: "Plasma_cell"
}

B.obs['bcell_subtype'] = B.obs['leiden_res_0.25'].map(cluster_to_bcell_subtype)
sc.pl.umap(
    B,
    color='bcell_subtype',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
B = B[B.obs['bcell_subtype'] != 'Myeloid_contamination'].copy()

In [ ]:
import pandas as pd

# 1. leiden_res_0.10 컬럼을 문자열 타입으로 변환 (cat 필요)
B.obs['leiden_res_0.25'] = B.obs['bcell_subtype'].astype(str)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(B, groupby='bcell_subtype', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = B.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = B.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    B,
    var_names=top5_genes_per_cluster,
    groupby='bcell_subtype',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    B,
    color='bcell_subtype',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
bcell_representative_markers = {
    "Follicular_B_cell": ['Cd79a', 'Cd79b', 'Ighd', 'Cd74', 'Fcrla'],
    "MZ_like_B_cell": ['Mzb1', 'Apoe', 'Cd9', 'Plac8', 'Bhlhe41'],
    "Naive_B_cell": ['Ly6d', 'Myc', 'Tmsb10', 'Egr1', 'Nr4a1'],
    "Plasma_cell": ['Jchain', 'Igkc', 'Xbp1', 'Slamf7', 'Pdia4'],
    "Proliferating_B_cell": ['Ncl', 'Gnl3', 'Mphosph10', 'Ptma', 'Mat2a'],
    "Stressed_B_cell": ['Malat1', 'mt-Co1', 'mt-Nd4', 'Xist', 'Gm42418']
}

existing_markers = {
    subtype: [gene for gene in genes if gene in B.var_names]
    for subtype, genes in bcell_representative_markers.items()
}

# dotplot 시각화
sc.pl.dotplot(
    B,
    var_names=existing_markers,
    groupby="bcell_subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(10, 4)
)

In [ ]:
B.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Bcell_subtype.h5ad")

In [ ]:
sc.tl.rank_genes_groups(B, 'bcell_subtype', method="wilcoxon", use_raw=False)
sc.pl.rank_genes_groups(B, n_genes=25, sharey=False)
sc.tl.dendrogram(B, groupby='bcell_subtype')
sc.pl.rank_genes_groups_dotplot(
    B,
    groupby="bcell_subtype",
    standard_scale="var",
    n_genes=5,
    use_raw=False
)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = B.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(B, groupby='bcell_subtype')
sc.pl.rank_genes_groups_dotplot(
        B, groupby="bcell_subtype", standard_scale="var",  n_genes=5, use_raw=False)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = B.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

In [ ]:
from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

In [ ]:
from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=5,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=13,
    ylabel_fontsize=13,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

# Subtyping Merge

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
import scanpy as sc
import pandas as pd

adata.obs['Final_annotation'] = 'Unknown'

# 3. 그대로 복사할 그룹
direct_copy = ['Erythroid lineage', 'Smooth muscle cells', 'Steroidogenic endocrine cells']
mask_direct = adata.obs['celltype_grouped'].isin(direct_copy)
adata.obs.loc[mask_direct, 'Final_annotation'] = adata.obs.loc[mask_direct, 'celltype_grouped']

# 4. Epithelial cells 처리
epi = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Epithelial_subtype.h5ad")
mask_epi = adata.obs['celltype_grouped'] == 'Epithelial cells'
adata.obs.loc[mask_epi, 'Final_annotation'] = epi.obs['epithelial_subtype'].reindex(adata.obs.index)[mask_epi]

# 5. Stromal / Fibroblasts 처리
fib = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Fibroblast_subtype.h5ad")
mask_fib = adata.obs['celltype_grouped'] == 'Stromal / Fibroblasts'
adata.obs.loc[mask_fib, 'Final_annotation'] = fib.obs['fibroblast_subtype'].reindex(adata.obs.index)[mask_fib]

# 6. Immune - Myeloid 처리
mye = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Myeloid_subtype.h5ad")
mask_mye = adata.obs['celltype_grouped'] == 'Immune - Myeloid'
adata.obs.loc[mask_mye, 'Final_annotation'] = mye.obs['myeloid_subtype'].reindex(adata.obs.index)[mask_mye]

# 7. Immune - Lymphoid 처리
lym = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Lymphoid_subtype.h5ad")
mask_lym = adata.obs['celltype_grouped'] == 'Immune - Lymphoid'
adata.obs.loc[mask_lym, 'Final_annotation'] = lym.obs['lymphoid_subtype'].reindex(adata.obs.index)[mask_lym]

# 8. Immune - B cells / APCs 처리
bcell = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Bcell_subtype.h5ad")
mask_b = adata.obs['celltype_grouped'] == 'Immune - B cells / APCs'
adata.obs.loc[mask_b, 'Final_annotation'] = bcell.obs['bcell_subtype'].reindex(adata.obs.index)[mask_b]

# 9. 누락된 값 처리
adata.obs['Final_annotation'] = adata.obs['Final_annotation'].fillna('Unknown')


In [ ]:
sc.pl.umap(
    adata,
    color=["Final_annotation"],
    legend_loc="right margin",
)

import scanpy as sc

# 'Unknown'인 셀만 필터링
unknown_adata = adata[adata.obs['Final_annotation'] == 'Unknown'].copy()

# UMAP plot
sc.pl.umap(
    unknown_adata,
    color=["Final_annotation"],
    legend_loc='right margin'
)


In [ ]:
###################################################################
#######################Unknown annodation##########################
###################################################################

In [ ]:
import scanpy as sc

# 1. Before_HVG_raw.h5ad 불러오기
adata_raw = sc.read_h5ad("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Before_HVG_raw.h5ad")

# 2. unknown_adata의 인덱스와 교집합만 남기기
common_index = adata_raw.obs_names.intersection(unknown_adata.obs_names)

# 3. 필터링하여 새로운 객체로 할당
adata_raw = adata_raw[common_index].copy()

In [ ]:
adata_raw.layers["counts"] = adata_raw.X.copy()
sc.pp.normalize_total(adata_raw)
sc.pp.log1p(adata_raw)

sc.pp.highly_variable_genes(adata_raw, n_top_genes=5000, batch_key="sample_id")
sc.pl.highly_variable_genes(adata_raw)
adata_raw = adata_raw[:, adata_raw.var['highly_variable']].copy()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

sc.tl.pca(adata_raw)
explained_variance_ratio = adata_raw.uns["pca"]["variance_ratio"]

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker="o", linestyle="-")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Elbow Plot for PCA")
plt.xticks(np.arange(1, len(explained_variance_ratio) + 1, step=5))
plt.grid()
plt.show()

In [ ]:
sc.tl.pca(adata_raw, n_comps=26)

In [ ]:
sc.pl.pca(
    adata_raw,
    color=["sample_id", "pct_counts_mt"],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(adata_raw)
sc.tl.umap(adata_raw)
sc.pl.umap(
    adata_raw,
    color="sample_id",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
sc.pl.umap(adata_raw, color=["sample_id"])
sc.pl.umap(adata_raw, color=["organism part"])
sc.pl.umap(adata_raw, color=["clinical information"])
sc.pl.umap(adata_raw, color=["age"])

In [ ]:
for res in [0.05, 0.10,0.25,0.35]:
    sc.tl.leiden(
        adata_raw, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph"
    )
sc.pl.umap(adata_raw, color=["leiden_res_0.05"])
sc.pl.umap(adata_raw, color=["leiden_res_0.10"])
sc.pl.umap(adata_raw, color=["leiden_res_0.25"])
sc.pl.umap(adata_raw, color=["leiden_res_0.35"])

In [ ]:
import pandas as pd

# 1. leiden_res_0.10 컬럼을 문자열 타입으로 변환 (cat 필요)
adata_raw.obs['leiden_res_0.10'] = adata_raw.obs['leiden_res_0.10'].astype(str)

# 4. rank_genes_groups 분석 실행 (새 그룹 기준)
sc.tl.rank_genes_groups(adata_raw, groupby='leiden_res_0.10', method='t-test')

# 5. top genes 추출
top_genes = {}
groups = adata_raw.uns['rank_genes_groups']['names'].dtype.names
for group in groups:
    genes = adata_raw.uns['rank_genes_groups']['names'][group][:15]
    top_genes[group] = genes.tolist()
for cluster, genes in top_genes.items():
    print(f"Cluster {cluster}: {genes}")
# 6. top5 genes만 추출
top5_genes_per_cluster = {cluster: genes[:20] for cluster, genes in top_genes.items()}

# 7. dotplot 시각화
sc.pl.dotplot(
    adata_raw,
    var_names=top5_genes_per_cluster,
    groupby='leiden_res_0.10',
    standard_scale='var',
    dendrogram=False,
    figsize=(12, 6)
)

# 8. UMAP 시각화 (optional)
sc.pl.umap(
    adata_raw,
    color='leiden_res_0.10',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:
adata_raw.obs["leiden_res_0.10"] = adata_raw.obs["leiden_res_0.10"].astype(int)

cluster_to_annotation = {
    0: "Tph/Th17_like",
    1: "M2_macrophage",
    2: "Stressed_myeloid",  # or Stressed_myeloid 가능
    3: "Inflammatory_monocyte",
    4: "Naive_T/Tcm",
    5: "APC_myeloid",
    6: "APC_myeloid",  # cDC 포함
    7: "APC_myeloid",  # 유사 APC이지만 스트레스 유전자 많음
    8: "APC_myeloid"  # or plasmacytoid_DC_like (근거 부족)
}


adata_raw.obs['subtype'] = adata_raw.obs['leiden_res_0.10'].map(cluster_to_annotation)
sc.pl.umap(
    adata_raw,
    color='subtype',
    title='UMAP',
    legend_loc='on data'
)

In [ ]:

# 대표 마커 5개씩 지정 (중복되지 않게 필터링한 결과)
lymphoid_myeloid_representative_markers = {
    "APC_myeloid": ['Cd74', 'H2-Aa', 'H2-Eb1', 'Cd83', 'Il1b'],
    "Inflammatory_monocyte": ['Cxcl2', 'Ccl3', 'Fabp5', 'Cebpb', 'Il1rn'],
    "M2_macrophage": ['Apoe', 'C1qa', 'C1qb', 'Ms4a7', 'Csf1r'],
    "Naive_T/Tcm": ['Ccr7', 'Id2', 'Tbc1d4', 'Rel', 'Samsn1'],
    "Stressed_myeloid": ['Malat1', 'H2-Eb1', 'St8sia4', 'mt-Co1', 'mt-Nd1'],  # MT 유전자 추가
    "Tph/Th17_like": ['Cxcr6', 'Icos', 'Rora', 'Cd3d', 'Ikzf3'],

}

# 실제 존재하는 유전자만 필터링
existing_markers = {
    subtype: [gene for gene in genes if gene in adata_raw.var_names]
    for subtype, genes in lymphoid_myeloid_representative_markers.items()
}

# dotplot 시각화
sc.pl.dotplot(
    adata_raw,
    var_names=existing_markers,
    groupby="subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(8, 5)
)

In [ ]:

# 대표 마커 5개씩 지정 (중복되지 않게 필터링한 결과)
lymphoid_myeloid_representative_markers = {
    "APC_myeloid": ['Cd74', 'H2-Aa', 'Lgals3', 'Cd83', 'Lyz2'],
    "Inflammatory_monocyte": ['S100a9', 'S100a8', 'Cxcl2', 'G0s2', 'Slc7a11'],
    "M2_macrophage": ['Apoe', 'C1qa', 'Mrc1', 'C1qb', 'Selenop','Cd163'],
    "Naive_T/Tcm": ['Lef1', 'Ccr7', 'Satb1', 'Klf2', 'S1pr1'],
    "Stressed_myeloid": ['Malat1', 'H2-Eb1', 'St8sia4', 'mt-Co1', 'mt-Nd1'],  # MT 유전자 추가
    "Tph/Th17_like": ['Cxcr6', 'Icos', 'Rora', 'Il7r', 'Ikzf3'],

}

# 실제 존재하는 유전자만 필터링
existing_markers = {
    subtype: [gene for gene in genes if gene in adata_raw.var_names]
    for subtype, genes in lymphoid_myeloid_representative_markers.items()
}

# dotplot 시각화
sc.pl.dotplot(
    adata_raw,
    var_names=existing_markers,
    groupby="subtype",
    standard_scale="var",
    dendrogram=False,
    figsize=(8, 5)
)

In [ ]:
# 문자열 타입으로 변환
adata.obs['Final_annotation'] = adata.obs['Final_annotation'].astype(str)

# 인덱스 교집합 찾기
common_idx = adata_raw.obs_names.intersection(adata.obs_names[adata.obs['Final_annotation'] == 'Unknown'])

# 값 대입
adata.obs.loc[common_idx, 'Final_annotation'] = adata_raw.obs.loc[common_idx, 'subtype'].astype(str)


In [ ]:
import numpy as np

# 변경할 Final_annotation 조건별 매핑
mapping = {
    'APC_myeloid': 'Immune - Myeloid',
    'Inflammatory_monocyte': 'Immune - Myeloid',
    'M2_macrophage': 'Immune - Myeloid',
    'Naive_T/Tcm': 'Immune - Lymphoid',
    'Tph/Th17_like': 'Immune - Lymphoid'
}

# 조건에 맞는 인덱스 선택
mask = adata.obs['Final_annotation'].isin(mapping.keys())

# 해당 조건에 대해서만 celltype_grouped 값 변경
adata.obs.loc[mask, 'celltype_grouped'] = adata.obs.loc[mask, 'Final_annotation'].map(mapping)

In [ ]:
sc.pl.umap(
    adata,
    color='Final_annotation',
    title='UMAP',
    legend_loc='right margin'
)

In [ ]:
sc.pl.umap(
    adata,
    color='celltype_grouped',
    title='UMAP',
    legend_loc='right margin'
)

In [ ]:
# 각 celltype_grouped 별로 Final_annotation unique 값들을 리스트로 묶기
grouped = adata.obs.groupby('celltype_grouped')['Final_annotation'].unique().reset_index()

# 보기 편하게 출력
for _, row in grouped.iterrows():
    print(f"{row['celltype_grouped']}: {', '.join(row['Final_annotation'])}")


In [ ]:
adata.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Subtype_clustering.h5ad")

In [ ]:
# Steroidgenic endocrine cells -> Granulosa cells

In [ ]:
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Subtype_clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'] = adata.obs['celltype_grouped'].replace(
    {'Steroidogenic endocrine cells': 'Granulosa cells'}
)

In [ ]:
plt.figure(figsize=(12, 10)) 
sc.pl.umap(
    adata,
    color='Final_annotation',
    title='UMAP',
    legend_loc='right margin',
    s = 0.8

)

In [ ]:
adata.write("/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Subtype_clustering.h5ad")

# Subtype Statistic

In [ ]:
import scanpy as sc
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Subtype_clustering.h5ad"
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['celltype_grouped'].unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
adata.obs["age_num"] = adata.obs["age"].str.extract(r"(\d+)").astype(int)
# 각 mouse 별로 Final_annotation 개수 집계
count_df = adata.obs.groupby(['Each mouse', 'Final_annotation']).size().reset_index(name='counts')

# seaborn 가로 막대그래프로 시각화
plt.figure(figsize=(10, 12))  # y축이 길게 보이도록 height를 크게
sns.barplot(data=count_df, y='Each mouse', x='counts', hue='Final_annotation')

plt.yticks(rotation=0)
plt.ylabel('Each mouse')
plt.xlabel('Cell count')
plt.title('Cell counts per Final_annotation by Each mouse')
plt.legend(title='Final_annotation', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# subtype별 score (이전과 동일)
subtype_scores = {
    # Epithelial
    "Inflammatory_epithelial": {"inflammatory": 5, "immunosuppressive": 0},
    "Stressed_epithelial": {"inflammatory": 2, "immunosuppressive": 0},
    "Regenerative_epithelial": {"inflammatory": 1, "immunosuppressive": 0},
    "Basal_epithelial": {"inflammatory": 1, "immunosuppressive": 1},
    "Differentiated_epithelial": {"inflammatory": 0, "immunosuppressive": 0},
    "Ciliated_epithelial": {"inflammatory": 0, "immunosuppressive": 0},

    # Fibroblast
    "EMT_fibroblast": {"inflammatory": 3, "immunosuppressive": 0},
    "Stressed_fibroblast": {"inflammatory": 2, "immunosuppressive": 0},
    "KRT_fibroblast": {"inflammatory": 3, "immunosuppressive": 0},
    "Immunesupp_fibroblast": {"inflammatory": 0, "immunosuppressive": 5},
    "ECM_fibroblast": {"inflammatory": 0, "immunosuppressive": 1},
    "Metabolism_fibroblast": {"inflammatory": 1, "immunosuppressive": 1},
    "Vascular_fibroblast": {"inflammatory": 0, "immunosuppressive": 1},

    # Myeloid
    "M1_macrophage": {"inflammatory": 5, "immunosuppressive": 0},
    "Inflammatory_monocyte": {"inflammatory": 5, "immunosuppressive": 0},
    "APC_myeloid": {"inflammatory": 1, "immunosuppressive": 0},
    "Tissue_remodeling_myeloid": {"inflammatory": 1, "immunosuppressive": 1},
    "M2_macrophage": {"inflammatory": 0, "immunosuppressive": 6},
    "LAM_macrophage": {"inflammatory": 0, "immunosuppressive": 4},

    # Lymphoid
    "Effector_CD8_T": {"inflammatory": 4, "immunosuppressive": 0},
    "NK/NKT_cell": {"inflammatory": 3, "immunosuppressive": 0},
    "Tph/Th17_like": {"inflammatory": 3, "immunosuppressive": 0},
    "Th2_like": {"inflammatory": 1, "immunosuppressive": 1},
    "Treg": {"inflammatory": 0, "immunosuppressive": 4},
    "Mast_cell": {"inflammatory": 1, "immunosuppressive": 0},
    "Naive_T/Tcm": {"inflammatory": 0, "immunosuppressive": 0},
}

# subtype → major cell type 매핑
subtype_to_category = {
    # Epithelial
    "Inflammatory_epithelial": "Epithelial",
    "Stressed_epithelial": "Epithelial",
    "Regenerative_epithelial": "Epithelial",
    "Basal_epithelial": "Epithelial",
    "Differentiated_epithelial": "Epithelial",
    "Ciliated_epithelial": "Epithelial",

    # Fibroblast
    "EMT_fibroblast": "Fibroblast",
    "Stressed_fibroblast": "Fibroblast",
    "KRT_fibroblast": "Fibroblast",
    "Immunesupp_fibroblast": "Fibroblast",
    "ECM_fibroblast": "Fibroblast",
    "Metabolism_fibroblast": "Fibroblast",
    "Vascular_fibroblast": "Fibroblast",

    # Myeloid
    "M1_macrophage": "Myeloid",
    "Inflammatory_monocyte": "Myeloid",
    "APC_myeloid": "Myeloid",
    "Tissue_remodeling_myeloid": "Myeloid",
    "M2_macrophage": "Myeloid",
    "LAM_macrophage": "Myeloid",

    # Lymphoid
    "Effector_CD8_T": "Lymphoid",
    "NK/NKT_cell": "Lymphoid",
    "Tph/Th17_like": "Lymphoid",
    "Th2_like": "Lymphoid",
    "Treg": "Lymphoid",
    "Mast_cell": "Lymphoid",
    "Naive_T/Tcm": "Lymphoid",
}

# 기준 설정
INFLAM_THRESHOLD = 2
IMMUNE_SUPPRESS_THRESHOLD = 2

# 결과 dict 초기화
inflammatory_by_type = {"Epithelial": [], "Fibroblast": [], "Myeloid": [], "Lymphoid": []}
immunosuppressive_by_type = {"Epithelial": [], "Fibroblast": [], "Myeloid": [], "Lymphoid": []}

# 필터링 및 분류
for subtype, scores in subtype_scores.items():
    category = subtype_to_category.get(subtype)
    if scores["inflammatory"] >= INFLAM_THRESHOLD:
        inflammatory_by_type[category].append(subtype)
    if scores["immunosuppressive"] >= IMMUNE_SUPPRESS_THRESHOLD:
        immunosuppressive_by_type[category].append(subtype)

# 출력
print("🔥 Inflammatory subtypes (score ≥ 2):")
for cat, subtypes in inflammatory_by_type.items():
    print(f"  {cat}: {subtypes}")

print("\n🧊 Immunosuppressive subtypes (score ≥ 2):")
for cat, subtypes in immunosuppressive_by_type.items():
    print(f"  {cat}: {subtypes}")


In [ ]:
adata = adata[adata.obs['clinical information'] != 'decidualized_pregnant'].copy()
adata.obs['clinical information'].value_counts()

In [ ]:
import numpy as np
adata.obs['age_group'] = np.where(adata.obs['age_num'] > 70, 'Aged', 'Young')

In [ ]:

# 필요한 열만 선택
cols_to_keep = [
    "organism part",
    "clinical information",
    "Each mouse",
    "celltype_grouped",
    "Final_annotation",
    "age_num",
    'age_group'
]

# 필터링된 obs DataFrame 생성
obs_df = adata.obs[cols_to_keep].copy()

# 조건별로 객체 할당
Epithelial = obs_df[obs_df["celltype_grouped"] == "Epithelial cells"]
Lymphoid = obs_df[obs_df["celltype_grouped"] == "Immune - Lymphoid"]
Fibroblast = obs_df[obs_df["celltype_grouped"] == "Stromal / Fibroblasts"]
Myeloid = obs_df[obs_df["celltype_grouped"] == "Immune - Myeloid"]
B_cells = obs_df[obs_df["celltype_grouped"] == "Immune - B cells / APCs"]
Endocrine = obs_df[obs_df["celltype_grouped"] == "Granulosa cells"]

# (선택) 개수 확인 출력
print("Epithelial:", len(Epithelial))
print("Lymphoid:", len(Lymphoid))
print("Fibroblast:", len(Fibroblast))
print("Myeloid:", len(Myeloid))
print("B cells:", len(B_cells))
print("Endocrine:", len(Endocrine))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Epithelial 데이터 기반 비율 계산 ---
organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]

age_sorted = sorted(Epithelial['age_num'].unique())
target_subtypes = ['Inflammatory_epithelial', 'Stressed_epithelial']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Epithelial[(Epithelial["organism part"] == part) & (Epithelial["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]

age_sorted = sorted(Fibroblast['age_num'].unique())
target_subtypes = ['EMT_fibroblast', 'Stressed_fibroblast','KRT_fibroblast']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Fibroblast[(Fibroblast["organism part"] == part) & (Fibroblast["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]

age_sorted = sorted(Fibroblast['age_num'].unique())
target_subtypes = ['Immunesupp_fibroblast']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Fibroblast[(Fibroblast["organism part"] == part) & (Fibroblast["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]

age_sorted = sorted(Myeloid['age_num'].unique())
target_subtypes = ['M1_macrophage', 'Inflammatory_monocyte']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Myeloid[(Myeloid["organism part"] == part) & (Myeloid["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]

age_sorted = sorted(Myeloid['age_num'].unique())
target_subtypes = ['M2_macrophage', 'LAM_macrophage']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Myeloid[(Myeloid["organism part"] == part) & (Myeloid["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]


age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = ['Effector_CD8_T', 'NK/NKT_cell', 'Tph/Th17_like']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Lymphoid[(Lymphoid["organism part"] == part) & (Lymphoid["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

organ_parts = ["ovary", "oviduct", "uterus","uterine cervix", "vagina", "spleen"]
cycle_stages = ["proestrus", "estrus", "metestrus", "diestrus"]
y_labels = [f"{part} - {stage}" for part in organ_parts for stage in cycle_stages]


age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = ['Treg']
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

for part in organ_parts:
    for stage in cycle_stages:
        subset = Lymphoid[(Lymphoid["organism part"] == part) & (Lymphoid["clinical information"] == stage)]
        row_label = f"{part} - {stage}"
        for age in age_sorted:
            subset_age = subset[subset["age_num"] == age]
            for subtype in target_subtypes:
                heatmap_df.loc[row_label, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 그리기 ---
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 4.0, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.5, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축 tick 설정
yticks_locs = np.arange(len(y_labels)) + 0.5
cycle_stage_labels = cycle_stages * len(organ_parts)
ax.set_yticks(yticks_locs)
ax.set_yticklabels(cycle_stage_labels, rotation=0, fontsize=10)

# organism part label 왼쪽에 표시
organism_part_positions = []
for i in range(len(organ_parts)):
    pos = i * len(cycle_stages) + len(cycle_stages) / 2
    organism_part_positions.append(pos)

for pos, label in zip(organism_part_positions, organ_parts):
    ax.text(
        -1.5, pos, label,
        va='center', ha='right',
        fontsize=12, fontweight='bold',
        transform=ax.transData
    )
for i in range(1, len(organ_parts)):
    y = i * len(cycle_stages)  # 4씩 증가
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='--', linewidth=1)

# ❌ x축 전체 라벨 제거
ax.set_xlabel("")  # 또는 생략 가능

plt.title("Age (Week)", pad=40)
plt.tight_layout()
plt.show()

In [ ]:
#############################################################################################################################

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Myeloid['age_num'].unique())
target_subtypes = ['M1_macrophage', 'Inflammatory_monocyte']

# y축: organism part만 사용
y_labels = organ_parts
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 비율 계산 ---
for part in organ_parts:
    subset = Myeloid[Myeloid["organism part"] == part]  # cycle stage 무시
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            heatmap_df.loc[part, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 ---
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# ⬇️ 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 0.6, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# ⬆️ 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.25, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선 (빨간색 점선)
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축: organism part tick
yticks_locs = np.arange(len(y_labels)) + 0.5
ax.set_yticks(yticks_locs)
ax.set_yticklabels(organ_parts, rotation=0, fontsize=11, fontweight='bold')

# 🔷 organism part 사이 파랑 실선
for i in range(1, len(organ_parts)):
    y = i
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='-', linewidth=1)

# ❌ x축 이름 제거
ax.set_xlabel("")

# 제목
plt.title("Age(Week)", pad=30)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Epithelial['age_num'].unique())
target_subtypes = ['Inflammatory_epithelial', 'Stressed_epithelial']

# y축: organism part만 사용
y_labels = organ_parts
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 비율 계산 ---
for part in organ_parts:
    subset = Epithelial[Epithelial["organism part"] == part]  # cycle stage 무시
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            heatmap_df.loc[part, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 ---
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# ⬇️ 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 0.6, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# ⬆️ 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.25, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선 (빨간색 점선)
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축: organism part tick
yticks_locs = np.arange(len(y_labels)) + 0.5
ax.set_yticks(yticks_locs)
ax.set_yticklabels(organ_parts, rotation=0, fontsize=11, fontweight='bold')

# 🔷 organism part 사이 파랑 실선
for i in range(1, len(organ_parts)):
    y = i
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='-', linewidth=1)

# ❌ x축 이름 제거
ax.set_xlabel("")

# 제목
plt.title("Age(Week)", pad=30)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Fibroblast['age_num'].unique())
target_subtypes = ['EMT_fibroblast', 'Stressed_fibroblast','KRT_fibroblast']

# y축: organism part만 사용
y_labels = organ_parts
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 비율 계산 ---
for part in organ_parts:
    subset = Fibroblast[Fibroblast["organism part"] == part]  # cycle stage 무시
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            heatmap_df.loc[part, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 ---
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# ⬇️ 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 0.6, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# ⬆️ 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.25, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선 (빨간색 점선)
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축: organism part tick
yticks_locs = np.arange(len(y_labels)) + 0.5
ax.set_yticks(yticks_locs)
ax.set_yticklabels(organ_parts, rotation=0, fontsize=11, fontweight='bold')

# 🔷 organism part 사이 파랑 실선
for i in range(1, len(organ_parts)):
    y = i
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='-', linewidth=1)

# ❌ x축 이름 제거
ax.set_xlabel("")

# 제목
plt.title("Age(Week)", pad=30)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Myeloid['age_num'].unique())
target_subtypes = ['M2_macrophage', 'LAM_macrophage']

# y축: organism part만 사용
y_labels = organ_parts
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 비율 계산 ---
for part in organ_parts:
    subset = Myeloid[Myeloid["organism part"] == part]  # cycle stage 무시
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            heatmap_df.loc[part, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 ---
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# ⬇️ 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 0.6, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# ⬆️ 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.25, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선 (빨간색 점선)
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축: organism part tick
yticks_locs = np.arange(len(y_labels)) + 0.5
ax.set_yticks(yticks_locs)
ax.set_yticklabels(organ_parts, rotation=0, fontsize=11, fontweight='bold')

# 🔷 organism part 사이 파랑 실선
for i in range(1, len(organ_parts)):
    y = i
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='-', linewidth=1)

# ❌ x축 이름 제거
ax.set_xlabel("")

# 제목
plt.title("Age(Week)", pad=30)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = ['Effector_CD8_T', 'NK/NKT_cell', 'Tph/Th17_like']

# y축: organism part만 사용
y_labels = organ_parts
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 비율 계산 ---
for part in organ_parts:
    subset = Lymphoid[Lymphoid["organism part"] == part]  # cycle stage 무시
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            heatmap_df.loc[part, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 ---
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# ⬇️ 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 0.6, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# ⬆️ 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.25, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선 (빨간색 점선)
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축: organism part tick
yticks_locs = np.arange(len(y_labels)) + 0.5
ax.set_yticks(yticks_locs)
ax.set_yticklabels(organ_parts, rotation=0, fontsize=11, fontweight='bold')

# 🔷 organism part 사이 파랑 실선
for i in range(1, len(organ_parts)):
    y = i
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='-', linewidth=1)

# ❌ x축 이름 제거
ax.set_xlabel("")

# 제목
plt.title("Age(Week)", pad=30)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = ['Effector_CD8_T', 'NK/NKT_cell', 'Tph/Th17_like']

# 빈 리스트 생성 (결과 저장용)
rows = []

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 각 organ part, age, subtype 별 비율 계산 ---
for part in organ_parts:
    subset = Lymphoid[Lymphoid["organism part"] == part]
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            prop = calc_proportion(subset_age, subtype)
            rows.append({
                "organ_part": part,
                "age": age,
                "subtype": subtype,
                "proportion": prop
            })

# 리스트를 DataFrame으로 변환
df_trend = pd.DataFrame(rows)

# --- 라인플롯 ---
sns.set(style="whitegrid")
g = sns.FacetGrid(df_trend, col="organ_part", col_wrap=3, height=4, sharey=False)
g.map_dataframe(
    sns.lineplot,
    x="age", y="proportion", hue="subtype", marker="o"
)
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Age (Week)", "Proportion")
g.add_legend(title="Subtype")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = ['Effector_CD8_T', 'NK/NKT_cell', 'Tph/Th17_like']

# 빈 리스트 생성 (결과 저장용)
rows = []

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 각 organ part, age, subtype 별 비율 계산 ---
for part in organ_parts:
    subset = Lymphoid[Lymphoid["organism part"] == part]
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            prop = calc_proportion(subset_age, subtype)
            rows.append({
                "organ_part": part,
                "age": age,
                "subtype": subtype,
                "proportion": prop
            })

# 리스트를 DataFrame으로 변환
df_trend = pd.DataFrame(rows)

# Age 그룹 추가
df_trend['age_group'] = np.where(df_trend['age'] > 70, 'Aged', 'Young')

# --- 박스플롯 + t-test 결과 표시 ---
sns.set(style="whitegrid")

fig, axes = plt.subplots(len(organ_parts), len(target_subtypes), figsize=(15, 4 * len(organ_parts)), sharey='row')

for i, organ in enumerate(organ_parts):
    for j, subtype in enumerate(target_subtypes):
        ax = axes[i, j] if len(organ_parts) > 1 else axes[j]
        data = df_trend[(df_trend['organ_part'] == organ) & (df_trend['subtype'] == subtype)]

        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax)
        sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=4, jitter=True, ax=ax)

        # t-test
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = ttest_ind(group1, group2, equal_var=False)
            p_text = f"p = {pval:.3e}"
        else:
            p_text = "n/a"

        # p-value 텍스트 추가
        ax.text(0.5, max(data['proportion'].dropna()) * 1.05, p_text,
                ha='center', va='bottom', fontsize=10, color='red', transform=ax.get_xaxis_transform())

        ax.set_title(f"{organ} - {subtype}")
        ax.set_xlabel("")
        ax.set_ylabel("Proportion")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = sorted(Lymphoid['Final_annotation'].dropna().unique())

# 빈 리스트 생성 (결과 저장용)
rows = []

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 각 organ part, age, subtype 별 비율 계산 ---
for part in organ_parts:
    subset = Lymphoid[Lymphoid["organism part"] == part]
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            prop = calc_proportion(subset_age, subtype)
            rows.append({
                "organ_part": part,
                "age": age,
                "subtype": subtype,
                "proportion": prop
            })

# 리스트를 DataFrame으로 변환
df_trend = pd.DataFrame(rows)
df_trend['age_group'] = np.where(df_trend['age'] > 70, 'Aged', 'Young')

# --- 각 organ part 별로 따로 plot ---
sns.set(style="whitegrid")

for organ in organ_parts:
    data_part = df_trend[df_trend['organ_part'] == organ]
    n_subtypes = len(target_subtypes)

    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]  # ensure axes is iterable

    for j, subtype in enumerate(target_subtypes):
        ax = axes[j]
        data = data_part[data_part['subtype'] == subtype]

        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)
        sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=4, jitter=True, ax=ax)

        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = ttest_ind(group1, group2, equal_var=False)
            p_text = f"p = {pval:.3e}"
        else:
            p_text = "n/a"

        ax.text(0.5, max(data['proportion'].dropna()) * 1.05 if len(data['proportion'].dropna()) > 0 else 0.1,
                p_text, ha='center', va='bottom', fontsize=10, color='red', transform=ax.get_xaxis_transform())

        ax.set_title(subtype)
        ax.set_xlabel("")
        if j == 0:
            ax.set_ylabel("Proportion")
        else:
            ax.set_ylabel("")

    plt.suptitle(f"{organ}", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Fibroblast['age_num'].unique())
target_subtypes = sorted(Fibroblast['Final_annotation'].dropna().unique())

# 빈 리스트 생성 (결과 저장용)
rows = []

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 각 organ part, age, subtype 별 비율 계산 ---
for part in organ_parts:
    subset = Fibroblast[Fibroblast["organism part"] == part]
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            prop = calc_proportion(subset_age, subtype)
            rows.append({
                "organ_part": part,
                "age": age,
                "subtype": subtype,
                "proportion": prop
            })

# 리스트를 DataFrame으로 변환
df_trend = pd.DataFrame(rows)
df_trend['age_group'] = np.where(df_trend['age'] > 70, 'Aged', 'Young')

# --- 각 organ part 별로 따로 plot ---
sns.set(style="whitegrid")

for organ in organ_parts:
    data_part = df_trend[df_trend['organ_part'] == organ]
    n_subtypes = len(target_subtypes)

    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]  # ensure axes is iterable

    for j, subtype in enumerate(target_subtypes):
        ax = axes[j]
        data = data_part[data_part['subtype'] == subtype]

        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)
        sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=4, jitter=True, ax=ax)

        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = ttest_ind(group1, group2, equal_var=False)
            p_text = f"p = {pval:.3e}"
        else:
            p_text = "n/a"

        ax.text(0.5, max(data['proportion'].dropna()) * 1.05 if len(data['proportion'].dropna()) > 0 else 0.1,
                p_text, ha='center', va='bottom', fontsize=10, color='red', transform=ax.get_xaxis_transform())

        ax.set_title(subtype)
        ax.set_xlabel("")
        if j == 0:
            ax.set_ylabel("Proportion")
        else:
            ax.set_ylabel("")

    plt.suptitle(f"{organ}", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Myeloid['age_num'].unique())
target_subtypes = sorted(Myeloid['Final_annotation'].dropna().unique())

# 빈 리스트 생성 (결과 저장용)
rows = []

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 각 organ part, age, subtype 별 비율 계산 ---
for part in organ_parts:
    subset = Myeloid[Myeloid["organism part"] == part]
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            prop = calc_proportion(subset_age, subtype)
            rows.append({
                "organ_part": part,
                "age": age,
                "subtype": subtype,
                "proportion": prop
            })

# 리스트를 DataFrame으로 변환
df_trend = pd.DataFrame(rows)
df_trend['age_group'] = np.where(df_trend['age'] > 70, 'Aged', 'Young')

# --- 각 organ part 별로 따로 plot ---
sns.set(style="whitegrid")

for organ in organ_parts:
    data_part = df_trend[df_trend['organ_part'] == organ]
    n_subtypes = len(target_subtypes)

    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]  # ensure axes is iterable

    for j, subtype in enumerate(target_subtypes):
        ax = axes[j]
        data = data_part[data_part['subtype'] == subtype]

        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)
        sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=4, jitter=True, ax=ax)

        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = ttest_ind(group1, group2, equal_var=False)
            p_text = f"p = {pval:.3e}"
        else:
            p_text = "n/a"

        ax.text(0.5, max(data['proportion'].dropna()) * 1.05 if len(data['proportion'].dropna()) > 0 else 0.1,
                p_text, ha='center', va='bottom', fontsize=10, color='red', transform=ax.get_xaxis_transform())

        ax.set_title(subtype)
        ax.set_xlabel("")
        if j == 0:
            ax.set_ylabel("Proportion")
        else:
            ax.set_ylabel("")

    plt.suptitle(f"{organ}", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Epithelial['age_num'].unique())
target_subtypes = sorted(Epithelial['Final_annotation'].dropna().unique())

# 빈 리스트 생성 (결과 저장용)
rows = []

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 각 organ part, age, subtype 별 비율 계산 ---
for part in organ_parts:
    subset = Epithelial[Epithelial["organism part"] == part]
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            prop = calc_proportion(subset_age, subtype)
            rows.append({
                "organ_part": part,
                "age": age,
                "subtype": subtype,
                "proportion": prop
            })

# 리스트를 DataFrame으로 변환
df_trend = pd.DataFrame(rows)
df_trend['age_group'] = np.where(df_trend['age'] > 70, 'Aged', 'Young')

# --- 각 organ part 별로 따로 plot ---
sns.set(style="whitegrid")

for organ in organ_parts:
    data_part = df_trend[df_trend['organ_part'] == organ]
    n_subtypes = len(target_subtypes)

    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]  # ensure axes is iterable

    for j, subtype in enumerate(target_subtypes):
        ax = axes[j]
        data = data_part[data_part['subtype'] == subtype]

        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)
        sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=4, jitter=True, ax=ax)

        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = ttest_ind(group1, group2, equal_var=False)
            p_text = f"p = {pval:.3e}"
        else:
            p_text = "n/a"

        ax.text(0.5, max(data['proportion'].dropna()) * 1.05 if len(data['proportion'].dropna()) > 0 else 0.1,
                p_text, ha='center', va='bottom', fontsize=10, color='red', transform=ax.get_xaxis_transform())

        ax.set_title(subtype)
        ax.set_xlabel("")
        if j == 0:
            ax.set_ylabel("Proportion")
        else:
            ax.set_ylabel("")

    plt.suptitle(f"{organ}", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 설정 ---
organ_parts = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
age_sorted = sorted(Lymphoid['age_num'].unique())
target_subtypes = ['Treg']

# y축: organism part만 사용
y_labels = organ_parts
columns = pd.MultiIndex.from_product([age_sorted, target_subtypes], names=["Age", "Subtype"])
heatmap_df = pd.DataFrame(index=y_labels, columns=columns, dtype=float)

# --- 비율 계산 함수 ---
def calc_proportion(df, subtype):
    if len(df) == 0:
        return np.nan
    return (df["Final_annotation"] == subtype).sum() / len(df)

# --- 비율 계산 ---
for part in organ_parts:
    subset = Lymphoid[Lymphoid["organism part"] == part]  # cycle stage 무시
    for age in age_sorted:
        subset_age = subset[subset["age_num"] == age]
        for subtype in target_subtypes:
            heatmap_df.loc[part, (age, subtype)] = calc_proportion(subset_age, subtype)

# --- 히트맵 ---
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor='gray',
    cbar_kws={"label": "Proportion"},
    ax=ax,
    xticklabels=False,
    yticklabels=False
)

# x축 기본 라벨 제거
ax.set_xticks([])

# x축 위치 계산
xticks = np.arange(len(heatmap_df.columns)) + 0.5
subtype_labels = [sub for age in age_sorted for sub in target_subtypes]

# ⬇️ 아래쪽: Subtype
for x, subtype in zip(xticks, subtype_labels):
    ax.text(
        x, len(y_labels) + 0.6, subtype,
        ha='right', va='bottom',
        rotation=90,
        fontsize=9.2,
        transform=ax.transData
    )

# ⬆️ 위쪽: Age
step = len(target_subtypes)
for i, age in enumerate(age_sorted):
    center_pos = i * step + step / 2
    ax.text(
        center_pos, -0.25, str(age),
        ha='center', va='top',
        fontsize=11,
        fontweight='bold',
        transform=ax.transData
    )

# 🔺 Age 경계선 (빨간색 점선)
for i in range(1, len(age_sorted)):
    ax.axvline(x=i * step, color='red', linestyle='--', linewidth=1)

# y축: organism part tick
yticks_locs = np.arange(len(y_labels)) + 0.5
ax.set_yticks(yticks_locs)
ax.set_yticklabels(organ_parts, rotation=0, fontsize=11, fontweight='bold')

# 🔷 organism part 사이 파랑 실선
for i in range(1, len(organ_parts)):
    y = i
    ax.hlines(y=y, xmin=0, xmax=len(heatmap_df.columns), colors='blue', linestyles='-', linewidth=1)

# ❌ x축 이름 제거
ax.set_xlabel("")

# 제목
plt.title("Age(Week)", pad=30)
plt.tight_layout()
plt.show()


# Upper & Lower (aged group base)

In [ ]:
##################Fibroblast#################

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Fibroblast  # 예: Fibroblast, Granulosa, Immune 등 원하는 데이터프레임 넣기

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 고정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- 특정 mouse 제외용 dict ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- 전체 subplot 크기 정의 (organs × subtypes) ---
n_organs = len(organs)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, organ in enumerate(organs):
    data_organ = data[data['organism part'] == organ].copy()

    # 특정 organ에서 제외할 mouse 필터링
    excluded_mice = mice_to_filter_by_organ.get(organ, [])
    data_organ = data_organ[~data_organ['Each mouse'].isin(excluded_mice)]

    # mouse 단위 proportion 계산
    mouse_props = (
        data_organ.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals,
                           on=['Each mouse', 'age_group'],
                           how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # dot 먼저
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # boxplot
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
            whiskerprops={'color': 'black'},
            medianprops={'color': 'red'},
            zorder=1
        )

        # 통계 (Mann–Whitney U)
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # 라벨 및 제목
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{organ}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")

        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# 전체 제목
plt.suptitle("Fibroblast Subtype – Each Organ", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Fibroblast  # 원하는 세포 타입
celltype_name = "Fibroblast"  # 제목용 문자열

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())
organ_groups = ["upper", "lower", "control"]

# --- 전체 subplot 크기 정의 (organ_groups × subtypes) ---
n_organs = len(organ_groups)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, og in enumerate(organ_groups):
    data_group = data[data['organ_group'] == og].copy()
    if data_group.empty:
        continue

    # mouse 단위 proportion 계산
    mouse_props = (
        data_group.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_group.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # --- dot 먼저 ---
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # --- boxplot ---
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
            whiskerprops={'color':'black'},
            medianprops={'color':'red'},
            zorder=1
        )

        # --- 통계: Mann–Whitney U ---
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001: sig = "****"
            elif pval < 0.001: sig = "***"
            elif pval < 0.01: sig = "**"
            elif pval < 0.05: sig = "*"
            else: sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{og}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")
        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# --- 전체 제목 (celltype_df에 맞게 f-string 처리) ---
plt.suptitle(f"{celltype_name} Subtype – Organ (upper & lower)", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Fibroblast  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'Final_annotation', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_sub,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_sub, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_sub_clean = data_sub.dropna(subset=['proportion'])
    group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
    group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Total - Fibroblast Subtype", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Fibroblast  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target organs 정의 (subplots 순서) ---
target_organs = sorted(data['organism part'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'organism part', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_organs = len(target_organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(target_organs):
    ax = axes[i]
    data_org = mouse_props[mouse_props['organism part'] == organ]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_org,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_org, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_org_clean = data_org.dropna(subset=['proportion'])
    group1 = data_org_clean[data_org_clean['age_group'] == 'Young']['proportion']
    group2 = data_org_clean[data_org_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(organ, fontsize=25)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_org['proportion'].max() if not data_org.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Organ-wise Aged vs Young", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
##################Epithelial#################

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Epithelial  # 예: Fibroblast, Granulosa, Immune 등 원하는 데이터프레임 넣기

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 고정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- 특정 mouse 제외용 dict ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- 전체 subplot 크기 정의 (organs × subtypes) ---
n_organs = len(organs)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, organ in enumerate(organs):
    data_organ = data[data['organism part'] == organ].copy()

    # 특정 organ에서 제외할 mouse 필터링
    excluded_mice = mice_to_filter_by_organ.get(organ, [])
    data_organ = data_organ[~data_organ['Each mouse'].isin(excluded_mice)]

    # mouse 단위 proportion 계산
    mouse_props = (
        data_organ.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals,
                           on=['Each mouse', 'age_group'],
                           how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # dot 먼저
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # boxplot
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
            whiskerprops={'color': 'black'},
            medianprops={'color': 'red'},
            zorder=1
        )

        # 통계 (Mann–Whitney U)
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # 라벨 및 제목
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{organ}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")

        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# 전체 제목
plt.suptitle("Epithelial Subtype – Each Organ", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Epithelial  # 원하는 세포 타입
celltype_name = "Epithelial"  # 제목용 문자열

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())
organ_groups = ["upper", "lower", "control"]

# --- 전체 subplot 크기 정의 (organ_groups × subtypes) ---
n_organs = len(organ_groups)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, og in enumerate(organ_groups):
    data_group = data[data['organ_group'] == og].copy()
    if data_group.empty:
        continue

    # mouse 단위 proportion 계산
    mouse_props = (
        data_group.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_group.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # --- dot 먼저 ---
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # --- boxplot ---
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
            whiskerprops={'color':'black'},
            medianprops={'color':'red'},
            zorder=1
        )

        # --- 통계: Mann–Whitney U ---
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001: sig = "****"
            elif pval < 0.001: sig = "***"
            elif pval < 0.01: sig = "**"
            elif pval < 0.05: sig = "*"
            else: sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{og}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")
        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# --- 전체 제목 (celltype_df에 맞게 f-string 처리) ---
plt.suptitle(f"{celltype_name} Subtype – Organ (upper & lower)", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Epithelial  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'Final_annotation', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_sub,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_sub, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_sub_clean = data_sub.dropna(subset=['proportion'])
    group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
    group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion per mouse (normalized per age group)", fontsize=12)
    else:
        ax.set_ylabel("")
    y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Total - Epithelial Subtype", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Epithelial  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target organs 정의 (subplots 순서) ---
target_organs = sorted(data['organism part'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'organism part', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_organs = len(target_organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(target_organs):
    ax = axes[i]
    data_org = mouse_props[mouse_props['organism part'] == organ]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_org,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_org, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_org_clean = data_org.dropna(subset=['proportion'])
    group1 = data_org_clean[data_org_clean['age_group'] == 'Young']['proportion']
    group2 = data_org_clean[data_org_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(organ, fontsize=25)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion per mouse (normalized per age group)", fontsize=12)
    else:
        ax.set_ylabel("")
    y_max = data_org['proportion'].max() if not data_org.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Organ-wise Aged vs Young", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
##################Myeloid#################

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Myeloid  # 예: Fibroblast, Granulosa, Immune 등 원하는 데이터프레임 넣기

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 고정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- 특정 mouse 제외용 dict ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- 전체 subplot 크기 정의 (organs × subtypes) ---
n_organs = len(organs)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, organ in enumerate(organs):
    data_organ = data[data['organism part'] == organ].copy()

    # 특정 organ에서 제외할 mouse 필터링
    excluded_mice = mice_to_filter_by_organ.get(organ, [])
    data_organ = data_organ[~data_organ['Each mouse'].isin(excluded_mice)]

    # mouse 단위 proportion 계산
    mouse_props = (
        data_organ.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals,
                           on=['Each mouse', 'age_group'],
                           how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # dot 먼저
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # boxplot
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
            whiskerprops={'color': 'black'},
            medianprops={'color': 'red'},
            zorder=1
        )

        # 통계 (Mann–Whitney U)
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # 라벨 및 제목
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{organ}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")

        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# 전체 제목
plt.suptitle("Myeloid Subtype – Each Organ", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Myeloid  # 원하는 세포 타입
celltype_name = "Myeloid"  # 제목용 문자열

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())
organ_groups = ["upper", "lower", "control"]

# --- 전체 subplot 크기 정의 (organ_groups × subtypes) ---
n_organs = len(organ_groups)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, og in enumerate(organ_groups):
    data_group = data[data['organ_group'] == og].copy()
    if data_group.empty:
        continue

    # mouse 단위 proportion 계산
    mouse_props = (
        data_group.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_group.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # --- dot 먼저 ---
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # --- boxplot ---
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
            whiskerprops={'color':'black'},
            medianprops={'color':'red'},
            zorder=1
        )

        # --- 통계: Mann–Whitney U ---
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001: sig = "****"
            elif pval < 0.001: sig = "***"
            elif pval < 0.01: sig = "**"
            elif pval < 0.05: sig = "*"
            else: sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{og}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")
        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# --- 전체 제목 (celltype_df에 맞게 f-string 처리) ---
plt.suptitle(f"{celltype_name} Subtype – Organ (upper & lower)", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Myeloid  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'Final_annotation', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_sub,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_sub, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_sub_clean = data_sub.dropna(subset=['proportion'])
    group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
    group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Total Myeloid Subtype", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Myeloid  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target organs 정의 (subplots 순서) ---
target_organs = sorted(data['organism part'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'organism part', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_organs = len(target_organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(target_organs):
    ax = axes[i]
    data_org = mouse_props[mouse_props['organism part'] == organ]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_org,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_org, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_org_clean = data_org.dropna(subset=['proportion'])
    group1 = data_org_clean[data_org_clean['age_group'] == 'Young']['proportion']
    group2 = data_org_clean[data_org_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(organ, fontsize=25)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_org['proportion'].max() if not data_org.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Organ-wise Aged vs Young", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
##################Lymphoid#################

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Lymphoid  # 예: Fibroblast, Granulosa, Immune 등 원하는 데이터프레임 넣기

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 고정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- 특정 mouse 제외용 dict ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- 전체 subplot 크기 정의 (organs × subtypes) ---
n_organs = len(organs)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, organ in enumerate(organs):
    data_organ = data[data['organism part'] == organ].copy()

    # 특정 organ에서 제외할 mouse 필터링
    excluded_mice = mice_to_filter_by_organ.get(organ, [])
    data_organ = data_organ[~data_organ['Each mouse'].isin(excluded_mice)]

    # mouse 단위 proportion 계산
    mouse_props = (
        data_organ.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals,
                           on=['Each mouse', 'age_group'],
                           how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # dot 먼저
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # boxplot
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
            whiskerprops={'color': 'black'},
            medianprops={'color': 'red'},
            zorder=1
        )

        # 통계 (Mann–Whitney U)
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # 라벨 및 제목
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{organ}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")

        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# 전체 제목
plt.suptitle("Lymphoid Subtype – Each Organ", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Lymphoid  # 원하는 세포 타입
celltype_name = "Lymphoid"  # 제목용 문자열

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())
organ_groups = ["upper", "lower", "control"]

# --- 전체 subplot 크기 정의 (organ_groups × subtypes) ---
n_organs = len(organ_groups)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, og in enumerate(organ_groups):
    data_group = data[data['organ_group'] == og].copy()
    if data_group.empty:
        continue

    # mouse 단위 proportion 계산
    mouse_props = (
        data_group.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_group.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # --- dot 먼저 ---
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # --- boxplot ---
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
            whiskerprops={'color':'black'},
            medianprops={'color':'red'},
            zorder=1
        )

        # --- 통계: Mann–Whitney U ---
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001: sig = "****"
            elif pval < 0.001: sig = "***"
            elif pval < 0.01: sig = "**"
            elif pval < 0.05: sig = "*"
            else: sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{og}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")
        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# --- 전체 제목 (celltype_df에 맞게 f-string 처리) ---
plt.suptitle(f"{celltype_name} Subtype – Organ (upper & lower)", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Lymphoid  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'Final_annotation', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_sub,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_sub, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_sub_clean = data_sub.dropna(subset=['proportion'])
    group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
    group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Total - Lymphoid Subtype", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Lymphoid  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target organs 정의 (subplots 순서) ---
target_organs = sorted(data['organism part'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'organism part', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_organs = len(target_organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(target_organs):
    ax = axes[i]
    data_org = mouse_props[mouse_props['organism part'] == organ]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_org,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_org, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_org_clean = data_org.dropna(subset=['proportion'])
    group1 = data_org_clean[data_org_clean['age_group'] == 'Young']['proportion']
    group2 = data_org_clean[data_org_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(organ, fontsize=25)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_org['proportion'].max() if not data_org.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Organ-wise Aged vs Young", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
##################B cells#################

In [ ]:
B_cells

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = B_cells  # 예: Fibroblast, Granulosa, Immune 등 원하는 데이터프레임 넣기

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 고정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- 특정 mouse 제외용 dict ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- 전체 subplot 크기 정의 (organs × subtypes) ---
n_organs = len(organs)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, organ in enumerate(organs):
    data_organ = data[data['organism part'] == organ].copy()

    # 특정 organ에서 제외할 mouse 필터링
    excluded_mice = mice_to_filter_by_organ.get(organ, [])
    data_organ = data_organ[~data_organ['Each mouse'].isin(excluded_mice)]

    # mouse 단위 proportion 계산
    mouse_props = (
        data_organ.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals,
                           on=['Each mouse', 'age_group'],
                           how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # dot 먼저
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # boxplot
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
            whiskerprops={'color': 'black'},
            medianprops={'color': 'red'},
            zorder=1
        )

        # 통계 (Mann–Whitney U)
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # 라벨 및 제목
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{organ}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")

        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# 전체 제목
plt.suptitle("B Cells Subtype – Each Organ", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = B_cells  # 원하는 세포 타입
celltype_name = "B cells"  # 제목용 문자열

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())
organ_groups = ["upper", "lower", "control"]

# --- 전체 subplot 크기 정의 (organ_groups × subtypes) ---
n_organs = len(organ_groups)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, og in enumerate(organ_groups):
    data_group = data[data['organ_group'] == og].copy()
    if data_group.empty:
        continue

    # mouse 단위 proportion 계산
    mouse_props = (
        data_group.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_group.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # --- dot 먼저 ---
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # --- boxplot ---
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
            whiskerprops={'color':'black'},
            medianprops={'color':'red'},
            zorder=1
        )

        # --- 통계: Mann–Whitney U ---
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001: sig = "****"
            elif pval < 0.001: sig = "***"
            elif pval < 0.01: sig = "**"
            elif pval < 0.05: sig = "*"
            else: sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{og}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")
        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# --- 전체 제목 (celltype_df에 맞게 f-string 처리) ---
plt.suptitle(f"{celltype_name} Subtype – Organ (upper & lower)", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = B_cells  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'Final_annotation', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_sub,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_sub, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_sub_clean = data_sub.dropna(subset=['proportion'])
    group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
    group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Total - B cells Subtype", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = B_cells  # 원하는 세포 타입

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target organs 정의 (subplots 순서) ---
target_organs = sorted(data['organism part'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'organism part', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_organs = len(target_organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(target_organs):
    ax = axes[i]
    data_org = mouse_props[mouse_props['organism part'] == organ]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_org,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_org, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_org_clean = data_org.dropna(subset=['proportion'])
    group1 = data_org_clean[data_org_clean['age_group'] == 'Young']['proportion']
    group2 = data_org_clean[data_org_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(organ, fontsize=25)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = data_org['proportion'].max() if not data_org.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Organ-wise Aged vs Young", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
################## Endocrine ################

In [ ]:
granulosa_df = Endocrine['Final_annotation'] = 'Granulosa'
granulosa_df

In [ ]:
Endocrine['Final_annotation'] = 'Granulosa cells'
Endocrine

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Endocrine.copy()
celltype_df['Final_annotation'] = 'Granulosa cells'  # 전체 덮어쓰기

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- 전체 subplot 가로 배치 ---
n_organs = len(organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

sns.set(style="whitegrid")

for i, organ in enumerate(organs):
    ax = axes[i]
    data_organ = data[data['organism part'] == organ].copy()
    if data_organ.empty:
        continue

    # --- mouse 단위 proportion 계산 ---
    mouse_props = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_organ.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']
    mouse_props = mouse_props.dropna(subset=['proportion'])

    # --- dot + boxplot ---
    sns.stripplot(x='age_group', y='proportion', data=mouse_props,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)
    sns.boxplot(x='age_group', y='proportion', data=mouse_props, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계 ---
    group1 = mouse_props[mouse_props['age_group'] == 'Young']['proportion']
    group2 = mouse_props[mouse_props['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title("Granulosa cells", fontsize=20)
    ax.set_xlabel("")
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=20)
    else:
        ax.set_ylabel("")
    y_max = mouse_props['proportion'].max() if not mouse_props.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=12)
    ax.set_title(organ, fontsize=18)
    ax.tick_params(axis='x', labelsize=25)

plt.suptitle("Granulosa cells – All Organs", fontsize=25, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = Endocrine.copy()
celltype_df['Final_annotation'] = 'Granulosa cells'  # 전체 덮어쓰기

# --- 데이터 복사 및 age_group 추가 ---
data = celltype_df.copy()
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target organs 정의 (subplots 순서) ---
target_organs = sorted(data['organism part'].dropna().unique())

# --- mouse 단위 count ---
mouse_props = (
    data.groupby(['Each mouse', 'organism part', 'age_group'])
    .size()
    .rename('count')
    .reset_index()
)

# --- mouse별 total per age_group ---
totals_per_age = (
    data.groupby(['age_group', 'Each mouse']).size()
    .rename('total')
    .reset_index()
)

# --- merge 후 proportion 계산 ---
mouse_props = pd.merge(mouse_props, totals_per_age, on=['Each mouse', 'age_group'], how='left')
mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

# --- 시각화 ---
sns.set(style="whitegrid")
n_organs = len(target_organs)
fig, axes = plt.subplots(1, n_organs, figsize=(4 * n_organs, 4), sharey=True)
if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(target_organs):
    ax = axes[i]
    data_org = mouse_props[mouse_props['organism part'] == organ]

    # --- dot 먼저 ---
    sns.stripplot(x='age_group', y='proportion', data=data_org,
                  color='black', size=6, jitter=True, ax=ax, zorder=2)

    # --- boxplot ---
    sns.boxplot(x='age_group', y='proportion', data=data_org, ax=ax,
                width=0.5, showcaps=True,
                boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
                whiskerprops={'color':'black'},
                medianprops={'color':'red'},
                zorder=1)

    # --- 통계: NaN 제거 후 Aged vs Young (Mann–Whitney U) ---
    data_org_clean = data_org.dropna(subset=['proportion'])
    group1 = data_org_clean[data_org_clean['age_group'] == 'Young']['proportion']
    group2 = data_org_clean[data_org_clean['age_group'] == 'Aged']['proportion']
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001: sig = "****"
        elif pval < 0.001: sig = "***"
        elif pval < 0.01: sig = "**"
        elif pval < 0.05: sig = "*"
        else: sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # --- 라벨 ---
    ax.set_title(organ, fontsize=25)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=20)
    if i == 0:
        ax.set_ylabel("Proportion", fontsize=12)
    else:
        ax.set_ylabel("")
    y_max = data_org['proportion'].max() if not data_org.empty else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=20, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("Organ-wise Aged vs Young", fontsize=35, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- 분석할 세포 타입 데이터 지정 ---
celltype_df = celltype_df  # 원하는 세포 타입
celltype_name = "Granulosa cells"  # 제목용 문자열

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 복사 및 organ_group, age_group 추가 ---
data = celltype_df.copy()
data['organ_group'] = data['organism part'].map(organ_group_map)
data['age_group'] = np.where(data['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(data['Final_annotation'].dropna().unique())
organ_groups = ["upper", "lower", "control"]

# --- 전체 subplot 크기 정의 (organ_groups × subtypes) ---
n_organs = len(organ_groups)
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(
    n_organs, n_subtypes,
    figsize=(4 * n_subtypes, 4 * n_organs),
    sharey=True
)

# axes를 항상 2D 배열로 취급
if n_organs == 1:
    axes = np.array([axes])
if n_subtypes == 1:
    axes = axes[:, np.newaxis]

sns.set(style="whitegrid")

for row, og in enumerate(organ_groups):
    data_group = data[data['organ_group'] == og].copy()
    if data_group.empty:
        continue

    # mouse 단위 proportion 계산
    mouse_props = (
        data_group.groupby(['Each mouse', 'Final_annotation', 'age_group'])
        .size()
        .rename('count')
        .reset_index()
    )
    mouse_totals = (
        data_group.groupby(['Each mouse', 'age_group'])
        .size()
        .rename('total')
        .reset_index()
    )
    mouse_props = pd.merge(mouse_props, mouse_totals, on=['Each mouse', 'age_group'], how='left')
    mouse_props['proportion'] = mouse_props['count'] / mouse_props['total']

    # 각 subtype별 subplot
    for col, subtype in enumerate(target_subtypes):
        ax = axes[row, col]
        data_sub = mouse_props[mouse_props['Final_annotation'] == subtype]

        # --- dot 먼저 ---
        sns.stripplot(
            x='age_group', y='proportion', data=data_sub,
            color='black', size=6, jitter=True, ax=ax, zorder=2
        )

        # --- boxplot ---
        sns.boxplot(
            x='age_group', y='proportion', data=data_sub, ax=ax,
            width=0.5, showcaps=True,
            boxprops={'facecolor':'lightgray', 'edgecolor':'black'},
            whiskerprops={'color':'black'},
            medianprops={'color':'red'},
            zorder=1
        )

        # --- 통계: Mann–Whitney U ---
        data_sub_clean = data_sub.dropna(subset=['proportion'])
        group1 = data_sub_clean[data_sub_clean['age_group'] == 'Young']['proportion']
        group2 = data_sub_clean[data_sub_clean['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001: sig = "****"
            elif pval < 0.001: sig = "***"
            elif pval < 0.01: sig = "**"
            elif pval < 0.05: sig = "*"
            else: sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=25)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=20)
        if col == 0:
            ax.set_ylabel(f"{og}\nProportion", fontsize=25)
        else:
            ax.set_ylabel("")
        y_max = data_sub['proportion'].max() if not data_sub.empty else 0.1
        ax.text(
            0.5, y_max * 1.05, p_text,
            ha='center', va='bottom',
            fontsize=20, color='red',
            transform=ax.get_xaxis_transform()
        )

# --- 전체 제목 (celltype_df에 맞게 f-string 처리) ---
plt.suptitle(f"{celltype_name} Subtype – Organ (upper & lower)", fontsize=40, y=1.02)
plt.tight_layout()
plt.show()


# Upper & Lower (Each stastic base)

In [ ]:
# Epithelial: 156101
# Lymphoid: 45172
# Fibroblast: 165841
# Myeloid: 34244
# B cells: 50554

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(Fibroblast['Final_annotation'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = Fibroblast['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = Fibroblast[Fibroblast['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(Fibroblast['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Fibroblast[Fibroblast['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(Fibroblast['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    Fibroblast.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Fibroblast

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Lymphoid = Lymphoid.copy()
Lymphoid['organ_group'] = Lymphoid['organism part'].map(organ_group_map)
Lymphoid['age_group'] = np.where(Lymphoid['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(Lymphoid['Final_annotation'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = Lymphoid['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = Lymphoid[Lymphoid['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Lymphoid[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Lymphoid = Lymphoid.copy()
Lymphoid['organ_group'] = Lymphoid['organism part'].map(organ_group_map)
Lymphoid['age_group'] = np.where(Lymphoid['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(Lymphoid['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Lymphoid['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Lymphoid[Lymphoid['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Lymphoid[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Lymphoid = Lymphoid.copy()
Lymphoid['organ_group'] = Lymphoid['organism part'].map(organ_group_map)
Lymphoid['age_group'] = np.where(Lymphoid['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(Lymphoid['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Lymphoid['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    Lymphoid.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Lymphoid[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Lymphoid

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Myeloid = Myeloid.copy()
Myeloid['organ_group'] = Myeloid['organism part'].map(organ_group_map)
Myeloid['age_group'] = np.where(Myeloid['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(Myeloid['Final_annotation'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = Myeloid['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = Myeloid[Myeloid['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Myeloid[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Myeloid = Myeloid.copy()
Myeloid['organ_group'] = Myeloid['organism part'].map(organ_group_map)
Myeloid['age_group'] = np.where(Myeloid['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(Myeloid['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Myeloid['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Myeloid[Myeloid['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Myeloid[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Myeloid = Myeloid.copy()
Myeloid['organ_group'] = Myeloid['organism part'].map(organ_group_map)
Myeloid['age_group'] = np.where(Myeloid['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(Myeloid['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Myeloid['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    Myeloid.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Myeloid[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Myeloid

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Endocrine

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Epithelial = Epithelial.copy()
Epithelial['organ_group'] = Epithelial['organism part'].map(organ_group_map)
Epithelial['age_group'] = np.where(Epithelial['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(Epithelial['Final_annotation'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = Epithelial['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = Epithelial[Epithelial['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Epithelial[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Epithelial = Epithelial.copy()
Epithelial['organ_group'] = Epithelial['organism part'].map(organ_group_map)
Epithelial['age_group'] = np.where(Epithelial['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(Epithelial['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Epithelial['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Epithelial[Epithelial['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Epithelial[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
Epithelial = Epithelial.copy()
Epithelial['organ_group'] = Epithelial['organism part'].map(organ_group_map)
Epithelial['age_group'] = np.where(Epithelial['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(Epithelial['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Epithelial['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    Epithelial.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Epithelial[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Epithelial

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
B_cells = B_cells.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = B_cells['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = B_cells[B_cells['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = B_cells[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
B_cells = B_cells.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = B_cells['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = B_cells[B_cells['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = B_cells[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 복사 및 organ_group, age_group 추가 ---
B_cells = B_cells.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = B_cells['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    B_cells.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = B_cells[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = B_cells

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
B_cells = Endocrine.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = B_cells['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = B_cells[B_cells['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = B_cells[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


# Upper & Lower (Filter)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
# Fibroblast = Fibroblast.copy()
# Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
# Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- subtypes 정의 ---
try:
    target_subtypes = sorted(Fibroblast['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- Organ별 반복 ---
for organ in organs:
    # 해당 organ 데이터만 필터링
    data_organ = Fibroblast[Fibroblast['organism part'] == organ].copy()

    # ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
    filtered_mice_for_organ = mice_to_filter_by_organ.get(organ, [])
    if filtered_mice_for_organ:
        data_organ = data_organ[~data_organ['Each mouse'].isin(filtered_mice_for_organ)]

    # 이 장기에 실제로 샘플이 있는 마우스만 추출 (필터링 후)
    mouse_list_organ = data_organ['Each mouse'].unique()

    # 해당 장기 데이터가 없는 경우 다음 루프로 건너뛰기
    if len(mouse_list_organ) == 0:
        print(f"No data available for organ: {organ}. Skipping plot generation.")
        continue

    # ① 해당 장기의 샘플을 가진 마우스와 Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_organ, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_organ, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기 (전체 조합에 카운트 merge)
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    # 서브타입이 없는 경우를 대비한 예외 처리
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ: {organ}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        # Box plot과 Stripplot에 사용할 데이터는 해당 장기 샘플을 가진 마우스로 이미 필터링되어 있음
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o',
                          edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        # 통계분석은 두 그룹 모두 샘플이 1개 이상일 때만 수행
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
# Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
try:
    target_subtypes = sorted(Fibroblast['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []


# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# 각 장기별로 필터링할 행을 식별합니다.
rows_to_filter_out = pd.DataFrame()
for organ, mice in mice_to_filter_by_organ.items():
    # 해당 organ과 mice에 해당하는 데이터만 추출합니다.
    filtered_df = Fibroblast[
        (Fibroblast['organism part'] == organ) &
        (Fibroblast['Each mouse'].isin(mice))
    ]
    if not filtered_df.empty:
        rows_to_filter_out = pd.concat([rows_to_filter_out, filtered_df])

# 원본 데이터프레임에서 필터링할 행을 제거합니다.
# 인덱스를 사용하여 안전하게 제거합니다.
if not rows_to_filter_out.empty:
    Fibroblast_filtered = Fibroblast.drop(rows_to_filter_out.index)
else:
    Fibroblast_filtered = Fibroblast.copy()

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Fibroblast_filtered[Fibroblast_filtered['organ_group'] == group_name].copy()
    
    # 해당 Organ Group에 데이터가 없는 경우 다음 루프로 건너뛰기
    if data_group.empty:
        print(f"No data available for organ group: {group_name}. Skipping plot generation.")
        continue

    # 이 그룹에 실제로 샘플이 있는 마우스 목록 추출 (필터링 후)
    mouse_list_group = data_group['Each mouse'].unique()

    # --- 모든 mouse × subtype 조합 생성 (이제 필터링된 마우스 목록 사용) ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_group, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Fibroblast_filtered[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ group: {group_name}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()
        
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
# spleen 제외
Fibroblast = Fibroblast[Fibroblast['organism part'] != 'spleen'].copy()

# --- 복사 및 organ_group, age_group 추가 ---
# Epithelial 객체를 Fibroblast 객체로 변경
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- 필터링 로직 추가: 각 organ별로 지정된 mouse를 제외하고 하나의 데이터프레임으로 합치기 ---
filtered_fibroblast_list = []
for organ in organs:
    data_organ = Fibroblast[Fibroblast['organism part'] == organ].copy()
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    filtered_fibroblast_list.append(data_organ)

# 필터링된 모든 데이터를 하나의 데이터프레임으로 결합
Fibroblast_filtered = pd.concat(filtered_fibroblast_list, ignore_index=True)


# --- target subtypes ---
target_subtypes = sorted(Fibroblast_filtered['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 (필터링된 데이터 기준) ---
mouse_list_all = Fibroblast_filtered['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 (필터링된 데이터 기준) ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts (필터링된 데이터 기준) ---
subtype_counts = (
    Fibroblast_filtered.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Fibroblast_filtered[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Fibroblast.copy()

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=15)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- spleen 제외 ---
df = Fibroblast[Fibroblast['organism part'] != 'spleen'].copy()

# --- age_group 정의 ---
df['age_group'] = ['Aged' if x > 70 else 'Young' for x in df['age_num']]

# --- 사용자 지정 필터링 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"]
}

# --- organs 순서 지정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina"]

# --- organ별 proportion 계산 ---
prop_list = []

for organ in organs:
    data_organ = df[df['organism part'] == organ].copy()
    
    # mouse 필터링
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    
    # Each mouse별 cell count
    mouse_counts = data_organ.groupby('Each mouse').size().reset_index(name='cell_count')
    
    # mouse별 proportion (organ 내 합=1)
    total_cells = mouse_counts['cell_count'].sum()
    if total_cells > 0:
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
    else:
        mouse_counts['proportion'] = 0
    
    # organ명과 age_group 추가
    mouse_age = data_organ[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = mouse_counts.merge(mouse_age, on='Each mouse', how='left')
    mouse_counts['organ'] = organ
    
    prop_list.append(mouse_counts)

# --- 모든 organ 결합 ---
prop_df = pd.concat(prop_list, ignore_index=True)

# --- 시각화 + 통계 ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(organs), figsize=(5 * len(organs), 5), sharey=True)
if len(organs) == 1:
    axes = [axes]

for ax, organ in zip(axes, organs):
    data = prop_df[prop_df['organ'] == organ]
    
    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, width=0.5, ax=ax)
    sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=10, jitter=True, ax=ax)
    
    # Mann-Whitney U test
    group_young = data[data['age_group'] == 'Young']['proportion']
    group_aged = data[data['age_group'] == 'Aged']['proportion']
    
    if len(group_young) > 1 and len(group_aged) > 1:
        stat, pval = mannwhitneyu(group_young, group_aged, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # p-value 표시 (box 중앙)
    y_max = data['proportion'].max() if len(data['proportion']) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=15, color='red', transform=ax.get_xaxis_transform())
    
    ax.set_title(organ, fontsize=24)
    ax.set_ylabel("Proportion per mouse",fontsize=16)
    ax.set_xlabel("")  # x축 레이블 제거
    ax.tick_params(axis='x', labelsize=18)  # Young / Aged 글씨 크기 조절

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
Lymphoid = Lymphoid.copy()
Lymphoid['organ_group'] = Lymphoid['organism part'].map(organ_group_map)
Lymphoid['age_group'] = np.where(Lymphoid['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- subtypes 정의 ---
try:
    target_subtypes = sorted(Lymphoid['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- Organ별 반복 ---
for organ in organs:
    # 해당 organ 데이터만 필터링
    data_organ = Lymphoid[Lymphoid['organism part'] == organ].copy()

    # ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
    filtered_mice_for_organ = mice_to_filter_by_organ.get(organ, [])
    if filtered_mice_for_organ:
        data_organ = data_organ[~data_organ['Each mouse'].isin(filtered_mice_for_organ)]

    # 이 장기에 실제로 샘플이 있는 마우스만 추출 (필터링 후)
    mouse_list_organ = data_organ['Each mouse'].unique()

    # 해당 장기 데이터가 없는 경우 다음 루프로 건너뛰기
    if len(mouse_list_organ) == 0:
        print(f"No data available for organ: {organ}. Skipping plot generation.")
        continue

    # ① 해당 장기의 샘플을 가진 마우스와 Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_organ, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_organ, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기 (전체 조합에 카운트 merge)
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Lymphoid[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    # 서브타입이 없는 경우를 대비한 예외 처리
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ: {organ}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        # Box plot과 Stripplot에 사용할 데이터는 해당 장기 샘플을 가진 마우스로 이미 필터링되어 있음
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o',
                          edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        # 통계분석은 두 그룹 모두 샘플이 1개 이상일 때만 수행
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
# Fibroblast = Fibroblast.copy()
Lymphoid['organ_group'] = Lymphoid['organism part'].map(organ_group_map)
Lymphoid['age_group'] = np.where(Lymphoid['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
try:
    target_subtypes = sorted(Lymphoid['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []


# --- 전체 mouse 목록 ---
mouse_list_all = Lymphoid['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# 각 장기별로 필터링할 행을 식별합니다.
rows_to_filter_out = pd.DataFrame()
for organ, mice in mice_to_filter_by_organ.items():
    # 해당 organ과 mice에 해당하는 데이터만 추출합니다.
    filtered_df = Lymphoid[
        (Lymphoid['organism part'] == organ) &
        (Lymphoid['Each mouse'].isin(mice))
    ]
    if not filtered_df.empty:
        rows_to_filter_out = pd.concat([rows_to_filter_out, filtered_df])

# 원본 데이터프레임에서 필터링할 행을 제거합니다.
# 인덱스를 사용하여 안전하게 제거합니다.
if not rows_to_filter_out.empty:
    Lymphoid_filtered = Lymphoid.drop(rows_to_filter_out.index)
else:
    Lymphoid_filtered = Lymphoid.copy()

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Lymphoid_filtered[Lymphoid_filtered['organ_group'] == group_name].copy()
    
    # 해당 Organ Group에 데이터가 없는 경우 다음 루프로 건너뛰기
    if data_group.empty:
        print(f"No data available for organ group: {group_name}. Skipping plot generation.")
        continue

    # 이 그룹에 실제로 샘플이 있는 마우스 목록 추출 (필터링 후)
    mouse_list_group = data_group['Each mouse'].unique()

    # --- 모든 mouse × subtype 조합 생성 (이제 필터링된 마우스 목록 사용) ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_group, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Lymphoid_filtered[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ group: {group_name}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()
        
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
# spleen 제외
Lymphoid = Lymphoid[Lymphoid['organism part'] != 'spleen'].copy()

# --- 복사 및 organ_group, age_group 추가 ---
# Epithelial 객체를 Fibroblast 객체로 변경
Lymphoid = Lymphoid.copy()
Lymphoid['organ_group'] = Lymphoid['organism part'].map(organ_group_map)
Lymphoid['age_group'] = np.where(Lymphoid['age_num'] > 70, 'Aged', 'Young')

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- 필터링 로직 추가: 각 organ별로 지정된 mouse를 제외하고 하나의 데이터프레임으로 합치기 ---
filtered_Lymphoid_list = []
for organ in organs:
    data_organ = Lymphoid[Lymphoid['organism part'] == organ].copy()
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    filtered_Lymphoid_list.append(data_organ)

# 필터링된 모든 데이터를 하나의 데이터프레임으로 결합
Lymphoid_filtered = pd.concat(filtered_Lymphoid_list, ignore_index=True)


# --- target subtypes ---
target_subtypes = sorted(Lymphoid_filtered['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 (필터링된 데이터 기준) ---
mouse_list_all = Lymphoid_filtered['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 (필터링된 데이터 기준) ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts (필터링된 데이터 기준) ---
subtype_counts = (
    Lymphoid_filtered.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Lymphoid_filtered[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Lymphoid.copy()

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=15)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- spleen 제외 ---
df = Lymphoid[Lymphoid['organism part'] != 'spleen'].copy()

# --- age_group 정의 ---
df['age_group'] = ['Aged' if x > 70 else 'Young' for x in df['age_num']]

# --- 사용자 지정 필터링 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"]
}

# --- organs 순서 지정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina"]

# --- organ별 proportion 계산 ---
prop_list = []

for organ in organs:
    data_organ = df[df['organism part'] == organ].copy()
    
    # mouse 필터링
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    
    # Each mouse별 cell count
    mouse_counts = data_organ.groupby('Each mouse').size().reset_index(name='cell_count')
    
    # mouse별 proportion (organ 내 합=1)
    total_cells = mouse_counts['cell_count'].sum()
    if total_cells > 0:
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
    else:
        mouse_counts['proportion'] = 0
    
    # organ명과 age_group 추가
    mouse_age = data_organ[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = mouse_counts.merge(mouse_age, on='Each mouse', how='left')
    mouse_counts['organ'] = organ
    
    prop_list.append(mouse_counts)

# --- 모든 organ 결합 ---
prop_df = pd.concat(prop_list, ignore_index=True)

# --- 시각화 + 통계 ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(organs), figsize=(5 * len(organs), 5), sharey=True)
if len(organs) == 1:
    axes = [axes]

for ax, organ in zip(axes, organs):
    data = prop_df[prop_df['organ'] == organ]
    
    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, width=0.5, ax=ax)
    sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=10, jitter=True, ax=ax)
    
    # Mann-Whitney U test
    group_young = data[data['age_group'] == 'Young']['proportion']
    group_aged = data[data['age_group'] == 'Aged']['proportion']
    
    if len(group_young) > 1 and len(group_aged) > 1:
        stat, pval = mannwhitneyu(group_young, group_aged, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # p-value 표시 (box 중앙)
    y_max = data['proportion'].max() if len(data['proportion']) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=15, color='red', transform=ax.get_xaxis_transform())
    
    ax.set_title(organ, fontsize=24)
    ax.set_ylabel("Proportion per mouse",fontsize=16)
    ax.set_xlabel("")  # x축 레이블 제거
    ax.tick_params(axis='x', labelsize=18)  # Young / Aged 글씨 크기 조절

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
Myeloid = Myeloid.copy()
Myeloid['organ_group'] = Myeloid['organism part'].map(organ_group_map)
Myeloid['age_group'] = np.where(Myeloid['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- subtypes 정의 ---
try:
    target_subtypes = sorted(Myeloid['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- Organ별 반복 ---
for organ in organs:
    # 해당 organ 데이터만 필터링
    data_organ = Myeloid[Myeloid['organism part'] == organ].copy()

    # ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
    filtered_mice_for_organ = mice_to_filter_by_organ.get(organ, [])
    if filtered_mice_for_organ:
        data_organ = data_organ[~data_organ['Each mouse'].isin(filtered_mice_for_organ)]

    # 이 장기에 실제로 샘플이 있는 마우스만 추출 (필터링 후)
    mouse_list_organ = data_organ['Each mouse'].unique()

    # 해당 장기 데이터가 없는 경우 다음 루프로 건너뛰기
    if len(mouse_list_organ) == 0:
        print(f"No data available for organ: {organ}. Skipping plot generation.")
        continue

    # ① 해당 장기의 샘플을 가진 마우스와 Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_organ, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_organ, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기 (전체 조합에 카운트 merge)
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Myeloid[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    # 서브타입이 없는 경우를 대비한 예외 처리
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ: {organ}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        # Box plot과 Stripplot에 사용할 데이터는 해당 장기 샘플을 가진 마우스로 이미 필터링되어 있음
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o',
                          edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        # 통계분석은 두 그룹 모두 샘플이 1개 이상일 때만 수행
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
# Fibroblast = Fibroblast.copy()
Myeloid['organ_group'] = Myeloid['organism part'].map(organ_group_map)
Myeloid['age_group'] = np.where(Myeloid['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
try:
    target_subtypes = sorted(Myeloid['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []


# --- 전체 mouse 목록 ---
mouse_list_all = Myeloid['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# 각 장기별로 필터링할 행을 식별합니다.
rows_to_filter_out = pd.DataFrame()
for organ, mice in mice_to_filter_by_organ.items():
    # 해당 organ과 mice에 해당하는 데이터만 추출합니다.
    filtered_df = Myeloid[
        (Myeloid['organism part'] == organ) &
        (Myeloid['Each mouse'].isin(mice))
    ]
    if not filtered_df.empty:
        rows_to_filter_out = pd.concat([rows_to_filter_out, filtered_df])

# 원본 데이터프레임에서 필터링할 행을 제거합니다.
# 인덱스를 사용하여 안전하게 제거합니다.
if not rows_to_filter_out.empty:
    Myeloid_filtered = Myeloid.drop(rows_to_filter_out.index)
else:
    Myeloid_filtered = Myeloid.copy()

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Myeloid_filtered[Myeloid_filtered['organ_group'] == group_name].copy()
    
    # 해당 Organ Group에 데이터가 없는 경우 다음 루프로 건너뛰기
    if data_group.empty:
        print(f"No data available for organ group: {group_name}. Skipping plot generation.")
        continue

    # 이 그룹에 실제로 샘플이 있는 마우스 목록 추출 (필터링 후)
    mouse_list_group = data_group['Each mouse'].unique()

    # --- 모든 mouse × subtype 조합 생성 (이제 필터링된 마우스 목록 사용) ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_group, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Myeloid_filtered[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ group: {group_name}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()
        
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
# spleen 제외
Myeloid = Myeloid[Myeloid['organism part'] != 'spleen'].copy()

# --- 복사 및 organ_group, age_group 추가 ---
# Epithelial 객체를 Fibroblast 객체로 변경
Myeloid = Myeloid.copy()
Myeloid['organ_group'] = Myeloid['organism part'].map(organ_group_map)
Myeloid['age_group'] = np.where(Myeloid['age_num'] > 70, 'Aged', 'Young')

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- 필터링 로직 추가: 각 organ별로 지정된 mouse를 제외하고 하나의 데이터프레임으로 합치기 ---
filtered_Myeloid_list = []
for organ in organs:
    data_organ = Myeloid[Myeloid['organism part'] == organ].copy()
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    filtered_Myeloid_list.append(data_organ)

# 필터링된 모든 데이터를 하나의 데이터프레임으로 결합
Myeloid_filtered = pd.concat(filtered_Myeloid_list, ignore_index=True)


# --- target subtypes ---
target_subtypes = sorted(Myeloid_filtered['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 (필터링된 데이터 기준) ---
mouse_list_all = Myeloid_filtered['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 (필터링된 데이터 기준) ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts (필터링된 데이터 기준) ---
subtype_counts = (
    Myeloid_filtered.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Myeloid_filtered[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Myeloid.copy()

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=15)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- spleen 제외 ---
df = Myeloid[Myeloid['organism part'] != 'spleen'].copy()

# --- age_group 정의 ---
df['age_group'] = ['Aged' if x > 70 else 'Young' for x in df['age_num']]

# --- 사용자 지정 필터링 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"]
}

# --- organs 순서 지정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina"]

# --- organ별 proportion 계산 ---
prop_list = []

for organ in organs:
    data_organ = df[df['organism part'] == organ].copy()
    
    # mouse 필터링
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    
    # Each mouse별 cell count
    mouse_counts = data_organ.groupby('Each mouse').size().reset_index(name='cell_count')
    
    # mouse별 proportion (organ 내 합=1)
    total_cells = mouse_counts['cell_count'].sum()
    if total_cells > 0:
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
    else:
        mouse_counts['proportion'] = 0
    
    # organ명과 age_group 추가
    mouse_age = data_organ[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = mouse_counts.merge(mouse_age, on='Each mouse', how='left')
    mouse_counts['organ'] = organ
    
    prop_list.append(mouse_counts)

# --- 모든 organ 결합 ---
prop_df = pd.concat(prop_list, ignore_index=True)

# --- 시각화 + 통계 ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(organs), figsize=(5 * len(organs), 5), sharey=True)
if len(organs) == 1:
    axes = [axes]

for ax, organ in zip(axes, organs):
    data = prop_df[prop_df['organ'] == organ]
    
    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, width=0.5, ax=ax)
    sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=10, jitter=True, ax=ax)
    
    # Mann-Whitney U test
    group_young = data[data['age_group'] == 'Young']['proportion']
    group_aged = data[data['age_group'] == 'Aged']['proportion']
    
    if len(group_young) > 1 and len(group_aged) > 1:
        stat, pval = mannwhitneyu(group_young, group_aged, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # p-value 표시 (box 중앙)
    y_max = data['proportion'].max() if len(data['proportion']) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=15, color='red', transform=ax.get_xaxis_transform())
    
    ax.set_title(organ, fontsize=24)
    ax.set_ylabel("Proportion per mouse",fontsize=16)
    ax.set_xlabel("")  # x축 레이블 제거
    ax.tick_params(axis='x', labelsize=18)  # Young / Aged 글씨 크기 조절

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
Epithelial = Epithelial.copy()
Epithelial['organ_group'] = Epithelial['organism part'].map(organ_group_map)
Epithelial['age_group'] = np.where(Epithelial['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- subtypes 정의 ---
try:
    target_subtypes = sorted(Epithelial['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- Organ별 반복 ---
for organ in organs:
    # 해당 organ 데이터만 필터링
    data_organ = Epithelial[Epithelial['organism part'] == organ].copy()

    # ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
    filtered_mice_for_organ = mice_to_filter_by_organ.get(organ, [])
    if filtered_mice_for_organ:
        data_organ = data_organ[~data_organ['Each mouse'].isin(filtered_mice_for_organ)]

    # 이 장기에 실제로 샘플이 있는 마우스만 추출 (필터링 후)
    mouse_list_organ = data_organ['Each mouse'].unique()

    # 해당 장기 데이터가 없는 경우 다음 루프로 건너뛰기
    if len(mouse_list_organ) == 0:
        print(f"No data available for organ: {organ}. Skipping plot generation.")
        continue

    # ① 해당 장기의 샘플을 가진 마우스와 Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_organ, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_organ, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기 (전체 조합에 카운트 merge)
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Epithelial[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    # 서브타입이 없는 경우를 대비한 예외 처리
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ: {organ}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        # Box plot과 Stripplot에 사용할 데이터는 해당 장기 샘플을 가진 마우스로 이미 필터링되어 있음
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o',
                          edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        # 통계분석은 두 그룹 모두 샘플이 1개 이상일 때만 수행
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
# Fibroblast = Fibroblast.copy()
Epithelial['organ_group'] = Epithelial['organism part'].map(organ_group_map)
Epithelial['age_group'] = np.where(Epithelial['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
try:
    target_subtypes = sorted(Epithelial['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []


# --- 전체 mouse 목록 ---
mouse_list_all = Epithelial['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# 각 장기별로 필터링할 행을 식별합니다.
rows_to_filter_out = pd.DataFrame()
for organ, mice in mice_to_filter_by_organ.items():
    # 해당 organ과 mice에 해당하는 데이터만 추출합니다.
    filtered_df = Epithelial[
        (Epithelial['organism part'] == organ) &
        (Epithelial['Each mouse'].isin(mice))
    ]
    if not filtered_df.empty:
        rows_to_filter_out = pd.concat([rows_to_filter_out, filtered_df])

# 원본 데이터프레임에서 필터링할 행을 제거합니다.
# 인덱스를 사용하여 안전하게 제거합니다.
if not rows_to_filter_out.empty:
   Epithelial_filtered = Epithelial.drop(rows_to_filter_out.index)
else:
    Epithelial_filtered = Epithelial.copy()

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Epithelial_filtered[Epithelial_filtered['organ_group'] == group_name].copy()
    
    # 해당 Organ Group에 데이터가 없는 경우 다음 루프로 건너뛰기
    if data_group.empty:
        print(f"No data available for organ group: {group_name}. Skipping plot generation.")
        continue

    # 이 그룹에 실제로 샘플이 있는 마우스 목록 추출 (필터링 후)
    mouse_list_group = data_group['Each mouse'].unique()

    # --- 모든 mouse × subtype 조합 생성 (이제 필터링된 마우스 목록 사용) ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_group, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Epithelial_filtered[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ group: {group_name}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()
        
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
# spleen 제외
Epithelial = Epithelial[Epithelial['organism part'] != 'spleen'].copy()

# --- 복사 및 organ_group, age_group 추가 ---
# Epithelial 객체를 Fibroblast 객체로 변경
Epithelial = Epithelial.copy()
Epithelial['organ_group'] = Epithelial['organism part'].map(organ_group_map)
Epithelial['age_group'] = np.where(Epithelial['age_num'] > 70, 'Aged', 'Young')

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- 필터링 로직 추가: 각 organ별로 지정된 mouse를 제외하고 하나의 데이터프레임으로 합치기 ---
filtered_Epithelial_list = []
for organ in organs:
    data_organ = Epithelial[Epithelial['organism part'] == organ].copy()
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    filtered_Epithelial_list.append(data_organ)

# 필터링된 모든 데이터를 하나의 데이터프레임으로 결합
Epithelial_filtered = pd.concat(filtered_Epithelial_list, ignore_index=True)


# --- target subtypes ---
target_subtypes = sorted(Epithelial_filtered['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 (필터링된 데이터 기준) ---
mouse_list_all = Epithelial_filtered['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 (필터링된 데이터 기준) ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts (필터링된 데이터 기준) ---
subtype_counts = (
    Epithelial_filtered.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Epithelial_filtered[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- spleen 제외 ---
df = Epithelial[Epithelial['organism part'] != 'spleen'].copy()

# --- age_group 정의 ---
df['age_group'] = ['Aged' if x > 70 else 'Young' for x in df['age_num']]

# --- 사용자 지정 필터링 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"]
}

# --- organs 순서 지정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina"]

# --- organ별 proportion 계산 ---
prop_list = []

for organ in organs:
    data_organ = df[df['organism part'] == organ].copy()
    
    # mouse 필터링
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    
    # Each mouse별 cell count
    mouse_counts = data_organ.groupby('Each mouse').size().reset_index(name='cell_count')
    
    # mouse별 proportion (organ 내 합=1)
    total_cells = mouse_counts['cell_count'].sum()
    if total_cells > 0:
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
    else:
        mouse_counts['proportion'] = 0
    
    # organ명과 age_group 추가
    mouse_age = data_organ[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = mouse_counts.merge(mouse_age, on='Each mouse', how='left')
    mouse_counts['organ'] = organ
    
    prop_list.append(mouse_counts)

# --- 모든 organ 결합 ---
prop_df = pd.concat(prop_list, ignore_index=True)

# --- 시각화 + 통계 ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(organs), figsize=(5 * len(organs), 5), sharey=True)
if len(organs) == 1:
    axes = [axes]

for ax, organ in zip(axes, organs):
    data = prop_df[prop_df['organ'] == organ]
    
    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, width=0.5, ax=ax)
    sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=10, jitter=True, ax=ax)
    
    # Mann-Whitney U test
    group_young = data[data['age_group'] == 'Young']['proportion']
    group_aged = data[data['age_group'] == 'Aged']['proportion']
    
    if len(group_young) > 1 and len(group_aged) > 1:
        stat, pval = mannwhitneyu(group_young, group_aged, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # p-value 표시 (box 중앙)
    y_max = data['proportion'].max() if len(data['proportion']) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=15, color='red', transform=ax.get_xaxis_transform())
    
    ax.set_title(organ, fontsize=24)
    ax.set_ylabel("Proportion per mouse",fontsize=16)
    ax.set_xlabel("")  # x축 레이블 제거
    ax.tick_params(axis='x', labelsize=18)  # Young / Aged 글씨 크기 조절

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = Epithelial.copy()

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=15)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
B_cells = B_cells.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- subtypes 정의 ---
try:
    target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- Organ별 반복 ---
for organ in organs:
    # 해당 organ 데이터만 필터링
    data_organ = B_cells[B_cells['organism part'] == organ].copy()

    # ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
    filtered_mice_for_organ = mice_to_filter_by_organ.get(organ, [])
    if filtered_mice_for_organ:
        data_organ = data_organ[~data_organ['Each mouse'].isin(filtered_mice_for_organ)]

    # 이 장기에 실제로 샘플이 있는 마우스만 추출 (필터링 후)
    mouse_list_organ = data_organ['Each mouse'].unique()

    # 해당 장기 데이터가 없는 경우 다음 루프로 건너뛰기
    if len(mouse_list_organ) == 0:
        print(f"No data available for organ: {organ}. Skipping plot generation.")
        continue

    # ① 해당 장기의 샘플을 가진 마우스와 Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_organ, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_organ, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기 (전체 조합에 카운트 merge)
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = B_cells[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    # 서브타입이 없는 경우를 대비한 예외 처리
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ: {organ}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        # Box plot과 Stripplot에 사용할 데이터는 해당 장기 샘플을 가진 마우스로 이미 필터링되어 있음
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o',
                          edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()

        # 통계분석은 두 그룹 모두 샘플이 1개 이상일 때만 수행
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast는 이미 준비되어 있다고 가정합니다.
# Fibroblast = Fibroblast.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
try:
    target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())
except KeyError:
    print("Warning: 'Final_annotation' column not found. Please ensure the dataframe has this column.")
    target_subtypes = []


# --- 전체 mouse 목록 ---
mouse_list_all = B_cells['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# ******* 핵심 변경사항: 사용자가 지정한 필터링 조건 적용 *******
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# 각 장기별로 필터링할 행을 식별합니다.
rows_to_filter_out = pd.DataFrame()
for organ, mice in mice_to_filter_by_organ.items():
    # 해당 organ과 mice에 해당하는 데이터만 추출합니다.
    filtered_df = B_cells[
        (B_cells['organism part'] == organ) &
        (B_cells['Each mouse'].isin(mice))
    ]
    if not filtered_df.empty:
        rows_to_filter_out = pd.concat([rows_to_filter_out, filtered_df])

# 원본 데이터프레임에서 필터링할 행을 제거합니다.
# 인덱스를 사용하여 안전하게 제거합니다.
if not rows_to_filter_out.empty:
   B_cells_filtered = B_cells.drop(rows_to_filter_out.index)
else:
    B_cells_filtered = B_cells.copy()

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = B_cells_filtered[B_cells_filtered['organ_group'] == group_name].copy()
    
    # 해당 Organ Group에 데이터가 없는 경우 다음 루프로 건너뛰기
    if data_group.empty:
        print(f"No data available for organ group: {group_name}. Skipping plot generation.")
        continue

    # 이 그룹에 실제로 샘플이 있는 마우스 목록 추출 (필터링 후)
    mouse_list_group = data_group['Each mouse'].unique()

    # --- 모든 mouse × subtype 조합 생성 (이제 필터링된 마우스 목록 사용) ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_group, target_subtypes],
        names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'Final_annotation'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'Final_annotation'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = B_cells_filtered[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    if n_subtypes == 0:
        print(f"No subtypes found in the data for organ group: {group_name}. Skipping plot generation.")
        continue
    
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data,
                          color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data,
                          color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion'].dropna()
        group2 = data[data['age_group'] == 'Aged']['proportion'].dropna()
        
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
# spleen 제외
B_cells = B_cells[B_cells['organism part'] != 'spleen'].copy()

# --- 복사 및 organ_group, age_group 추가 ---
# Epithelial 객체를 Fibroblast 객체로 변경
B_cells = B_cells.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# --- 필터링 로직 추가: 각 organ별로 지정된 mouse를 제외하고 하나의 데이터프레임으로 합치기 ---
filtered_B_cells_list = []
for organ in organs:
    data_organ = B_cells[B_cells['organism part'] == organ].copy()
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    filtered_B_cells_list.append(data_organ)

# 필터링된 모든 데이터를 하나의 데이터프레임으로 결합
B_cells_filtered = pd.concat(filtered_B_cells_list, ignore_index=True)


# --- target subtypes ---
target_subtypes = sorted(B_cells_filtered['Final_annotation'].dropna().unique())

# --- 전체 mouse 목록 (필터링된 데이터 기준) ---
mouse_list_all = B_cells_filtered['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 (필터링된 데이터 기준) ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'Final_annotation']
).to_frame(index=False)

# --- mouse × subtype counts (필터링된 데이터 기준) ---
subtype_counts = (
    B_cells_filtered.groupby(['Each mouse', 'Final_annotation'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'Final_annotation'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = B_cells_filtered[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['Final_annotation'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- spleen 제외 ---
df = B_cells[B_cells['organism part'] != 'spleen'].copy()

# --- age_group 정의 ---
df['age_group'] = ['Aged' if x > 70 else 'Young' for x in df['age_num']]

# --- 사용자 지정 필터링 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"]
}

# --- organs 순서 지정 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina"]

# --- organ별 proportion 계산 ---
prop_list = []

for organ in organs:
    data_organ = df[df['organism part'] == organ].copy()
    
    # mouse 필터링
    mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
    if mice_to_exclude:
        data_organ = data_organ[~data_organ['Each mouse'].isin(mice_to_exclude)]
    
    # Each mouse별 cell count
    mouse_counts = data_organ.groupby('Each mouse').size().reset_index(name='cell_count')
    
    # mouse별 proportion (organ 내 합=1)
    total_cells = mouse_counts['cell_count'].sum()
    if total_cells > 0:
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
    else:
        mouse_counts['proportion'] = 0
    
    # organ명과 age_group 추가
    mouse_age = data_organ[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = mouse_counts.merge(mouse_age, on='Each mouse', how='left')
    mouse_counts['organ'] = organ
    
    prop_list.append(mouse_counts)

# --- 모든 organ 결합 ---
prop_df = pd.concat(prop_list, ignore_index=True)

# --- 시각화 + 통계 ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, len(organs), figsize=(5 * len(organs), 5), sharey=True)
if len(organs) == 1:
    axes = [axes]

for ax, organ in zip(axes, organs):
    data = prop_df[prop_df['organ'] == organ]
    
    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, width=0.5, ax=ax)
    sns.stripplot(x='age_group', y='proportion', data=data, color='black', size=10, jitter=True, ax=ax)
    
    # Mann-Whitney U test
    group_young = data[data['age_group'] == 'Young']['proportion']
    group_aged = data[data['age_group'] == 'Aged']['proportion']
    
    if len(group_young) > 1 and len(group_aged) > 1:
        stat, pval = mannwhitneyu(group_young, group_aged, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # p-value 표시 (box 중앙)
    y_max = data['proportion'].max() if len(data['proportion']) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=15, color='red', transform=ax.get_xaxis_transform())
    
    ax.set_title(organ, fontsize=24)
    ax.set_ylabel("Proportion per mouse",fontsize=16)
    ax.set_xlabel("")  # x축 레이블 제거
    ax.tick_params(axis='x', labelsize=18)  # Young / Aged 글씨 크기 조절

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# 복사
df = B_cells.copy()

# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=15)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('Cell proportion per mouse by age group')
plt.tight_layout()
plt.show()


In [ ]:
Endocrine['Final_annotation'] = 'Granulosa cells'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Endocrine 데이터프레임이 있다고 가정
B_cells = Endocrine.copy()
B_cells['organ_group'] = B_cells['organism part'].map(organ_group_map)
B_cells['age_group'] = np.where(B_cells['age_num'] > 70, 'Aged', 'Young')

# --- Organ별 필터링 조건 딕셔너리 ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- organs 순서 ---
# organ 순서
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]

# subtypes
target_subtypes = sorted(B_cells['Final_annotation'].dropna().unique())

for organ in organs:
    # organ 데이터 추출
    data_organ = B_cells[B_cells['organism part'] == organ].copy()

    # 필터링
    filtered_mice = mice_to_filter_by_organ.get(organ, [])
    if filtered_mice:
        data_organ = data_organ[~data_organ['Each mouse'].isin(filtered_mice)]

    mouse_list = data_organ['Each mouse'].unique()
    if len(mouse_list) == 0:
        mouse_list = [None]  # 데이터 없으면 placeholder

    # organ × subtype × mouse 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list, target_subtypes], names=['Each mouse', 'Final_annotation']
    ).to_frame(index=False)

    # count 계산
    if not data_organ.empty:
        subtype_counts = (
            data_organ.groupby(['Each mouse', 'Final_annotation']).size()
            .rename('subtype_count').reset_index()
        )
        subtype_counts = pd.merge(all_combinations, subtype_counts,
                                  on=['Each mouse', 'Final_annotation'], how='left')
        subtype_counts['subtype_count'] = subtype_counts['subtype_count'].fillna(0)
    else:
        subtype_counts = all_combinations.copy()
        subtype_counts['subtype_count'] = 0

    # proportion 계산 (subtype별 합 = 1, 없으면 0)
    prop_df_list = []
    for subtype in target_subtypes:
        df_sub = subtype_counts[subtype_counts['Final_annotation'] == subtype].copy()
        total = df_sub['subtype_count'].sum()
        df_sub['proportion'] = df_sub['subtype_count'] / total if total > 0 else 0
        prop_df_list.append(df_sub)
    prop_df = pd.concat(prop_df_list, axis=0)

    # age_group 추가
    prop_df = pd.merge(prop_df, B_cells[['Each mouse', 'age_group']].drop_duplicates(),
                       on='Each mouse', how='left')

    # --- 시각화 ---
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['Final_annotation'] == subtype]

        # boxplot (항상 X축 = Young, Aged)
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax,
                    order=['Aged','Young'], width=0.4)

        # 점 표시 (없으면 빈 상태)
        if not data.empty:
            sns.stripplot(x='age_group', y='proportion',
                          data=data[~data['Each mouse'].isin(red_dots_mice)],
                          color='black', size=8, jitter=True, ax=ax)
            sns.stripplot(x='age_group', y='proportion',
                          data=data[data['Each mouse'].isin(red_dots_mice)],
                          color='red', size=10, jitter=True, ax=ax, marker='o',
                          edgecolor='black', linewidth=1.5)

        # p-value 표시
        group1 = data[data['age_group']=='Young']['proportion'].dropna()
        group2 = data[data['age_group']=='Aged']['proportion'].dropna()
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2)
            sig = ("****" if pval<0.0001 else
                   "***" if pval<0.001 else
                   "**" if pval<0.01 else
                   "*" if pval<0.05 else "n.s.")
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = ""

        y_max = data['proportion'].max() if len(data['proportion'].dropna())>0 else 0.1
        ax.text(0.5, y_max*1.05, p_text, ha='center', va='bottom',
                fontsize=12, color='red', transform=ax.get_xaxis_transform())

        # 라벨
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.set_ylabel("Proportion" if i==0 else "")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)

    plt.suptitle(f"Organ: {organ}", fontsize=30, y=1.05)
    plt.tight_layout()
    plt.show()


# Pathway Analysis

In [ ]:
import scanpy as sc
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Subtype_clustering.h5ad"
adata = sc.read_h5ad(file_path)

In [ ]:
adata = adata[adata.obs['clinical information'] != 'decidualized_pregnant'].copy()
adata.obs['clinical information'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata, 'Final_annotation', method="wilcoxon", use_raw=False)
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)
sc.tl.dendrogram(adata, groupby='Final_annotation')
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby="Final_annotation",
    standard_scale="var",
    n_genes=5,
    use_raw=False
)

In [ ]:
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata, groupby='Final_annotation')
sc.pl.rank_genes_groups_dotplot(
        adata, groupby="Final_annotation", standard_scale="var",  n_genes=5, use_raw=False)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

In [ ]:
from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=5,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=8,
    ylabel_fontsize=8,
    figsize=(10, 20),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

In [ ]:
import pandas as pd

# 예시 DataFrame
# df = pd.DataFrame({'Term': ['tgf-beta', 'EGF', 'MAPK', 'TGFA']})

# 'tgf'가 포함된 행 찾기 (대소문자 구분X)
mask = df['Term'].str.contains('infla', case=False, na=False)
df_tgf = df[mask]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= df_tgf,
    n=10,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=12,
    ylabel_fontsize=12,
    figsize=(10, 20),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

In [ ]:
A = df_tgf[df_tgf['P-value'] <= 0.05]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = heatmap_with_groupwise_ordering(
    df= A,
    n=10,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=12,
    ylabel_fontsize=12,
    figsize=(10, 20),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

# Pathway Analysis (Fibroblast subtype)

In [ ]:
import scanpy as sc
import numpy as np
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Fibroblast_subtype.h5ad"
Fibroblast = sc.read_h5ad(file_path)
Fibroblast.obs["age_num"] = Fibroblast.obs["age"].str.extract(r"(\d+)").astype(int)
Fibroblast.obs['age_group'] = np.where(Fibroblast.obs['age_num'] > 70, 'Aged', 'Young')

In [ ]:
Fibroblast = Fibroblast[Fibroblast.obs['clinical information'] != "decidualized_pregnant"].copy()

Fibroblast.obs['clinical information'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(Fibroblast, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(Fibroblast, n_genes=50, sharey=False)
sc.tl.dendrogram(Fibroblast, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        Fibroblast, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Fibroblast.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(Fibroblast, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        Fibroblast, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Fibroblast.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=40,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

In [ ]:
df

In [ ]:
adata_ECM = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'ECM_fibroblast'].copy()
adata_Metabolism = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'Metabolism_fibroblast'].copy()
adata_Stressed = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'Stressed_fibroblast'].copy()
adata_Immunesupp = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'Immunesupp_fibroblast'].copy()
adata_KRT = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'KRT_fibroblast'].copy()
adata_Vascular = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'Vascular_fibroblast'].copy()
adata_EMT = Fibroblast[Fibroblast.obs['fibroblast_subtype'] == 'EMT_fibroblast'].copy()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sc.tl.rank_genes_groups(adata_ECM, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_ECM, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_ECM, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_ECM, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_ECM.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_ECM, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_ECM, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_ECM.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(adata_Metabolism, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_Metabolism, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_Metabolism, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Metabolism, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Metabolism.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_Metabolism, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Metabolism, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Metabolism.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(adata_EMT, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_EMT, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_EMT, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_EMT, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_EMT.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_EMT, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_EMT, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_EMT.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
df.query("Leiden == 'Aged'").sort_values(by="P-value", ascending=True).head(60)[['Term','Database']]

In [ ]:
sc.tl.rank_genes_groups(adata_Stressed, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_Stressed, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_Stressed, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Stressed, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Stressed.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_Stressed, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Stressed, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Stressed.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(adata_Immunesupp, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_Immunesupp, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_Immunesupp, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Immunesupp, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Immunesupp.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_Immunesupp, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Immunesupp, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Immunesupp.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)

df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=5,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=13,
    ylabel_fontsize=13,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
print(adata_Immunesupp.obs['age_group'].value_counts())

In [ ]:
print(group, "전체:", result_df.shape[0])
print("유의미 DEG:", (result_df[p_col] < 0.05).sum())
print("Up:", ((result_df[p_col] < 0.05) & (result_df[l_col] > 1)).sum())
print("Down:", ((result_df[p_col] < 0.05) & (result_df[l_col] < -1)).sum())
print(result_df[l_col].describe())

In [ ]:
sc.tl.rank_genes_groups(adata_KRT, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_KRT, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_KRT, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_KRT, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_KRT.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_KRT, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_KRT, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_KRT.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

In [ ]:
df.query("Leiden == 'Aged'").sort_values(by="P-value", ascending=True).head(50)[['Term','Database']]

In [ ]:
sc.tl.rank_genes_groups(adata_Vascular, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(adata_Vascular, n_genes=50, sharey=False)
sc.tl.dendrogram(adata_Vascular, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Vascular, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Vascular.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(adata_Vascular, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        adata_Vascular, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = adata_Vascular.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=20,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


plot_significant_terms(df, leiden_value='Aged')
plot_significant_terms(df, leiden_value='Young')

# Pathway analysis (Fibroblast - organ)

In [ ]:
import scanpy as sc
import numpy as np
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/Save_data/Fibroblast_subtype.h5ad"
Fibroblast = sc.read_h5ad(file_path)
Fibroblast.obs["age_num"] = Fibroblast.obs["age"].str.extract(r"(\d+)").astype(int)
Fibroblast.obs['age_group'] = np.where(Fibroblast.obs['age_num'] > 70, 'Aged', 'Young')
Fibroblast = Fibroblast[Fibroblast.obs['clinical information'] != "decidualized_pregnant"].copy()

Fibroblast.obs['clinical information'].value_counts()

In [ ]:
Fibroblast.obs['organism part'].value_counts()

In [ ]:
# organism part 별로 나누기
oviduct_fb = Fibroblast[Fibroblast.obs["organism part"] == "oviduct", :].copy()
ovary_fb   = Fibroblast[Fibroblast.obs["organism part"] == "ovary", :].copy()
uterus_fb  = Fibroblast[Fibroblast.obs["organism part"] == "uterus", :].copy()
vagina_fb  = Fibroblast[Fibroblast.obs["organism part"] == "vagina", :].copy()
cervix_fb  = Fibroblast[Fibroblast.obs["organism part"] == "uterine cervix", :].copy()
spleen_fb  = Fibroblast[Fibroblast.obs["organism part"] == "spleen", :].copy()

In [ ]:
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Fibroblast.obs['organ_group'] = Fibroblast.obs['organism part'].map(organ_group_map)
upper  = Fibroblast[Fibroblast.obs["organ_group"] == "upper", :].copy()
lower  = Fibroblast[Fibroblast.obs["organ_group"] == "lower", :].copy()

In [ ]:
##### upper & lower

In [ ]:
sc.tl.rank_genes_groups(Fibroblast, 'organ_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(Fibroblast, n_genes=50, sharey=False)
sc.tl.dendrogram(Fibroblast, groupby='organ_group')
sc.pl.rank_genes_groups_dotplot(
        Fibroblast, groupby="organ_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Fibroblast.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(Fibroblast, groupby='organ_group')
sc.pl.rank_genes_groups_dotplot(
        Fibroblast, groupby="organ_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = Fibroblast.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
##### upper & lower - each

In [ ]:
sc.tl.rank_genes_groups(upper, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(upper, n_genes=50, sharey=False)
sc.tl.dendrogram(upper, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        upper, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = upper.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(upper, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        upper, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = upper.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(lower, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(lower, n_genes=50, sharey=False)
sc.tl.dendrogram(lower, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        lower, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = lower.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(lower, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        lower, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = lower.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
# Each organ#

In [ ]:
sc.tl.rank_genes_groups(oviduct_fb, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(oviduct_fb, n_genes=50, sharey=False)
sc.tl.dendrogram(oviduct_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        oviduct_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = oviduct_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(oviduct_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        oviduct_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = oviduct_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(ovary_fb, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(ovary_fb, n_genes=50, sharey=False)
sc.tl.dendrogram(ovary_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        ovary_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = ovary_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(ovary_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        ovary_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = ovary_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', None)

In [ ]:
df.head(50)[['Term','Database','Leiden']]

In [ ]:
sc.tl.rank_genes_groups(uterus_fb, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(uterus_fb, n_genes=50, sharey=False)
sc.tl.dendrogram(uterus_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        uterus_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = uterus_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(uterus_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        uterus_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = uterus_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(vagina_fb, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(vagina_fb, n_genes=50, sharey=False)
sc.tl.dendrogram(vagina_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        vagina_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = vagina_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(vagina_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        vagina_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = vagina_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(cervix_fb, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(cervix_fb, n_genes=50, sharey=False)
sc.tl.dendrogram(cervix_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        cervix_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = cervix_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(cervix_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        cervix_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = cervix_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

In [ ]:
sc.tl.rank_genes_groups(spleen_fb, 'age_group', method="wilcoxon",use_raw =False)
sc.pl.rank_genes_groups(spleen_fb, n_genes=50, sharey=False)
sc.tl.dendrogram(spleen_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        spleen_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)
import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = spleen_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

sc.tl.dendrogram(spleen_fb, groupby='age_group')
sc.pl.rank_genes_groups_dotplot(
        spleen_fb, groupby="age_group", standard_scale="var",  n_genes=5, use_raw=False
)

import pandas as pd
import scanpy as sc

# 2. 결과 저장
result = spleen_fb.uns["rank_genes_groups"]
groups = result["names"].dtype.names  # 예: ("Tcells", "Bcells", ...)

# 3. DataFrame 변환
result_df = pd.DataFrame(
    {
        f"{group}_{key}": result[key][group]
        for group in groups
        for key in ["names", "pvals_adj", "logfoldchanges", "scores"]
    }
)

# 4. 결과 저장 딕셔너리
up_dict = {}
down_dict = {}

# 5. 자동 반복 처리
for group in groups:
    n_col = f"{group}_names"
    p_col = f"{group}_pvals_adj"
    l_col = f"{group}_logfoldchanges"
    s_col = f"{group}_scores"

    degs_sig = result_df[result_df[p_col] < 0.05]
    degs_up = degs_sig[degs_sig[l_col] > 1].sort_values(by=s_col, ascending=False)
    degs_dw = degs_sig[degs_sig[l_col] < -1].sort_values(by=s_col, ascending=True)

    # 유전자 이름만 추출
    up_dict[group] = degs_up[n_col].reset_index(drop=True)
    down_dict[group] = degs_dw[n_col].reset_index(drop=True)

# 6. 병합
df_merge_up = pd.concat(up_dict.values(), axis=1)
df_merge_up.columns = list(up_dict.keys())  # 각 열 이름을 group 이름으로

df_merge_down = pd.concat(down_dict.values(), axis=1)
df_merge_down.columns = list(down_dict.keys())

from qed.data import readtxt, readseurat, readscanpy
from qed.data import get_enrichment_dataframes
from qed.data import merge_df
from qed.data import query
import inspect
import pandas as pd

# ✅ 사용자 정의 GeneSet 클래스
class GeneSet:
    def __init__(self, genes, name, annot=None):
        self.genes = genes
        self.name = name
        self.params = {}
        self.GO = []
        self.annot = annot or {}

# ✅ enrichment에 사용할 데이터베이스 목록
dblist = [
    'KEGG_2021_Human',
    'Reactome_2022',
    'MSigDB_Hallmark_2020',
    'GO_Biological_Process_2023',
    'WikiPathway_2024_Human'
]

# ✅ DEG 테이블 상위 20개 추출 및 저장
df_merge_up_100 = df_merge_up.head(20)
df_merge_up_100.to_csv('breast_cancer_DEG_up_leiden.csv', index=False, header=True)

# ✅ CSV 불러오기
alist = pd.read_csv('breast_cancer_DEG_up_leiden.csv')

# ✅ 각 열마다 GeneSet 객체로 변환
geneset_list = []
for col in alist.columns:
    genes = alist[col].dropna().tolist()
    gs = GeneSet(genes=genes, name=col, annot={"Leiden": col})
    geneset_list.append(gs)

# ✅ enrichment 실행
aalist = get_enrichment_dataframes(
    geneset_list=geneset_list,
    dblist=dblist,
    annot_colname="Leiden",
    n_jobs=None,
    handle_error=True,
    max_iter=1
)

# ✅ 결과 병합 및 후처리
df = merge_df(aalist)
df['Term'] = df['Term'].str.split(r'\(GO:').str[0].str.strip()
df['Term'] = df['Term'].str.split('R-H').str[0].str.strip()
df['Term'] = df['Term'].str.split('WP').str[0].str.strip()

from typing import List, Tuple, Dict
def heatmap_with_groupwise_ordering(df: pd.DataFrame,
                                     n: int,
                                     group_by: str,
                                     order_by: str = "Adjusted p-value",
                                     cmap: str = 'Blues',
                                     xlabel_rotation: int = 90,
                                     xlabel_fontsize: int = 15,
                                     ylabel_fontsize: int = 15,
                                     figsize: Tuple = (15, 10),
                                     vmin: float = None,
                                     vmax: float = None,
                                     cbar_kws: Dict = None,
                                     linewidths: float = 0.1,
                                     title: str = None,
                                     save_to_file: str = None,
                                     enable_column_order: bool = False,
                                     column_order: list = None,
                                     enable_highlight_columns: bool = False,
                                     highlight_column_names: list = None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("입력 'df'는 pandas DataFrame이어야 합니다.")
    if group_by not in df.columns:
        raise ValueError(f"'{group_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if order_by not in df.columns:
        raise ValueError(f"'{order_by}' 컬럼이 DataFrame에 존재하지 않습니다.")
    if 'Term' not in df.columns:
        raise ValueError("'Term' 컬럼이 DataFrame에 존재하지 않습니다.")

    has_overlapping_genes = 'Overlapping genes' in df.columns

    # P-value 컬럼 숫자형 강제 변환
    df_copy = df.copy()
    df_copy[order_by] = pd.to_numeric(df_copy[order_by], errors='coerce')
    df = df_copy

    df['Term_lower'] = df['Term'].str.lower()

    # 상위 n개 Term 추출
    df_filtered_by_order = df[df[order_by].notna()]
    df_ranked_per_group = (
        df_filtered_by_order
        .sort_values(order_by)
        .groupby(group_by)
        .head(n)
        .reset_index(drop=True)
    )
    if df_ranked_per_group.empty:
        print(f"⚠️ 각 그룹에서 상위 {n}개의 Term을 선택할 수 없습니다.")
        return None, None

    all_selected_terms_lower = df_ranked_per_group['Term_lower'].unique()

    # Overlapping genes 처리
    overlapping_genes_pivot = pd.DataFrame()
    if has_overlapping_genes:
        overlapping_genes_df = df[df['Term_lower'].isin(all_selected_terms_lower)][['Term_lower', group_by, 'Overlapping genes']]
        overlapping_genes_df['Overlapping genes'] = overlapping_genes_df['Overlapping genes'].apply(
            lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x)
        )
        overlapping_genes_df = overlapping_genes_df.groupby(['Term_lower', group_by])['Overlapping genes'].apply(', '.join).reset_index()
        overlapping_genes_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
        overlapping_genes_pivot = overlapping_genes_df.pivot(index='Term', columns=group_by, values='Overlapping genes').fillna('')
        overlapping_genes_pivot.index = overlapping_genes_pivot.index.map(lambda x: x.capitalize())

    # P-value pivot table
    aggregated_df = df[df['Term_lower'].isin(all_selected_terms_lower)].groupby(['Term_lower', group_by], as_index=False)[order_by].min()
    aggregated_df.rename(columns={'Term_lower': 'Term'}, inplace=True)
    pivot_df = aggregated_df.pivot(index="Term", columns=group_by, values=order_by).fillna(1).rename_axis(None, axis=1)

    # Log 변환
    is_pvalue_column = order_by.lower() in ['adjusted p-value', 'p-value']
    pivot_df_for_heatmap = -np.log10(np.maximum(pivot_df, np.finfo(float).eps)) if is_pvalue_column else pivot_df.copy()
    pivot_df_for_heatmap.index = pivot_df_for_heatmap.index.map(lambda x: x.capitalize())

    # ✅ Overlapping genes와 결합
    final_pivot_df = pd.concat([pivot_df_for_heatmap, overlapping_genes_pivot], axis=1)

    if save_to_file:
        try:
            final_pivot_df.to_csv(save_to_file, encoding='utf-8-sig')
            print(f"✅ 히트맵 데이터 저장 완료: '{save_to_file}'")
        except Exception as e:
            print(f"❌ 저장 실패: {e}")

    # ✅ column_order 기준 row 정렬
    row_order = []
    seen_terms = set()

    if enable_column_order and column_order:
        group_order_for_plot = [group for group in column_order if group in df[group_by].unique()]
    else:
        group_order_for_plot = df[group_by].unique()

    for group in group_order_for_plot:
        current_group_terms = df_ranked_per_group[df_ranked_per_group[group_by] == group].sort_values(order_by)
        for term_lower_val in current_group_terms['Term_lower']:
            capitalized_term = term_lower_val.capitalize()
            if capitalized_term in pivot_df_for_heatmap.index and capitalized_term not in seen_terms:
                row_order.append(capitalized_term)
                seen_terms.add(capitalized_term)

    clustered_data = final_pivot_df.loc[row_order]
    heatmap_data = clustered_data.select_dtypes(include=[np.number])
   # 1️⃣ 사용자 지정 column_order가 있으면 그 순서 적용
    if enable_column_order and column_order:
        valid_cols = [c for c in column_order if c in heatmap_data.columns]
        if valid_cols:
            heatmap_data = heatmap_data[valid_cols]

# 2️⃣ column_order가 없고, 열 이름이 전부 숫자로 해석 가능하면 오름차순 정렬
    else:
        try:
        # 모든 열 이름을 float으로 바꿔 보고 성공하면 오름차순 재정렬
            numeric_col_keys = [float(c) for c in heatmap_data.columns]
            sorted_cols = [col for _, col in sorted(zip(numeric_col_keys, heatmap_data.columns))]
            heatmap_data = heatmap_data[sorted_cols]
        except ValueError:
        # 열 이름이 숫자가 아닌 게 섞여 있으면 그대로 둡니다
            pass


    if is_pvalue_column:
        p_values_for_annotation = 10 ** (-np.clip(heatmap_data, None, 300))
        annotations = np.where(p_values_for_annotation <= 0.05, '*', '')
        annotations = annotations.astype(object)
    else:
        annotations = False

    # ✅ 열 강조 인덱스 계산 (선 추가)
    highlight_column_indices = []
    if enable_highlight_columns and highlight_column_names:
        highlight_column_indices = [
            list(heatmap_data.columns).index(col)
            for col in highlight_column_names if col in heatmap_data.columns
        ]

    # ✅ 시각화
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(heatmap_data, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidths=linewidths, annot=annotations, fmt='',
                linecolor='white', ax=ax, cbar_kws=cbar_kws, square=True,
                annot_kws={"size": 12, "color": "white", "weight": "bold"})

    # 수직선 추가
    for x in highlight_column_indices:
        ax.vlines(x + 1, *ax.get_ylim(), colors='black', linewidth=2)

    plt.xticks(rotation=xlabel_rotation, fontsize=xlabel_fontsize)
    plt.yticks(fontsize=ylabel_fontsize)
    if title:
        ax.set_title(title, fontsize=18, pad=20)
    plt.grid(False)

    if 'Term_lower' in df.columns:
        df.drop(columns='Term_lower', inplace=True)

    return fig, ax

fig, ax = heatmap_with_groupwise_ordering(
    df= df,
    n=30,
    group_by='Leiden',
    order_by="Adjusted p-value",
    cmap='Reds',
    xlabel_rotation=90,
    xlabel_fontsize=10,
    ylabel_fontsize=10,
    figsize=(3, 14),
    vmin=0, vmax=6,
    cbar_kws={'shrink': 0.5, 'label': '-log10(P-value)'},
    enable_column_order=False,
   # column_order=[
   #     "Contractile cardiomyocyte", 
   #     "Myogenerative cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ],
    enable_highlight_columns=False,
   # highlight_column_names=[
   #     "Contractile cardiomyocyte", 
   #     "Inflammatory cardiomyocyte"
   # ]
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_significant_terms(df, leiden_value='Aged', pvalue_thresh=0.05):
    # 필터링
    df_filtered = df[(df['Leiden'] == leiden_value) & (df['P-value'] <= pvalue_thresh)].copy()
    
    if df_filtered.empty:
        print("조건을 만족하는 Term이 없습니다.")
        return
    
    # P-value를 -log10 변환 (옵션)
    df_filtered['neg_log10_p'] = -np.log10(df_filtered['P-value'])
    
    # y축 Term 순서를 P-value 기준으로 정렬
    df_filtered = df_filtered.sort_values('neg_log10_p', ascending=True)
    
    # figure 크기 조절 (Term 개수에 따라)
    plt.figure(figsize=(8, max(4, 0.3*len(df_filtered))))
    
    plt.barh(df_filtered['Term'], df_filtered['neg_log10_p'], color='skyblue')
    plt.xlabel('-log10(P-value)')
    plt.ylabel('Term')
    plt.title(f'Significant Terms in {leiden_value} (P <= {pvalue_thresh})')
    
    plt.tight_layout()
    plt.show()


# plot_significant_terms(df, leiden_value='Aged')
# plot_significant_terms(df, leiden_value='Young')

# Aucell

In [ ]:
Fibroblast = Fibroblast.obs.copy()

In [ ]:
import scanpy as sc
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/Fibroblast_pathway.h5ad"
Fibroblast = sc.read_h5ad(file_path)

In [ ]:
import pandas as pd

# CSV 파일 경로
files = {
    "HALLMARK_INTERFERON_GAMMA_RESPONSE" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/HALLMARK_INTERFERON_GAMMA_RESPONSE_barcode.csv",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/HALLMARK_INTERFERON_ALPHA_RESPONSE_barcode.csv",
    "HALLMARK_IL6_JAK_STAT3_SIGNALING" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/HALLMARK_IL6_JAK_STAT3_SIGNALING_barcode.csv",
    "HALLMARK_INFLAMMATORY_RESPONSE" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/HALLMARK_INFLAMMATORY_RESPONSE_barcode.csv",#
    "HALLMARK_TGF_BETA_SIGNALING" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/HALLMARK_TGF_BETA_SIGNALING_barcode.csv",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/HALLMARK_TNFA_SIGNALING_VIA_NFKB_barcode.csv",
    "KEGG_IL_17_Signaling_pathway" : "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/KEGG_IL_17_Signaling_pathway_barcode.csv"
}

# fib.obs에 새로운 열 추가
for col_name, file_path in files.items():
    # CSV 읽기 (barcode 컬럼이 하나라 가정)
    df = pd.read_csv(file_path, header=None)
    barcodes = set(df[0].astype(str))  # barcode를 set으로 변환
    
    # Fibroblast에 1/0 추가
    Fibroblast.obs[col_name] = Fibroblast.obs.index.astype(str).map(lambda x: 1 if x in barcodes else 0)



In [ ]:
import numpy as np

Fibroblast.obs['age_group'] = np.where(Fibroblast.obs['age_num'] > 70, 'Aged', 'Young')
Fibroblast = Fibroblast.obs.copy()

In [ ]:
# Fibroblast.write("/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/Fibroblast_pathway.h5ad")

In [ ]:
import scanpy as sc
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/Fibroblast_pathway.h5ad"
Fibroblast = sc.read_h5ad(file_path)

In [ ]:
Fibroblast['age_num'].value_counts()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from upsetplot import UpSet

cols = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING',
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_TGF_BETA_SIGNALING',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'KEGG_IL_17_Signaling_pathway'
]

# 각 hallmark의 binary 조합을 MultiIndex로 변환
upset_data = pd.Series(1, index=pd.MultiIndex.from_frame(Fibroblast[cols]))

# UpSet plot 그리기
UpSet(upset_data, subset_size='count', show_counts=True).plot()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Fibroblast = Fibroblast[Fibroblast['HALLMARK_INTERFERON_GAMMA_RESPONSE'] == 1].copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(Fibroblast['fibroblast_subtype'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = Fibroblast['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = Fibroblast[Fibroblast['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'fibroblast_subtype']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'fibroblast_subtype'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'fibroblast_subtype'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['fibroblast_subtype'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['fibroblast_subtype'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
# Fibroblast = Fibroblast[Fibroblast['HALLMARK_INTERFERON_ALPHA_RESPONSE'] == 1].copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
n_organs = len(organs)
# --- 전체 mouse ---
mouse_list_all = Fibroblast['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 한 줄 subplot ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, n_organs, figsize=(5 * n_organs, 5), sharey=True)

if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(organs):
    ax = axes[i]
    data_organ = Fibroblast[Fibroblast['organism part'] == organ]

    # 각 mouse별 cell count
    mouse_counts = (
        data_organ.groupby('Each mouse')
        .size()
        .rename('cell_count')
        .reset_index()
    )

    # proportion 계산
    total_cells = mouse_counts['cell_count'].sum()
    mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells if total_cells > 0 else 0

    # age_group 붙이기
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = pd.merge(mouse_counts, meta_info, on='Each mouse', how='left')

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=mouse_counts, ax=ax, width=0.4)

    # 기본 검은 점 stripplot
    black_data = mouse_counts[~mouse_counts['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=6, jitter=True, ax=ax)

    # 빨간 점
    red_data = mouse_counts[mouse_counts['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=8, jitter=True, ax=ax, marker='o',
                  edgecolor='black', linewidth=1.5)

    # 통계
    group1 = mouse_counts[mouse_counts['age_group'] == 'Young']['proportion']
    group2 = mouse_counts[mouse_counts['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # 라벨 및 글씨 크기 조정
    ax.set_title(f"{organ + ' (control)' if organ == 'spleen' else organ}", fontsize=22)
    ax.set_xlabel("", fontsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (per Organ Group, normalized)", fontsize=14)
    else:
        ax.set_ylabel("")
    ax.tick_params(axis='x', labelsize=20)
    ax.tick_params(axis='y', labelsize=20)

    # p-value 표시
    y_max = mouse_counts['proportion'].max() if len(mouse_counts['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
Fibroblast = Fibroblast[Fibroblast['HALLMARK_TNFA_SIGNALING_VIA_NFKB'] == 1].copy()
# --- 복사 및 organ_group, age_group 추가 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(Fibroblast['fibroblast_subtype'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Fibroblast[Fibroblast['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'fibroblast_subtype']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'fibroblast_subtype'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'fibroblast_subtype'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['fibroblast_subtype'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['fibroblast_subtype'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
Fibroblast = Fibroblast[Fibroblast['HALLMARK_TNFA_SIGNALING_VIA_NFKB'] == 1].copy()
# --- 복사 및 organ_group, age_group 추가 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(Fibroblast['fibroblast_subtype'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'fibroblast_subtype']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    Fibroblast.groupby(['Each mouse', 'fibroblast_subtype'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'fibroblast_subtype'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['fibroblast_subtype'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['fibroblast_subtype'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

Fibroblast = Fibroblast[Fibroblast['HALLMARK_TNFA_SIGNALING_VIA_NFKB'] == 1].copy()
df = Fibroblast.copy()
# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('HALLMARK_TNFA_SIGNALING_VIA_NFKB activation proportion')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# 예시: Fibroblast.obs가 이미 존재한다고 가정
# 네 가지 항목이 모두 1이면 Self_immune = 1
fibroblast_cols_self = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB'
]

fibroblast_cols_chronic = [
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_TGF_BETA_SIGNALING'
]

# Self_immune 열 추가
Fibroblast['Self_immune'] = ((Fibroblast[fibroblast_cols_self] == 1).all(axis=1)).astype(int)

# Chronic_inflam 열 추가
Fibroblast['Chronic_inflam'] = ((Fibroblast[fibroblast_cols_chronic] == 1).all(axis=1)).astype(int)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Fibroblast = Fibroblast[Fibroblast['Self_immune'] == 1].copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
# --- subtypes 정의 ---
target_subtypes = sorted(Fibroblast['fibroblast_subtype'].dropna().unique())
# --- 전체 mouse ---
mouse_list_all = Fibroblast['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ별 반복 ---
for organ in organs:
    data_organ = Fibroblast[Fibroblast['organism part'] == organ]

    # ① 전체 mouse × Final_annotation 조합 생성
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'fibroblast_subtype']
    ).to_frame(index=False)

    # ② 각 mouse별 total cell
    total_cells = (
        data_organ.groupby('Each mouse').size()
        .rename('total_cells')
        .reindex(mouse_list_all, fill_value=0)
        .reset_index()
    )

    # ③ 각 mouse × Final_annotation 카운트
    subtype_counts = (
        data_organ.groupby(['Each mouse', 'fibroblast_subtype'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # ④ NaN 채우기
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'fibroblast_subtype'], how='left'
    ).fillna({'subtype_count': 0})

    # ⑤ proportion 계산 → Organ × Subtype 내 합 = 1
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['fibroblast_subtype'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # ⑥ age_group 붙이기
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['fibroblast_subtype'] == subtype]

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 기본 검은 점 stripplot (빨간 점 샘플 제외)
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)

        # --- 통계 ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per mouse, normalized within subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=15, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ: {organ + ' (control)' if organ == 'spleen' else organ}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- 데이터 준비 ---
Fibroblast = Fibroblast[Fibroblast['Self_immune'] == 1].copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
n_organs = len(organs)
# --- 전체 mouse ---
mouse_list_all = Fibroblast['Each mouse'].unique()
# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 한 줄 subplot ---
sns.set(style="whitegrid")
fig, axes = plt.subplots(1, n_organs, figsize=(5 * n_organs, 5), sharey=True)

if n_organs == 1:
    axes = [axes]

for i, organ in enumerate(organs):
    ax = axes[i]
    data_organ = Fibroblast[Fibroblast['organism part'] == organ]

    # 각 mouse별 cell count
    mouse_counts = (
        data_organ.groupby('Each mouse')
        .size()
        .rename('cell_count')
        .reset_index()
    )

    # proportion 계산
    total_cells = mouse_counts['cell_count'].sum()
    mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells if total_cells > 0 else 0

    # age_group 붙이기
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    mouse_counts = pd.merge(mouse_counts, meta_info, on='Each mouse', how='left')

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=mouse_counts, ax=ax, width=0.4)

    # 기본 검은 점 stripplot
    black_data = mouse_counts[~mouse_counts['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=6, jitter=True, ax=ax)

    # 빨간 점
    red_data = mouse_counts[mouse_counts['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=8, jitter=True, ax=ax, marker='o',
                  edgecolor='black', linewidth=1.5)

    # 통계
    group1 = mouse_counts[mouse_counts['age_group'] == 'Young']['proportion']
    group2 = mouse_counts[mouse_counts['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    # 라벨 및 글씨 크기 조정
    ax.set_title(f"{organ + ' (control)' if organ == 'spleen' else organ}", fontsize=22)
    ax.set_xlabel("", fontsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (per Organ Group, normalized)", fontsize=14)
    else:
        ax.set_ylabel("")
    ax.tick_params(axis='x', labelsize=20)
    ax.tick_params(axis='y', labelsize=20)

    # p-value 표시
    y_max = mouse_counts['proportion'].max() if len(mouse_counts['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
Fibroblast = Fibroblast[Fibroblast['Self_immune'] == 1].copy()
# --- 복사 및 organ_group, age_group 추가 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes 정의 ---
target_subtypes = sorted(Fibroblast['fibroblast_subtype'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- organ_group 순서 ---
organ_groups = ["upper", "lower", "control"]

# --- 빨간 점 찍을 샘플 리스트 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- Organ Group별 반복 ---
for group_name in organ_groups:
    data_group = Fibroblast[Fibroblast['organ_group'] == group_name]

    # --- 모든 mouse × subtype 조합 생성 ---
    all_combinations = pd.MultiIndex.from_product(
        [mouse_list_all, target_subtypes],
        names=['Each mouse', 'fibroblast_subtype']
    ).to_frame(index=False)

    # --- mouse × Subtype counts ---
    subtype_counts = (
        data_group.groupby(['Each mouse', 'fibroblast_subtype'])
        .size()
        .rename('subtype_count')
        .reset_index()
    )

    # --- NaN → 0 채우기 ---
    subtype_counts = pd.merge(
        all_combinations, subtype_counts,
        on=['Each mouse', 'fibroblast_subtype'], how='left'
    ).fillna({'subtype_count': 0})

    # --- Organ Group × Subtype 내 proportion 계산 (합 = 1) ---
    prop_df_list = []
    for subtype in target_subtypes:
        data_subtype = subtype_counts[subtype_counts['fibroblast_subtype'] == subtype].copy()
        total_subtype_count = data_subtype['subtype_count'].sum()
        if total_subtype_count > 0:
            data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
        else:
            data_subtype['proportion'] = 0
        prop_df_list.append(data_subtype)
    prop_df = pd.concat(prop_df_list, axis=0)

    # --- age_group 붙이기 ---
    meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
    prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

    # --- 시각화 ---
    sns.set(style="whitegrid")
    n_subtypes = len(target_subtypes)
    fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
    if n_subtypes == 1:
        axes = [axes]

    for i, subtype in enumerate(target_subtypes):
        ax = axes[i]
        data = prop_df[prop_df['fibroblast_subtype'] == subtype]

        # dot 개수 출력
        n_young = (data['age_group'] == 'Young').sum()
        n_aged = (data['age_group'] == 'Aged').sum()

        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

        # 검은 점: red_dots_mice 제외
        black_data = data[~data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data,
                      color='black', size=8, jitter=True, ax=ax)

        # 빨간 점: red_dots_mice 만
        red_data = data[data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax, marker='o', edgecolor='black', linewidth=1.5)

        # --- 통계 (Mann-Whitney U test) ---
        group1 = data[data['age_group'] == 'Young']['proportion']
        group2 = data[data['age_group'] == 'Aged']['proportion']

        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # --- 라벨 ---
        ax.set_title(subtype, fontsize=20)
        ax.set_xlabel("")
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
        if i == 0:
            ax.set_ylabel("Proportion (per Organ Group, normalized within Subtype)", fontsize=12)
        else:
            ax.set_ylabel("")

        # --- p-value 표시 ---
        y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())

    plt.suptitle(
        f"Organ Group: {group_name}",
        fontsize=30, y=1.05
    )
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}
Fibroblast = Fibroblast[Fibroblast['Self_immune'] == 1].copy()
# --- 복사 및 organ_group, age_group 추가 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')

# --- target subtypes ---
target_subtypes = sorted(Fibroblast['fibroblast_subtype'].dropna().unique())

# --- 전체 mouse 목록 ---
mouse_list_all = Fibroblast['Each mouse'].unique()

# --- 모든 mouse × subtype 조합 ---
all_combinations = pd.MultiIndex.from_product(
    [mouse_list_all, target_subtypes],
    names=['Each mouse', 'fibroblast_subtype']
).to_frame(index=False)

# --- mouse × subtype counts ---
subtype_counts = (
    Fibroblast.groupby(['Each mouse', 'fibroblast_subtype'])
    .size()
    .rename('subtype_count')
    .reset_index()
)

# --- NaN → 0 채우기 ---
subtype_counts = pd.merge(
    all_combinations, subtype_counts,
    on=['Each mouse', 'fibroblast_subtype'], how='left'
).fillna({'subtype_count': 0})

# --- Organ 전체 합산 기준 proportion 계산 (각 Subtype 합 = 1) ---
prop_df_list = []
for subtype in target_subtypes:
    data_subtype = subtype_counts[subtype_counts['fibroblast_subtype'] == subtype].copy()
    total_subtype_count = data_subtype['subtype_count'].sum()
    if total_subtype_count > 0:
        data_subtype['proportion'] = data_subtype['subtype_count'] / total_subtype_count
    else:
        data_subtype['proportion'] = 0
    prop_df_list.append(data_subtype)

prop_df = pd.concat(prop_df_list, axis=0)

# --- age_group 붙이기 ---
meta_info = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()
prop_df = pd.merge(prop_df, meta_info, on='Each mouse', how='left')

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- 시각화 ---
sns.set(style="whitegrid")
n_subtypes = len(target_subtypes)
fig, axes = plt.subplots(1, n_subtypes, figsize=(4 * n_subtypes, 4), sharey=True)
if n_subtypes == 1:
    axes = [axes]

for i, subtype in enumerate(target_subtypes):
    ax = axes[i]
    data = prop_df[prop_df['fibroblast_subtype'] == subtype]

    # dot 개수 출력
    n_young = (data['age_group'] == 'Young').sum()
    n_aged = (data['age_group'] == 'Aged').sum()

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=data, ax=ax, width=0.4)

    # 검은 점: 빨간 점 제외
    black_data = data[~data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=black_data,
                  color='black', size=8, jitter=True, ax=ax)

    # 빨간 점: 지정 샘플만
    red_data = data[data['Each mouse'].isin(red_dots_mice)]
    sns.stripplot(x='age_group', y='proportion', data=red_data,
                  color='red', size=10, jitter=True, ax=ax,
                  marker='o', edgecolor='black', linewidth=1.5)

    # --- 통계 ---
    group1 = data[data['age_group'] == 'Young']['proportion']
    group2 = data[data['age_group'] == 'Aged']['proportion']

    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(subtype, fontsize=20)
    ax.set_xlabel("")
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    if i == 0:
        ax.set_ylabel("Proportion (All organs, normalized within Subtype)", fontsize=12)
    else:
        ax.set_ylabel("")

    y_max = data['proportion'].max() if len(data['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=18, color='red', transform=ax.get_xaxis_transform())

plt.suptitle("ALL organs - Age group comparison", fontsize=30, y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

Fibroblast = Fibroblast[Fibroblast['Self_immune'] == 1].copy()
df = Fibroblast.copy()
# age_group 추가
df['age_group'] = np.where(df['age_num'] > 70, 'Aged', 'Young')
df = df[df['clinical information'] != 'decidualized_pregnant']
# 'Each mouse' 문자열 변환
df['Each mouse'] = df['Each mouse'].astype(str)

# 각 mouse별 전체 셀 개수 (index도 문자열)
mouse_counts = df['Each mouse'].value_counts()
mouse_counts.index = mouse_counts.index.astype(str)

# 전체 셀 개수
total_cells = len(df)

# 비율 계산
df['mouse_ratio'] = df['Each mouse'].map(mouse_counts) / total_cells

# 중복 제거
df_unique = df[['Each mouse', 'age_group', 'mouse_ratio']].drop_duplicates()

# t-test
aged_ratio = df_unique[df_unique['age_group'] == 'Aged']['mouse_ratio']
young_ratio = df_unique[df_unique['age_group'] == 'Young']['mouse_ratio']
t_stat, p_val = ttest_ind(aged_ratio, young_ratio)

# 빨간 점 찍을 샘플 리스트
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

plt.figure(figsize=(6, 5))
sns.boxplot(data=df_unique, x='age_group', y='mouse_ratio', width=0.5)

# 검은 점: red_dots_mice 제외
black_data = df_unique[~df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=black_data, x='age_group', y='mouse_ratio',
              color='black', size=6, jitter=True)

# 빨간 점: red_dots_mice만
red_data = df_unique[df_unique['Each mouse'].isin(red_dots_mice)]
sns.stripplot(data=red_data, x='age_group', y='mouse_ratio',
              color='red', size=8, jitter=True,
              marker='o', edgecolor='black', linewidth=1.5)

plt.xticks(fontsize=18)  # x축 라벨 크기 키우기

plt.text(0.5, df_unique['mouse_ratio'].max(),
         f"p = {p_val:.3e}",
         ha='center', va='bottom', color='red', fontsize=12)

plt.ylabel('Proportion of cells per mouse')
plt.xlabel('')
plt.title('HALLMARK_TNFA_SIGNALING_VIA_NFKB activation proportion')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- hallmark 조건 리스트 ---
hallmark_cols = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING',
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_TGF_BETA_SIGNALING',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'KEGG_IL_17_Signaling_pathway'
]

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
n_organs = len(organs)

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- hallmark 컬럼별 반복 ---
for hallmark in hallmark_cols:
    print(f"Plotting for: {hallmark}")
    
    # hallmark 1인 데이터만 사용
    Fibroblast_filtered = Fibroblast[Fibroblast[hallmark] == 1].copy()
    Fibroblast_filtered['organ_group'] = Fibroblast_filtered['organism part'].map(organ_group_map)
    Fibroblast_filtered['age_group'] = np.where(Fibroblast_filtered['age_num'] > 70, 'Aged', 'Young')
    
    # 한 줄 subplot
    sns.set(style="whitegrid")
    fig, axes = plt.subplots(1, n_organs, figsize=(5 * n_organs, 5), sharey=True)
    if n_organs == 1:
        axes = [axes]
    
    for i, organ in enumerate(organs):
        ax = axes[i]
        data_organ = Fibroblast_filtered[Fibroblast_filtered['organism part'] == organ]
        
        # mouse별 count
        mouse_counts = (
            data_organ.groupby('Each mouse')
            .size()
            .rename('cell_count')
            .reset_index()
        )
        
        # proportion 계산
        total_cells = mouse_counts['cell_count'].sum()
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells if total_cells > 0 else 0
        
        # age_group 붙이기
        meta_info = Fibroblast_filtered[['Each mouse', 'age_group']].drop_duplicates()
        mouse_counts = pd.merge(mouse_counts, meta_info, on='Each mouse', how='left')
        
        # proportion > 0인 데이터만 사용
        plot_data = mouse_counts[mouse_counts['proportion'] > 0]
        
        # boxplot
        sns.boxplot(x='age_group', y='proportion', data=plot_data, ax=ax, width=0.4)
        
        # stripplot
        black_data = plot_data[~plot_data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=black_data, color='black', size=6, jitter=True, ax=ax)
        red_data = plot_data[plot_data['Each mouse'].isin(red_dots_mice)]
        sns.stripplot(x='age_group', y='proportion', data=red_data, color='red', size=8, jitter=True, ax=ax, marker='o',
                      edgecolor='black', linewidth=1.5)
        
        # 통계
        group1 = plot_data[plot_data['age_group'] == 'Young']['proportion']
        group2 = plot_data[plot_data['age_group'] == 'Aged']['proportion']
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"
        
        # 라벨 및 글씨 크기
        ax.set_title(f"{organ + ' (control)' if organ == 'spleen' else organ}", fontsize=22)
        ax.set_xlabel("", fontsize=18)
        if i == 0:
            ax.set_ylabel(f"Proportion ({hallmark})", fontsize=14)
        else:
            ax.set_ylabel("")
        ax.tick_params(axis='x', labelsize=20)
        ax.tick_params(axis='y', labelsize=20)
        
        # p-value 표시
        y_max = plot_data['proportion'].max() if len(plot_data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=18, color='red', transform=ax.get_xaxis_transform())
    
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- organ_group 매핑 ---
organ_group_map = {
    "ovary": "upper",
    "oviduct": "upper",
    "uterus": "upper",
    "uterine cervix": "lower",
    "vagina": "lower",
    "spleen": "control"
}

# --- hallmark 조건 리스트 ---
hallmark_cols = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING',
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_TGF_BETA_SIGNALING',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'KEGG_IL_17_Signaling_pathway'
]

# --- organs 순서 ---
organs = ["ovary", "oviduct", "uterus", "uterine cervix", "vagina", "spleen"]
n_organs = len(organs)

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- organ별 filtering 조건 (제외할 마우스 목록) ---
mice_to_filter_by_organ = {
    "ovary": ["18mo_Ind001", "18mo_Ind002"],
    "oviduct": ["18mo_Ind004", "18mo_Ind005"],
    "uterus": ["Ind007", "18mo_Ind004", "18mo_Ind005"],
    "uterine cervix": ["18mo_Ind004", "18mo_Ind005"],
    "vagina": ["Ind006", "18mo_Ind004", "18mo_Ind005"],
    "spleen": ["18mo_Ind001", "18mo_Ind004"]
}

# --- 전체 메타(나이그룹) 미리 생성 ---
Fibroblast = Fibroblast.copy()
Fibroblast['organ_group'] = Fibroblast['organism part'].map(organ_group_map)
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')
meta_info_all = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()

sns.set(style="whitegrid")

for hallmark in hallmark_cols:
    print(f"\n                                                      === Plotting: {hallmark} ===")
    # hallmark==1 인 데이터 (셀 단위)
    filtered_h = Fibroblast[Fibroblast[hallmark] == 1].copy()

    fig, axes = plt.subplots(1, n_organs, figsize=(5 * n_organs, 5), sharey=True)
    if n_organs == 1:
        axes = [axes]

    for i, organ in enumerate(organs):
        ax = axes[i]

        # --- organ에 존재하는 모든 마우스(전체 데이터 기준) ---
        mice_in_organ_all = Fibroblast[Fibroblast['organism part'] == organ]['Each mouse'].unique().tolist()
        mice_to_exclude = mice_to_filter_by_organ.get(organ, [])
        # 제외 적용
        mice_used = [m for m in mice_in_organ_all if m not in mice_to_exclude]

        # 디버그 출력: 제외된 마우스 / 사용된 마우스
        # print(f"{organ}: excluded -> {mice_to_exclude}")
        # print(f"{organ}: used (after exclusion) -> {mice_used}")

        # hallmark==1 인 경우에 대해 organ별로 셀 카운트(있는 마우스만)
        counts = (
            filtered_h[filtered_h['organism part'] == organ]
            .groupby('Each mouse')
            .size()
            .rename('cell_count')
            .reset_index()
        )

        # 사용될 마우스 전체 목록과 merge 해서 없는 마우스는 cell_count=0으로 채움
        all_mice_df = pd.DataFrame({'Each mouse': mice_used})
        mouse_counts = pd.merge(all_mice_df, counts, on='Each mouse', how='left').fillna({'cell_count': 0})
        mouse_counts['cell_count'] = mouse_counts['cell_count'].astype(int)

        # proportion 계산 (organ 내 hallmark==1 총합 대비 각 마우스 비율)
        total_cells = mouse_counts['cell_count'].sum()
        if total_cells > 0:
            mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
        else:
            mouse_counts['proportion'] = 0.0  # 모든 마우스가 0인 경우

        # age_group 붙이기 (meta_info_all에서 가져오기)
        mouse_counts = pd.merge(mouse_counts, meta_info_all, on='Each mouse', how='left')

        # plot 데이터 (0 포함)
        plot_data = mouse_counts.copy()

        # boxplot
        # NOTE: boxplot needs at least some points per category; seaborn will handle single points.
        sns.boxplot(x='age_group', y='proportion', data=plot_data, ax=ax, width=0.4)

        # stripplot (검정/빨강 점)
        black_data = plot_data[~plot_data['Each mouse'].isin(red_dots_mice)]
        if not black_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=black_data, color='black', size=10, jitter=True, ax=ax)
        red_data = plot_data[plot_data['Each mouse'].isin(red_dots_mice)]
        if not red_data.empty:
            sns.stripplot(x='age_group', y='proportion', data=red_data, color='red', size=10, jitter=True, ax=ax,
                          marker='o', edgecolor='black', linewidth=1.5)

        # 통계 (Mann-Whitney U)
        group1 = plot_data[plot_data['age_group'] == 'Young']['proportion'].dropna()
        group2 = plot_data[plot_data['age_group'] == 'Aged']['proportion'].dropna()
        if len(group1) > 1 and len(group2) > 1:
            stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
            if pval < 0.0001:
                sig = "****"
            elif pval < 0.001:
                sig = "***"
            elif pval < 0.01:
                sig = "**"
            elif pval < 0.05:
                sig = "*"
            else:
                sig = "n.s."
            p_text = f"{sig}\n(p={pval:.2e})"
        else:
            p_text = "n/a"

        # 라벨/스타일
        ax.set_title(f"{organ + ' (control)' if organ == 'spleen' else organ}", fontsize=25)
        ax.set_xlabel("", fontsize=25)
        if i == 0:
            ax.set_ylabel(f"Proportion ({hallmark})", fontsize=14)
        else:
            ax.set_ylabel("")
        ax.tick_params(axis='x', labelsize=25)
        ax.tick_params(axis='y', labelsize=20)

        # p-value 표시 (y 위치는 데이터 최대값 기반)
        y_max = plot_data['proportion'].max() if len(plot_data['proportion'].dropna()) > 0 else 0.1
        ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
                fontsize=20, color='red', transform=ax.get_xaxis_transform())

    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# --- hallmark 조건 리스트 ---
hallmark_cols = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING',
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_TGF_BETA_SIGNALING',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'KEGG_IL_17_Signaling_pathway'
]

# --- 빨간 점 샘플 ---
red_dots_mice = ["18mo_Ind004", "18mo_Ind005"]

# --- age_group 생성 ---
Fibroblast = Fibroblast.copy()
Fibroblast['age_group'] = np.where(Fibroblast['age_num'] > 70, 'Aged', 'Young')
meta_info_all = Fibroblast[['Each mouse', 'age_group']].drop_duplicates()

sns.set(style="whitegrid")

# ------------------------------
# hallmark별 반복
# ------------------------------
for hallmark in hallmark_cols:
    print(f"\n=== Plotting: {hallmark} ===")

    # hallmark==1 셀만 추출
    filtered_h = Fibroblast[Fibroblast[hallmark] == 1].copy()

    # mouse별 cell count
    mouse_counts = (
        filtered_h.groupby('Each mouse')
        .size()
        .rename('cell_count')
        .reset_index()
    )

    # 모든 샘플 포함
    all_mice_df = pd.DataFrame({'Each mouse': Fibroblast['Each mouse'].unique()})
    mouse_counts = pd.merge(all_mice_df, mouse_counts, on='Each mouse', how='left').fillna({'cell_count': 0})
    mouse_counts['cell_count'] = mouse_counts['cell_count'].astype(int)

    # proportion 계산 (각 mouse / 전체 hallmark+ 셀 수)
    total_cells = mouse_counts['cell_count'].sum()
    if total_cells > 0:
        mouse_counts['proportion'] = mouse_counts['cell_count'] / total_cells
    else:
        mouse_counts['proportion'] = 0.0

    # age_group 붙이기
    mouse_counts = pd.merge(mouse_counts, meta_info_all, on='Each mouse', how='left')

    # ------------------------------
    # 시각화: age_group boxplot
    # ------------------------------
    fig, ax = plt.subplots(figsize=(6, 5))

    # boxplot
    sns.boxplot(x='age_group', y='proportion', data=mouse_counts, ax=ax, width=0.4)

    # dot
    black_data = mouse_counts[~mouse_counts['Each mouse'].isin(red_dots_mice)]
    if not black_data.empty:
        sns.stripplot(x='age_group', y='proportion', data=black_data, color='black', size=10, jitter=True, ax=ax)
    red_data = mouse_counts[mouse_counts['Each mouse'].isin(red_dots_mice)]
    if not red_data.empty:
        sns.stripplot(x='age_group', y='proportion', data=red_data,
                      color='red', size=10, jitter=True, ax=ax,
                      marker='o', edgecolor='black', linewidth=1.5)

    # 통계
    group1 = mouse_counts[mouse_counts['age_group'] == 'Young']['proportion'].dropna()
    group2 = mouse_counts[mouse_counts['age_group'] == 'Aged']['proportion'].dropna()
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"

    ax.set_title(f"{hallmark}", fontsize=20)
    ax.set_xlabel("", fontsize=14)
    ax.set_ylabel("Proportion", fontsize=14)
    y_max = mouse_counts['proportion'].max() if len(mouse_counts['proportion'].dropna()) > 0 else 0.1
    ax.text(0.5, y_max * 1.05, p_text, ha='center', va='bottom',
            fontsize=14, color='red', transform=ax.get_xaxis_transform())

    plt.tight_layout()
    plt.show()


# Geneset score

In [ ]:
import scanpy as sc
file_path = "/home/Data_Drive_8TB/kykim/1. Oval/AUCell/Fibroblast/Fibroblast_pathway.h5ad"
Fibroblast = sc.read_h5ad(file_path)

In [ ]:
Fibroblast = Fibroblast[Fibroblast.obs['organism part'] == 'ovary'].copy()
Fibroblast.obs['organism part'].value_counts()

In [ ]:
import scanpy as sc

# Gene set dictionary 정의
gene_sets = {
    "HALLMARK_INTERFERON_GAMMA_RESPONSE": ["Cd86","Trim25","Il15","Stat3","Stat2","Stat4","Stat1","Fgl2","Cdkn1a","Xcl1",
                                           "Wars1","Il15ra","Pml","Nfkbia","Psma3","Psma2","Tnfaip2","Il4ra","Cfb","St8sia4",
                                           "Ly6e","Trim21","Hif1a","Tnfsf10","Fpr1","Cd38","Irf9","Casp4","Casp3","Gpr18",
                                           "Myd88","Ripk1","Ciita","Gzma","Casp7","Cmklr1","Psme2","Psme1","Psmb10","Irf4",
                                           "Upp1","Bpgm","Ifnar2","Ifit3","Pla2g4a","Tnfaip6","Tnfaip3","Tapbp","Socs3","Casp8",
                                           "Ncoa3","Ifi27","Rnf213","Lcp2","Il18bp","Vamp8","Trim26","Mthfd2","St3gal5","Nod1",
                                           "Samd9l","Usp18","Rbck1","Psmb9","Psmb8","Psmb2","Irf5","Cxcl9","Cxcl10","Eif2ak2",
                                           "Sspn","Tor1b","Lats2","Socs1","C1s1","Isg15","Vamp5","Irf7","Cxcl11","Txnip",
                                           "Adar","Ripk2","Pfkp","Ifitm3","Isoc1","Sppl2a","Eif4e3","Lap3","Herc6","Peli1",
                                           "Ube2l6","Rtp4","Epsti1","Bst2","Lysmd2","Ifi35","Ifih1","Tmt1b","Pnpt1","Nup93",
                                           "Apol6","Ogfr","Parp14","Auts2","Marchf1","Cmtr1","Batf2","Trim14","Slamf7","Sp110",
                                           "Trafd1","Mvp","Gbp3","Cd274","Zbp1","Samhd1","Nmi","Isg20","Rsad2","Nampt",
                                           "Dhx58","Ifitm2","Rnf31","Ifi30","Znfx1","Tdrd7","Parp12","P2ry14","Arid5b","Slc25a28",
                                           "Oasl1","Oas3","Oas2","Ddx60","Rapgef6","Sectm1a","Helz2","Bank1","Rigi","Ifi44",
                                           "Nlrc5","Xaf1","B2m","Btg1","Cd40","Cd69","Cfh","Plscr1","Serping1","Fas",
                                           "Fcgr1","Gch1","H2-Aa","H2-DMa","Ifi44l","Ptpn6","Icam1","Irf8","Ido1","Cd74",
                                           "Il10ra","Casp1","Il2rb","Il6","Il7","Irf1","Irf2","Itgb7","Jak2","Mx2",
                                           "Nfkb1","Pnp","Pim1","Ptgs2","Ptpn1","Ptpn2","Ccl5","Selp","Sod2","Sri",
                                           "Tap1","Vcam1","Arl4a","Ifit2","Ccl7","Lgals3bp","Pde4b","Cmpk2"],
    "HALLMARK_INTERFERON_ALPHA_RESPONSE": ["Trim25","Il15","Stat2","Procr","Wars1","Psma3","Il4ra","Ly6e","Trim21","Elf1",
                                           "Irf9","Psme2","Psme1","Ifit3","Casp8","Ifi27","Trim26","Csf1","Samd9l","Usp18",
                                           "Psmb9","Psmb8","Uba7","Cxcl10","Eif2ak2","C1s1","Isg15","Irf7","Cxcl11","Nub1",
                                           "Txnip","Adar","Ripk2","Ifitm3","Gmpr","Lap3","Herc6","Lpar6","Ube2l6","Rtp4",
                                           "Epsti1","Tmem140","Ifitm1","Bst2","Ifi35","Ifih1","Pnpt1","Ogfr","Parp14","Ccrl2",
                                           "Mvb12a","Cmtr1","Batf2","Trim14","Sp110","Trafd1","Gbp3","Nmi","Isg20","Rsad2",
                                           "Dhx58","Parp9","Ifitm2","Rnf31","Ifi30","Tdrd7","Parp12","Slc25a28","Oasl1","Oas1a",
                                           "Ddx60","Helz2","Lamp3","Ifi44","Ncoa7","Tent5a","Trim12c","B2m","Cnp","Plscr1",
                                           "Ifi44l","Cd74","Casp1","Il7","Irf1","Irf2","Cd47","Mov10","Mx2","Sell",
                                           "Tap1","Ifit2","Lgals3bp","Cmpk2"],
    "HALLMARK_IL6_JAK_STAT3_SIGNALING": ["Stat3","Stat2","Stat1","Il12rb1","Ccr1","Pla2g2a","Il15ra","Ltb","Tnf","Ltbr",
                                         "Lepr","Il13ra1","Il4ra","Il18r1","Il17ra","Cd38","Irf9","Ifngr2","Ifngr1","Ifnar1",
                                         "Cd36","Myd88","Inhbe","Il10rb","Bak1","Socs3","Tnfrsf1b","Tnfrsf1a","Osmr","Acvr1b",
                                         "Acvrl1","Csf2","Csf1","Csf2ra","Csf3r","Cxcl2","Tlr2","Hax1","Map3k8","Tnfrsf12a",
                                         "Cxcl9","Cxcl10","Ebi3","Socs1","Il17rb","Pdgfc","Cxcl11","Cxcl13","Pf4","Crlf2",
                                         "Stam2","Tyk2","Tnfrsf21","Pik3r5","A2m","Cbl","Cd14","Cd44","Cd9","Fas",
                                         "Grb2","Hmox1","Il1b","Il1r1","Il1r2","Il2ra","Il2rg","Il3ra","Il6","Il6st",
                                         "Il7","Il9r","Irf1","Itga4","Itgb3","Jun","Pim1","Ptpn1","Ptpn2","Reg1",
                                         "Dntt","Tgfb1","Ptpn11","Ccl7","Cntfr"],
    "HALLMARK_INFLAMMATORY_RESPONSE" : ["Mmp14","Slc1a2","Ndp","Acvr2a","Ccr7","Il15","Pdpn","Rela","Ptger4","Cdkn1a",
                                     "Il15ra","Cd82","Atp2b1","Nfkbia","Kcnj2","Osm","Lta","Gpc3","Abi1","Ahr",
                                     "P2ry2","Il4ra","Il18r1","Ptafr","Ly6e","Adgre1","Hif1a","Tnfsf10","Fpr1","Ereg",
                                     "Hrh1","Ifngr2","Ifnar1","Pvr","Has2","Il18","Adm","Btg2","Rgs16","Kif1b",
                                     "Lpar1","Fzd5","Sema4d","Cmklr1","Cxcl5","Cx3cl1","C3ar1","Emp3","Tnfsf9","Tnfrsf9",
                                     "Tnfaip6","Cd70","Hpn","Tapbp","Psen1","Olr1","Ccl22","Marco","Rasgrp1","Tnfrsf1b",
                                     "Sphk1","Lcp2","Ccl20","Ccl17","Osmr","Gp1ba","Slc31a1","Slc31a2","Itgb8","P2rx4",
                                     "Il18rap","Acvr1b","Csf3","Csf1","Csf3r","Cxcl15","P2rx7","Tpbg","Tlr1","Nmur1",
                                     "Clec5a","Slc11a2","Tlr2","Aplnr","Klf6","Best1","Axl","Slamf1","Cxcl9","Cxcl10",
                                     "Eif2ak2","Ebi3","Prok2","Rgs1","Icosl","Irf7","Mefv","Npffr2","Gabbr1","Cxcl11",
                                     "Pcdh7","Atp2c1","Gpr132","Aqp9","Chst2","Ripk2","Slc28a2","Rtp4","Ifitm1","Bst2",
                                     "Kcnmb2","Dcbld2","Ccrl2","Sgms2","Icam4","Calcrl","Slc4a4","Nmi","Rhog","Ccl24",
                                     "Adrm1","Nampt","Cxcr6","Tlr3","Stab1","Tnfsf15","Rnf144b","Nod2","Irak2","Lamp3",
                                     "Ffar2","Gpr183","Pik3r5","Scarf1","Nlrp3","Atp2a2","Slc7a1","Bdkrb1","Cd14","Cd40",
                                     "Cd48","Cd69","F3","Cybb","Tacr3","Edn1","Gch1","Gna15","Gnai3","Selenos",
                                     "Hbegf","Icam1","Il10","Il10ra","Il12b","Il1a","Il1b","Il1r1","Il2rb","Il6",
                                     "Il7r","Inhba","Irf1","Itga5","Itgb3","Kcna3","Lck","Ldlr","Lif","Lyn",
                                     "Mxd1","Mep1a","Met","Myc","Nfkb1","Oprk1","Serpine1","Plaur","Ptger2","Ptpre",
                                     "Raf1","Ros1","Scn1b","Msr1","Ccl5","Sele","Sell","Sri","Tacr1","Timp1",
                                     "Vip","Adora2b","Ccl7","Ptgir","Pde4b","Abca1","Slc7a2"],
    "HALLMARK_TGF_BETA_SIGNALING" : ["Ppp1ca","Nog","Cdkn1c","Fnta","Skil","Xiap","Ifngr2","Hdac1","Slc20a1","Smad1",
                                  "Bmpr2","Rhoa","Smad7","Klf10","Tgif1","Smad3","Hipk2","Cdk9","Smad6","Ncor2",
                                  "Bmpr1a","Map3k7","Bcar3","Ube2d3","Smurf2","Rab31","Wwtr1","Smurf1","Ppp1r15a","Pmepa1",
                                  "Trim33","Arid4b","Acvr1","Apc","Bmp2","Ctnnb1","Cdh1","Eng","Fkbp1a","Id1",
                                  "Id2","Id3","Junb","Furin","Serpine1","Ski","Sptbn1","Tgfb1","Tgfbr1","Thbs1",
                                  "Tjp1","Ltbp2","Ppm1a"],
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB" : ["Mcl1","Cd80","Traf1","F2rl1","Dusp2","Tnc","Fosl2","Stat5a","Vegfa","Efna1",
                                       "Relb","Rela","Cebpd","Ptger4","Cdkn1a","Ptx3","Il15ra","Atp2b1","Nfkbia","Tnf",
                                       "Ier3","Ier2","Hes1","Tnfaip2","Dusp1","Eif1","Fosl1","Bcl6","Ifngr2","Tank",
                                       "Gadd45b","Gadd45a","Tubb2a","Sqstm1","Il18","Rhob","Cxcl1","Btg2","Nfe2l2","Tsc22d1",
                                       "Irs2","Atf3","Nfil3","Btg3","Jag1","Ackr3","Cxcl5","Phlda1","Bhlhe40","Per1",
                                       "Plk2","Nfkb2","Tnfsf9","Tnfrsf9","Klf10","Tgif1","Nfkbie","Tnfaip6","Tnfaip3","Ninj1",
                                       "Birc3","Birc2","Smad3","Dennd5a","Socs3","Phlda2","Olr1","Snn","Bcl2a1d","Egr3",
                                       "Sphk1","G0s2","Cd83","Ccl20","Klf9","Msc","Cflar","Ier5","Gfpt2","Csf2",
                                       "Csf1","Sgk1","Cxcl2","Ehd1","Fjx1","Klf4","Klf2","Tlr2","Klf6","Map2k3",
                                       "Map3k8","Sdc4","Cxcl10","Nr4a1","Nr4a2","Nr4a3","Icosl","Nfat5","Panx1","Cxcl11",
                                       "Plek","Rcan1","Ripk2","Dnajb4","Plpp3","Pnrc1","Kynu","Ifih1","Dram1","Ccrl2",
                                       "Spsb1","Rnf19b","Ccnl1","Tnip1","Ppp1r15a","B4galt5","Pdlim5","Litaf","Pmepa1","Nampt",
                                       "Clcf1","Il23a","Zbtb10","Slc16a6","Trip10","Tnfaip8","Tiparp","Pfkfb3","Zc3h12a","Tnip2",
                                       "Yrdc","Gpr183","Dusp4","Rigi","Slc2a6","Trib1","Kdm6b","Dusp5","Areg","Bcl3",
                                       "Bmp2","Btg1","Ccnd1","Cd44","Cd69","Cebpb","F3","Ccn1","Serpinb8","Edn1",
                                       "Egr1","Egr2","Ets2","Fos","Fosb","Fut4","Gch1","B4galt1","Slc2a3","Hbegf",
                                       "Icam1","Id2","Il12b","Il1a","Il1b","Il6","Il6st","Il7r","Inhba","Irf1",
                                       "Jun","Junb","Ldlr","Lif","Marcks","Mxd1","Maff","Myc","Nfkb1","Serpine1",
                                       "Serpinb2","Plau","Plaur","Ptgs2","Ptpre","Rel","Sat1","Ccl5","Sod2","Tap1",
                                       "Zfp36","Ifit2","Pde4b","Abca1","Gem","Lamb3"],
    "KEGG_IL_17_Signaling_pathway" : ["Traf3ip2","Srsf1","Defb6","Casp3","Casp8","Cebpb","Chuk","Csf2","Csf3",
    "Il25","Fadd","Fos","Fosb","Fosl1","Cxcl1","Hsp90ab1","Hsp90aa1","Elavl1",
    "Cxcl10","Ifng","Ikbkb","Ikbkg","Il13","Il17a","Il17ra","Il1b","Il4","Il5",
    "Il6","Jun","Jund","Lcn2","Il17rc","Mmp13","Mmp3","Mmp9","Muc5ac","Nfkb1",
    "Nfkbia","Mapk11","Ptgs2","Rela","S100a8","S100a9","Ccl11","Ccl12","Ccl17",
    "Ccl2","Ccl20","Ccl7","Cxcl2","Cxcl5","Tnf","Tnfaip3","Hsp90b1","Traf2",
    "Traf3","Traf4","Traf5","Traf6","Mapk4","Il17c","Il17d","Mapk7","Il17f",
    "Map3k7","Mapk1","Mapk10","Mapk13","Mapk14","Mapk3","Mapk8","Mapk9","Defb3",
    "Mapk12","Usp25","Cxcl3","Mapk15","Mapk6","Il17rb","Gm5849","Il17b","Tbk1",
    "Ikbke","Defb4","Gsk3b","Il17re","Anapc5","Tab3","Tab2","Tradd","Muc5b",
    "Mmp1a","Mmp1b"]
    
}

# Gene set scoring
for name, genes in gene_sets.items():
    sc.tl.score_genes(Fibroblast, gene_list=genes, score_name=name)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# plot할 pathway 리스트
pathways = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE', 
    'HALLMARK_INTERFERON_ALPHA_RESPONSE', 
    'HALLMARK_IL6_JAK_STAT3_SIGNALING', 
    'HALLMARK_INFLAMMATORY_RESPONSE', 
    'HALLMARK_TGF_BETA_SIGNALING', 
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB', 
    'KEGG_IL_17_Signaling_pathway'
]

# subplot grid 생성 (행 4, 열 2, 마지막은 비워둘 수도 있음)
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(15,20))
axes = axes.flatten()  # 2D 배열 -> 1D flatten

for i, pathway in enumerate(pathways):
    sns.boxplot(
        data=Fibroblast.obs, 
        x='age_group', 
        y=pathway, 
        hue='organism part',
        ax=axes[i]
    )
    axes[i].set_title(f'{pathway} score by Age Group and Organism Part')
    axes[i].set_xlabel('Age Group')
    axes[i].set_ylabel('Score')
    axes[i].legend(title='Organism Part', bbox_to_anchor=(1.05, 1), loc='upper left')

# 마지막 subplot이 비어있으면 제거
if len(pathways) < len(axes):
    for j in range(len(pathways), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
pathways = [
    'HALLMARK_INTERFERON_GAMMA_RESPONSE', 
    'HALLMARK_INTERFERON_ALPHA_RESPONSE', 
    'HALLMARK_IL6_JAK_STAT3_SIGNALING', 
    'HALLMARK_INFLAMMATORY_RESPONSE', 
    'HALLMARK_TGF_BETA_SIGNALING', 
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB', 
    'KEGG_IL_17_Signaling_pathway'
]


sc.pl.umap(
    Fibroblast,
    color="HALLMARK_INTERFERON_GAMMA_RESPONSE",
    # Setting a smaller point size to get prevent overlap
    size=2,
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ovary만 필터링
df = Fibroblast.obs[Fibroblast.obs['organism part'] == 'ovary'][['fibroblast_subtype'] + pathways].copy()

# long format으로 변환
df_long = df.melt(id_vars='fibroblast_subtype', var_name='Pathway', value_name='Score')

# boxplot 그리기
plt.figure(figsize=(12,6))
sns.boxplot(x='Pathway', y='Score', hue='fibroblast_subtype', data=df_long)
plt.xticks(rotation=45, ha='right')
plt.title('Pathway Scores by Fibroblast Subtype (Ovary)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ovary 샘플만 필터링
df = Fibroblast.obs[Fibroblast.obs['organism part'] == 'ovary'][['fibroblast_subtype', 'age_group'] + pathways].copy()

# long format으로 변환
df_long = df.melt(id_vars=['fibroblast_subtype', 'age_group'], var_name='Pathway', value_name='Score')

# age_group별로 boxplot 2번 생성
for age in df_long['age_group'].unique():
    plt.figure(figsize=(12,6))
    sns.boxplot(x='Pathway', y='Score', hue='fibroblast_subtype',
                data=df_long[df_long['age_group'] == age])
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Pathway Scores by Fibroblast Subtype ({age}) - Ovary')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ovary 샘플만 필터링
df = Fibroblast.obs[Fibroblast.obs['organism part'] == 'ovary'][['fibroblast_subtype', 'age_group'] + pathways].copy()

# long format으로 변환
df_long = df.melt(id_vars=['fibroblast_subtype', 'age_group'], var_name='Pathway', value_name='Score')

# pathway별 subplot 생성
pathway_list = df_long['Pathway'].unique()
n_pathways = len(pathway_list)
fig, axes = plt.subplots(1, n_pathways, figsize=(5*n_pathways, 6), sharey=True)

for i, pathway in enumerate(pathway_list):
    sns.boxplot(x='fibroblast_subtype', y='Score', hue='age_group',
                data=df_long[df_long['Pathway'] == pathway],
                ax=axes[i])
    axes[i].set_title(pathway)
    axes[i].set_xlabel('')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')

axes[0].set_ylabel('Score')
axes[-1].legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# ovary 샘플만 필터링
df = Fibroblast.obs[Fibroblast.obs['organism part'] == 'ovary'][['age_group'] + pathways].copy()

# long format으로 변환
df_long = df.melt(id_vars='age_group', var_name='Pathway', value_name='Score')

# pathway별 subplot 생성
pathway_list = df_long['Pathway'].unique()
n_pathways = len(pathway_list)
fig, axes = plt.subplots(1, n_pathways, figsize=(6*n_pathways, 6), sharey=False)  # sharey=False

for i, pathway in enumerate(pathway_list):
    data = df_long[df_long['Pathway'] == pathway]
    
    # violin plot
    sns.violinplot(
        x='age_group', y='Score',
        data=data,
        ax=axes[i],
        width=0.6,
        inner='box',
        palette='Set2'
    )
    
    # 통계분석 (Mann-Whitney U test)
    group1 = data[data['age_group'] == 'Young']['Score']
    group2 = data[data['age_group'] == 'Aged']['Score']
    
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # y축 최대값에 맞춰 통계 텍스트 위치 설정
    y_min, y_max = axes[i].get_ylim()  # 현재 y축 범위
    axes[i].text(0.5, y_max*0.8, p_text, ha='center', fontsize=24)  # y_max보다 조금 아래에 표시
    
    # 글씨 크기 조절
    axes[i].set_title(pathway, fontsize=20)
    axes[i].set_xlabel('', fontsize=12)
    axes[i].set_ylabel('Score', fontsize=12 if i == 0 else 0)
    axes[i].tick_params(axis='x', labelsize=25)
    axes[i].tick_params(axis='y', labelsize=25)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# valid fibroblast subtype 정의
valid_types = ['ECM_fibroblast', 'Metabolism_fibroblast', 'Vascular_fibroblast']

# ovary 샘플 + fibroblast_subtype 필터링
df = Fibroblast.obs[
    (Fibroblast.obs['organism part'] == 'ovary') &
    (Fibroblast.obs['fibroblast_subtype'].isin(valid_types))
][['age_group'] + pathways].copy()

# long format으로 변환
df_long = df.melt(id_vars='age_group', var_name='Pathway', value_name='Score')

# pathway별 subplot 생성
pathway_list = df_long['Pathway'].unique()
n_pathways = len(pathway_list)
fig, axes = plt.subplots(1, n_pathways, figsize=(6*n_pathways, 6), sharey=False)

for i, pathway in enumerate(pathway_list):
    data = df_long[df_long['Pathway'] == pathway]
    
    # violin plot (기존 코드 그대로)
    sns.violinplot(
        x='age_group', y='Score',
        data=data,
        ax=axes[i],
        width=0.6,
        inner='box',
        palette='Set2'
    )
    
    # 통계분석 (Mann-Whitney U test)
    group1 = data[data['age_group'] == 'Young']['Score']
    group2 = data[data['age_group'] == 'Aged']['Score']
    
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # y축 최대값에 맞춰 통계 텍스트 위치 설정
    y_min, y_max = axes[i].get_ylim()
    axes[i].text(0.5, y_max*0.8, p_text, ha='center', fontsize=24)
    
    # 글씨 크기 조절
    axes[i].set_title(pathway, fontsize=20)
    axes[i].set_xlabel('', fontsize=12)
    axes[i].set_ylabel('Score', fontsize=12 if i == 0 else 0)
    axes[i].tick_params(axis='x', labelsize=25)
    axes[i].tick_params(axis='y', labelsize=25)

plt.tight_layout()
plt.show()


In [ ]:
sc.pl.umap(
    Fibroblast,
    color="fibroblast_subtype",
    # Setting a smaller point size to get prevent overlap
    size=2,
    legend_loc="on data",
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import random

# subtype별 개수 집계
counts = Fibroblast.obs['fibroblast_subtype'].value_counts()

# 색상 고정
color_map = {
    "Metabolism_fibroblast": "purple",
    "ECM_fibroblast": "blue",
    "Vascular_fibroblast": "pink"
}

# 이미 사용한 색상
used_colors = set(color_map.values())

# 사용할 수 있는 matplotlib 색상 중에서 남은 색
available_colors = [c for c in mcolors.CSS4_COLORS.keys() if c not in used_colors]

# 랜덤 색상 배정
for subtype in counts.index:
    if subtype not in color_map:
        color_map[subtype] = random.choice(available_colors)
        available_colors.remove(color_map[subtype])  # 겹치지 않게 제거

# 최종 색상 리스트 (순서는 counts.index 순서와 맞춤)
colors = [color_map[sub] for sub in counts.index]

# 파이차트 그리기
plt.figure(figsize=(6,6))
wedges, texts, autotexts = plt.pie(
    counts,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False
)

# 범례 추가
plt.legend(
    wedges,
    counts.index,
    title="Fibroblast Subtypes",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.title("Fibroblast Subtype Distribution")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 원하는 색상 지정
color_map = {
    "Metabolism_fibroblast": "purple",
    "ECM_fibroblast": "blue",
    "Vascular_fibroblast": "pink"
}

# 현재 subtype 카테고리 순서 확인
categories = Fibroblast.obs['fibroblast_subtype'].cat.categories

# 모든 subtype에 대해 색상 할당 (나머지는 기본 팔레트에서 가져오기)
from matplotlib import cm
import random

# 팔레트에서 여러 색 뽑기 (나머지용)
palette = [plt.cm.tab20(i) for i in range(20)]
random.shuffle(palette)

colors = []
for cat in categories:
    if cat in color_map:
        colors.append(color_map[cat])
    else:
        # 랜덤에서 하나 꺼내서 배정
        colors.append(palette.pop())

# Scanpy에 색상 저장
Fibroblast.uns['fibroblast_subtype_colors'] = colors

# 이제 UMAP 다시 그리면 원하는 색 적용됨
sc.pl.umap(
    Fibroblast,
    color="fibroblast_subtype",
    size=2,
    legend_loc="on data"
)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# ovary 샘플만 필터링
df = Fibroblast.obs[Fibroblast.obs['organism part'] == 'ovary'][['clinical information'] + pathways].copy()

# long format으로 변환
df_long = df.melt(id_vars='clinical information', var_name='Pathway', value_name='Score')

# pathway별 subplot 생성
pathway_list = df_long['Pathway'].unique()
n_pathways = len(pathway_list)
fig, axes = plt.subplots(1, n_pathways, figsize=(6*n_pathways, 6), sharey=False)

for i, pathway in enumerate(pathway_list):
    data = df_long[df_long['Pathway'] == pathway]
    
    # violin plot
    sns.violinplot(
        x='clinical information', y='Score',
        data=data,
        ax=axes[i],
        width=0.6,
        inner='box',
        palette='Set2'
    )
    
    # 통계분석 (Mann-Whitney U test)
    group1 = data[data['clinical information'] == 'unestrus']['Score']
    group2 = data[data['clinical information'] != 'unestrus']['Score']
    
    if len(group1) > 1 and len(group2) > 1:
        stat, pval = mannwhitneyu(group1, group2, alternative='two-sided')
        if pval < 0.0001:
            sig = "****"
        elif pval < 0.001:
            sig = "***"
        elif pval < 0.01:
            sig = "**"
        elif pval < 0.05:
            sig = "*"
        else:
            sig = "n.s."
        p_text = f"{sig}\n(p={pval:.2e})"
    else:
        p_text = "n/a"
    
    # y축 최대값에 맞춰 통계 텍스트 위치 설정
    y_min, y_max = axes[i].get_ylim()
    axes[i].text(0.5, y_max*0.8, p_text, ha='left', fontsize=24)
    
    # 글씨 크기 조절
    axes[i].set_title(pathway, fontsize=20)
    axes[i].set_xlabel('', fontsize=15)
    axes[i].set_ylabel('Score', fontsize=12 if i == 0 else 0)
    axes[i].tick_params(axis='x', labelsize=14)
    axes[i].tick_params(axis='y', labelsize=25)

plt.tight_layout()
plt.show()


# Old cell

In [ ]:
import scanpy as sc
file_path = '/home/Data_Drive_8TB/kykim/1. Oval/Save_data/After_HVG_Clustering.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
adata.obs['organism part'].value_counts()

In [ ]:
import numpy as np
adata.obs["age_num"] = adata.obs["age"].str.extract(r"(\d+)").astype(int)
adata.obs['age_group'] = np.where(adata.obs['age_num'] > 70, 'Aged', 'Young')

In [ ]:
adata = adata[adata.obs['clinical information'] != 'decidualized_pregnant'].copy()

In [ ]:
adata.obs['Each mouse'].value_counts()

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1️⃣ decidualized_pregnant 아닌 셀만 추출
adata_sub = adata[adata.obs['clinical information'] != 'decidualized_pregnant'].copy()

# 2️⃣ 관심 유전자
genes_of_interest = ['Pcdhgb4', 'Pcdhgb5', 'Pcdhga9', 'Pcdhga12', 'Pcdhgc4', 'Pcdhgc5']

# 3️⃣ organism part 별로 평균 발현 계산
# adata.raw가 있으면 raw를 사용하고, 아니면 adata.X 사용
if adata_sub.raw is not None:
    expr = pd.DataFrame(adata_sub.raw[:, genes_of_interest].X.toarray(),
                        index=adata_sub.obs_names, columns=genes_of_interest)
else:
    expr = pd.DataFrame(adata_sub[:, genes_of_interest].X.toarray(),
                        index=adata_sub.obs_names, columns=genes_of_interest)

expr = expr.join(adata_sub.obs[['organism part', 'age_group']])

# 4️⃣ organism part 별, age_group 별 평균 발현 계산
expr_mean = expr.groupby(['organism part', 'age_group'])[genes_of_interest].mean().reset_index()

# 5️⃣ Boxplot 시각화 (유전자별)
for gene in genes_of_interest:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=expr, x='age_group', y=gene, hue='organism part')
    sns.stripplot(data=expr, x='age_group', y=gene, hue='organism part', 
                  dodge=True, color='black', alpha=0.3, legend=False)  # 범례 제거
    plt.title(f'{gene} expression by age_group and organism part')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

